# RSNA Knee — complément 320 (séries manquantes)

**Shard 0/2** `t1` (CPU, internet off). **320×320** uint8.

Les séries fluid 320 sont déjà le clone du 224 (mêmes UID, même crop, mêmes coupes). Ce kernel ne les refait pas.

UID **figés** depuis `manifest_t1_part` du crop 130 mm 224 (1 série non-fluide extra par étude, hors des 3 picks fluid). Même recette spatiale :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 % si spacing absent
2. bande profondeur **20–80 %**
3. resample **24** coupes, resize **320×320**, uint8, percentiles 1–99

Sortie dans `/kaggle/working/` :
- `volumes_crop130_320_t1_part0.zip`
- `manifest_t1_part0.csv`
- `RECIPE.json`

A déposer dans `data/crop130_320/` à côté des zip fluid.

In [ ]:
SHARD_ID = 0
N_SHARDS = 2

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        # debug local : CSV seulement, pas de DICOM
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
# Kaggle : train_series/<StudyUID>/<SeriesUID>/*.dcm
# (parfois a la racine COMP_DIR, parfois un cran plus bas)
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10013663742400736029415768216599146542","series":"1.2.826.0.1.3680043.8.498.17496901872433262274477070632489305322","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231","series":"1.2.826.0.1.3680043.8.498.33836648034724869721686434744559693623","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260","series":"1.2.826.0.1.3680043.8.498.75714899997203615784077798038670363546","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10018749256316287542884435387067561101","series":"1.2.826.0.1.3680043.8.498.69783908410864161211158583420543900194","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513","series":"1.2.826.0.1.3680043.8.498.10887922642378821953149095448791960311","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10029520033957968044048068502185417806","series":"1.2.826.0.1.3680043.8.498.11788164428905677775308855476550714595","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10048842414095994814596043262132965018","series":"1.2.826.0.1.3680043.8.498.60519317366971570170934879228359117304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10026410865298348818136520318965356394","series":"1.2.826.0.1.3680043.8.498.10039810104099273075015998627058683073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269","series":"1.2.826.0.1.3680043.8.498.60540822688937024151515211249136527086","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10077497528995409346950814927366582736","series":"1.2.826.0.1.3680043.8.498.52609252494549544989037353476227811522","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10081117556977868957150377234052259018","series":"1.2.826.0.1.3680043.8.498.63761209305397181817615997660125409593","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10059200627175320481920009076697103801","series":"1.2.826.0.1.3680043.8.498.99708703036078965966876737788881385029","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10087350795575215080051569638065974347","series":"1.2.826.0.1.3680043.8.498.10238197199619807565017429401905996014","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10088707526476055079083082931629995449","series":"1.2.826.0.1.3680043.8.498.78772886889622746005644245047670703647","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10090631811016939819371208464230332678","series":"1.2.826.0.1.3680043.8.498.11063356380296977745889863467626087110","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10096919689272424388113819845481617364","series":"1.2.826.0.1.3680043.8.498.26233672939359458850492656306941072976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10112440805412493412191181061985753636","series":"1.2.826.0.1.3680043.8.498.35820233711714570510950714573636799689","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10120841513315425967980213160839994882","series":"1.2.826.0.1.3680043.8.498.38478068839340197314332463139684733946","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343","series":"1.2.826.0.1.3680043.8.498.98027629871036248874002203265341743237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166","series":"1.2.826.0.1.3680043.8.498.84878015408307681578702917309931549382","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10124701778993602117271554854098856833","series":"1.2.826.0.1.3680043.8.498.63588274439837263681578203156909569430","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10136486417446492428821313562961937182","series":"1.2.826.0.1.3680043.8.498.69425699921184484485239858679622033597","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10144735099702632881322032871591573824","series":"1.2.826.0.1.3680043.8.498.34551518955124776381391941770845659774","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10133843191898064967257276100779798471","series":"1.2.826.0.1.3680043.8.498.39937387430982382677410262100791824950","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10096063158753121251217031021181036863","series":"1.2.826.0.1.3680043.8.498.91348587995236577129146264967955930115","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10150914432481980534086434136814789205","series":"1.2.826.0.1.3680043.8.498.12635219996711974925045224418830382646","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10148517526101576084961355066637531019","series":"1.2.826.0.1.3680043.8.498.14860289040260638280423011827899962328","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10157549662917062273945150928668828930","series":"1.2.826.0.1.3680043.8.498.70348539454982244256928330288311963066","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10159551371305936155888341171022480536","series":"1.2.826.0.1.3680043.8.498.27634092237325443249951719414199451510","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10147717839726734017869499799005084725","series":"1.2.826.0.1.3680043.8.498.67728755011495175722943980277876949575","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10165669527316708372190215034240922433","series":"1.2.826.0.1.3680043.8.498.21444945880760356085615309481698551742","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10181119664728335020976262955776463973","series":"1.2.826.0.1.3680043.8.498.12198739706602489282892432162777111999","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10189392162056736163704940747328177364","series":"1.2.826.0.1.3680043.8.498.10857715131994138598011004110668996188","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10182565688010898600749222617373997197","series":"1.2.826.0.1.3680043.8.498.98532718918289001119806201719594950072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10197515493414564227930855210542221039","series":"1.2.826.0.1.3680043.8.498.12980681846875147350649586386360411702","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10199522802568420589167284753473646773","series":"1.2.826.0.1.3680043.8.498.32008633693425842340822037558562279727","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10207134048598925821261298801127950516","series":"1.2.826.0.1.3680043.8.498.69357085928739043872700418618329301964","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10201660732108909756259359440298787775","series":"1.2.826.0.1.3680043.8.498.10967877394372926199437908171011016345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10162168547770229150261479353227398478","series":"1.2.826.0.1.3680043.8.498.61436613152337143341651094242302156767","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10229223546412441924801428332647762115","series":"1.2.826.0.1.3680043.8.498.55304327072157257009276640992522901216","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10218285921271312931891912812808052375","series":"1.2.826.0.1.3680043.8.498.10278488061269623205313792244369792044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10211034309703220222647821582770542129","series":"1.2.826.0.1.3680043.8.498.11112142671354810696616635369245080090","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10247930792182001003497189021027466833","series":"1.2.826.0.1.3680043.8.498.11211335876233026549969879030404416523","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10240166440044087554182120832488098189","series":"1.2.826.0.1.3680043.8.498.57165240487460004762476306963303841880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766","series":"1.2.826.0.1.3680043.8.498.80258559168539047134052592979729222310","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10261794102024899290234062909324134903","series":"1.2.826.0.1.3680043.8.498.44180132265297497459590473332939948840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10262784969531507133353458615303007839","series":"1.2.826.0.1.3680043.8.498.96225637366864847874681571967453119985","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10252163031705285141538200795896549203","series":"1.2.826.0.1.3680043.8.498.41174475351551546519650719301924245446","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10273861326753707436672556008991334542","series":"1.2.826.0.1.3680043.8.498.12831627238089871245025491720802928097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10274956886032850418742861294277331382","series":"1.2.826.0.1.3680043.8.498.10281359408836745211721920919607939009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10284263494252700630857173843779724077","series":"1.2.826.0.1.3680043.8.498.60337584896692774071317547908359669895","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10299618611626190927173051117228647808","series":"1.2.826.0.1.3680043.8.498.89596859812892749464024763130346035215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10241639721703882165429374175491428414","series":"1.2.826.0.1.3680043.8.498.11522915971551197221627494920191937549","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10287488031206457680758379452936914669","series":"1.2.826.0.1.3680043.8.498.54137868922970612269408471390702543246","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10309477218703786877612946261946268132","series":"1.2.826.0.1.3680043.8.498.73686837107266161783005896227858611804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10304242666970626179776896768140580106","series":"1.2.826.0.1.3680043.8.498.10716836225584421877601423730579557959","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10312350714164739081126996907250465738","series":"1.2.826.0.1.3680043.8.498.43835505875637030850523826237531274924","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10321556388278642027434955207617064548","series":"1.2.826.0.1.3680043.8.498.12642773720081203807866549236267377452","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10322724865012212544754724404226583315","series":"1.2.826.0.1.3680043.8.498.75892971565241529082451849455469914163","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10326352669696598934883805147731461341","series":"1.2.826.0.1.3680043.8.498.10029607251604215549686423856477964809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10318366890716843554246682186865048846","series":"1.2.826.0.1.3680043.8.498.85895097111243735822865879103678845392","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10330805760778577501878048995484987687","series":"1.2.826.0.1.3680043.8.498.11868301510827889028863462311529831208","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10336300544286006569730381727202519821","series":"1.2.826.0.1.3680043.8.498.62928058878742677290148933101280575226","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10348611327224519953673858098392135936","series":"1.2.826.0.1.3680043.8.498.29230858168705832280815309056644742215","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10351168527688244230248760520733020605","series":"1.2.826.0.1.3680043.8.498.37243230079513932589115488417034538512","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10352011938574988727803498915608467179","series":"1.2.826.0.1.3680043.8.498.51328118613639433991332641995495843243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10354576637002647863048798607011636355","series":"1.2.826.0.1.3680043.8.498.31190305534499887879578927967668398461","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10382074558506016193900889176239337801","series":"1.2.826.0.1.3680043.8.498.12314009451722701931884835374730405401","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10370612191212456931506340727148605247","series":"1.2.826.0.1.3680043.8.498.46635992881064097163699454530900405525","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10387227472110643414333498953513810066","series":"1.2.826.0.1.3680043.8.498.46104776405641995274731288111846446818","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10374791734880310643587680407133671218","series":"1.2.826.0.1.3680043.8.498.24139970067000919877444819504826219769","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10395211177230760565674811878437948933","series":"1.2.826.0.1.3680043.8.498.68027724471167073605029503597769156815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10415616591430643294142401482460086713","series":"1.2.826.0.1.3680043.8.498.13185080101639073577674981654606412299","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10402431429660337753694827208437698007","series":"1.2.826.0.1.3680043.8.498.18406932297731874762027035646250716415","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10418888172081069709796646795446885715","series":"1.2.826.0.1.3680043.8.498.96557260978387941500935579002254150061","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10407865232593182855530878478701744409","series":"1.2.826.0.1.3680043.8.498.12439486678704691419506069105688882944","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10419993051271996183722192755371692029","series":"1.2.826.0.1.3680043.8.498.11771752034240094063118985489471941832","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10422536395266111565753575369125069689","series":"1.2.826.0.1.3680043.8.498.75290556037896370144248993657476895450","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10429032846672074760218668873927822056","series":"1.2.826.0.1.3680043.8.498.48867926226552814715014504242639517711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10387876708310019345913209598612198432","series":"1.2.826.0.1.3680043.8.498.12512504807746259515139594057306618701","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10442828926117258624043497279351962738","series":"1.2.826.0.1.3680043.8.498.60860261135880937152632561056796252538","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10447920208680921688438738178539475509","series":"1.2.826.0.1.3680043.8.498.86726828908210715470371549783664000011","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10434135978927658622039675788413759069","series":"1.2.826.0.1.3680043.8.498.51952136568262733406993305724610375041","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10455589411958266725060942213473374667","series":"1.2.826.0.1.3680043.8.498.10299525200880174330696187663982648634","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10459901081962705567398560859229437818","series":"1.2.826.0.1.3680043.8.498.57698081383786123075761666916775177472","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10425131723456547056365404016184663035","series":"1.2.826.0.1.3680043.8.498.34459191379528356295889111188107065683","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10472519452146922631626638652503679695","series":"1.2.826.0.1.3680043.8.498.55209899621551879954731886130222982852","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10485026284343436524865922231113748546","series":"1.2.826.0.1.3680043.8.498.12865605300249850040931642953040243874","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10492484093480313596330475837639172625","series":"1.2.826.0.1.3680043.8.498.94537004975841218661336487972859324209","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10505282086293876725410939800054491454","series":"1.2.826.0.1.3680043.8.498.67890987531409138009444151065151011784","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10495855084258410825408867079175646590","series":"1.2.826.0.1.3680043.8.498.12066111103599414456618020249053783516","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10512466806929129734785318932315918522","series":"1.2.826.0.1.3680043.8.498.80951243654806465601559003669953177710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10522585237725235866013737575867341946","series":"1.2.826.0.1.3680043.8.498.23620199704815482140329754787371560009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10533568124272062696642461303295694739","series":"1.2.826.0.1.3680043.8.498.62407820280040385144206393521553225916","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10519566075312736546346469231439794627","series":"1.2.826.0.1.3680043.8.498.73833957956120486166129566557088880576","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10501068619333941198079851978532746117","series":"1.2.826.0.1.3680043.8.498.12493182928542097939674663986508624073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10536557774352716497356905292652597953","series":"1.2.826.0.1.3680043.8.498.93743879713071798854638156048315549541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10540980111087702906123811593906850156","series":"1.2.826.0.1.3680043.8.498.11671876921323325996821863019118857275","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10530226787566708645477257565528473018","series":"1.2.826.0.1.3680043.8.498.79970781243706967595058064724816180190","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10538502578022388392340022120166213828","series":"1.2.826.0.1.3680043.8.498.11488188191637132309135956282182919822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10541215772904358938061268226270989801","series":"1.2.826.0.1.3680043.8.498.67331826976507682746158193548216152388","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10563016328082326759266280856730347260","series":"1.2.826.0.1.3680043.8.498.23410431414728261692500154720536915875","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10549360192675575195491716398555148512","series":"1.2.826.0.1.3680043.8.498.11874189525742157715923761034656062465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10578013533179563448221888658506023084","series":"1.2.826.0.1.3680043.8.498.63812132595953551557962684818043251692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10581718331223731732815674582085184169","series":"1.2.826.0.1.3680043.8.498.81217025910453057976632765369170319920","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10542526906255526405613760733936070251","series":"1.2.826.0.1.3680043.8.498.11722565324009483640791704775511225225","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10590169032156570386146081157598789893","series":"1.2.826.0.1.3680043.8.498.77636139070295531239702483298199972343","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10593694750925047629619718537436931628","series":"1.2.826.0.1.3680043.8.498.26378948108703251678067277220989021967","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10589889055723532336864750992142411145","series":"1.2.826.0.1.3680043.8.498.19456599046232988845616700737431591582","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10601711635138681269270505152909539290","series":"1.2.826.0.1.3680043.8.498.79454707772150673356555223291882794951","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10615816709018712955642546871639043403","series":"1.2.826.0.1.3680043.8.498.10537502922202855015405248746117100342","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10619967804774235096151367674093177430","series":"1.2.826.0.1.3680043.8.498.13116655939906914102412938793438358404","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10603952065748622499432940419853732241","series":"1.2.826.0.1.3680043.8.498.51570111973883658903465110193466174048","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10626071553432761206594400919097854593","series":"1.2.826.0.1.3680043.8.498.37550004584356291649744033589967316474","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10627019153481144392726933328057818971","series":"1.2.826.0.1.3680043.8.498.13065515215081880112949009003231205554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10600654987896381983520458496207741798","series":"1.2.826.0.1.3680043.8.498.13312095336842228726408531221265860304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10642286343421893916348249679230632066","series":"1.2.826.0.1.3680043.8.498.56406882535116717485888469266420101844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10631238417904652761254062374051500506","series":"1.2.826.0.1.3680043.8.498.10524474659425920579843251311720556593","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10652764425372125015319160977894047658","series":"1.2.826.0.1.3680043.8.498.11013503879363481829718099650262938749","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10656388982978239967297620214873636996","series":"1.2.826.0.1.3680043.8.498.10943910845018484217338945109491283099","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10657584461509809047185519742966035356","series":"1.2.826.0.1.3680043.8.498.41459726027665864200670417725083238935","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10661650555947788912511478165000737040","series":"1.2.826.0.1.3680043.8.498.51224367562544770550607778759918101421","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10674673641872871321890995010616205087","series":"1.2.826.0.1.3680043.8.498.23247160794046439464701551526864300130","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10676253869668012791900504544106495589","series":"1.2.826.0.1.3680043.8.498.62470371377203367563768335291483540698","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10686407053180376367066321489809395592","series":"1.2.826.0.1.3680043.8.498.87027045537634189229233868802694678419","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10715956446507906169485719013515169244","series":"1.2.826.0.1.3680043.8.498.24798410035008900772717543940339253672","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080","series":"1.2.826.0.1.3680043.8.498.71212058308973787784071375381728596541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10650751856837517181090589171468851374","series":"1.2.826.0.1.3680043.8.498.11020709800412337373031465812752146821","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10699348333280906956391277892869999716","series":"1.2.826.0.1.3680043.8.498.38682049893333823113099736637625051102","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10736346409444614168571281812134757732","series":"1.2.826.0.1.3680043.8.498.12603138509227972416136242592110244995","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10729725525284633011179038668107963557","series":"1.2.826.0.1.3680043.8.498.48377584831792562024468929675955745572","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10739653650865735867752564520952904802","series":"1.2.826.0.1.3680043.8.498.10942461904183210272103699881285131377","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10726624821670417064567481545750543282","series":"1.2.826.0.1.3680043.8.498.99960262147452532931792773145775174384","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10756245293186761042431460535210737327","series":"1.2.826.0.1.3680043.8.498.11472076706249107993786646114442808583","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10748806810865802591313438958975301375","series":"1.2.826.0.1.3680043.8.498.59835176827598675828740113097303781913","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10762741411077456296656623422020928762","series":"1.2.826.0.1.3680043.8.498.88613225461250490619871769447186164572","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10760847155803463653759690525160815826","series":"1.2.826.0.1.3680043.8.498.11634002969367898745876429892053351796","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10768305343887790230989914438483009428","series":"1.2.826.0.1.3680043.8.498.39536662693908738028065822261191745642","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10783501127779906316117667790784040770","series":"1.2.826.0.1.3680043.8.498.84744523579847593352579274247888005350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10777331080659372520952742261140923555","series":"1.2.826.0.1.3680043.8.498.70714881625944686453974789098637689717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10770877369291591553624215762368014357","series":"1.2.826.0.1.3680043.8.498.25573131331180306717123935696050392797","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10751346650129497348457481341700957445","series":"1.2.826.0.1.3680043.8.498.37447209175207308675029027317449518955","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10808663384878038371710260386807468588","series":"1.2.826.0.1.3680043.8.498.18822579387269360700711574326558520541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10802653832050736637820225444196833450","series":"1.2.826.0.1.3680043.8.498.10902309271785780903009242072477327622","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10829579914809383212986717269028170263","series":"1.2.826.0.1.3680043.8.498.47230705940847980836862238609803916019","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10835889400414225749339608096119636738","series":"1.2.826.0.1.3680043.8.498.91812583922265019574285944405820522364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10818980124981334596808438908811623528","series":"1.2.826.0.1.3680043.8.498.32313617084861986764034097455265452407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10843917293035279597622029625809500972","series":"1.2.826.0.1.3680043.8.498.32272734202654093192896786802731816876","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10842906919708856642000130208791281263","series":"1.2.826.0.1.3680043.8.498.70812696834097098874616664473334967936","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10852517092524464338210061305792376636","series":"1.2.826.0.1.3680043.8.498.83438674730395968108311222520048994831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10850742579000782417673172155504794301","series":"1.2.826.0.1.3680043.8.498.85269246959179815285767515178965737876","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10854223557359336882116168394037581766","series":"1.2.826.0.1.3680043.8.498.75116070091968307886032230843296723873","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10862481204237376580557516731952961213","series":"1.2.826.0.1.3680043.8.498.63364197589900765731023792551632908470","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10884987036948550523442514147204321748","series":"1.2.826.0.1.3680043.8.498.44862956891135202690079439236238171850","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10876177342063734507664188801203192617","series":"1.2.826.0.1.3680043.8.498.92463453066307270178961393649259400682","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10896245499713818848442117831482755673","series":"1.2.826.0.1.3680043.8.498.12797443673236930975220436155240348846","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10917606677895195837968927716728385881","series":"1.2.826.0.1.3680043.8.498.13045332286239299804742518239931048251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10890699078723972693302029683905114834","series":"1.2.826.0.1.3680043.8.498.99084327872206066550067402621196598033","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10857485602186206735293765836065270655","series":"1.2.826.0.1.3680043.8.498.74431192163106040506345719450364424557","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10903003118372337016190215211789983264","series":"1.2.826.0.1.3680043.8.498.82796750257138652366597580704627332569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10924685588962763768106617368647678902","series":"1.2.826.0.1.3680043.8.498.11055208927178096623715671958092370751","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10919665962088003807153039302898262235","series":"1.2.826.0.1.3680043.8.498.67922870009354031704812995034985923566","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10927716750119165925963563081261605359","series":"1.2.826.0.1.3680043.8.498.10715532741981405984762674590482771651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10931111730564885182398163970692599398","series":"1.2.826.0.1.3680043.8.498.56890150707652366259277436073998436960","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10933003998197427979325590791540457658","series":"1.2.826.0.1.3680043.8.498.89353978859200722680282155538207557264","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10938719716535758543863717421521822626","series":"1.2.826.0.1.3680043.8.498.10340991516756845021228509420537433828","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10951328287594724738476663200798896670","series":"1.2.826.0.1.3680043.8.498.23874901446318550372688392717199915987","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10960770430559425421609885784366918575","series":"1.2.826.0.1.3680043.8.498.10263657348351241577341013406449965514","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10965573028231658307306828073340279415","series":"1.2.826.0.1.3680043.8.498.67235052812284911073248085414863632772","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10979105661314213697886164132444977106","series":"1.2.826.0.1.3680043.8.498.72628343424479460231481681550855067629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10983416091338854252058678062116243962","series":"1.2.826.0.1.3680043.8.498.11919315236035687516961955980068208408","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10937504474673840994058935502377258768","series":"1.2.826.0.1.3680043.8.498.42531309363032085435419792637210613938","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994554560803816407362754025763036100","series":"1.2.826.0.1.3680043.8.498.97402954713203934027166075172322937236","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10998405409474759941744305665540376339","series":"1.2.826.0.1.3680043.8.498.98108336645013977100056131173594421075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994854354532517072359864903910225174","series":"1.2.826.0.1.3680043.8.498.28941279603892704447348805971872761009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11007527194601989039620198927241696256","series":"1.2.826.0.1.3680043.8.498.36026175253308207971026568249768281456","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11011061033863592500429481719734134286","series":"1.2.826.0.1.3680043.8.498.59212444311121427152789813716263135767","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10995980880227536691827030731644023480","series":"1.2.826.0.1.3680043.8.498.70610524876150888552633131657190533763","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11043926695471016727738735328317975282","series":"1.2.826.0.1.3680043.8.498.57462365302372032360924466110473295034","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11040296449332764126530525726037541310","series":"1.2.826.0.1.3680043.8.498.26443599583584344176376757590396667877","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11049882927267755769995376553330493298","series":"1.2.826.0.1.3680043.8.498.54430662917765706706366356262726693442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11043081829401721095803441219553837608","series":"1.2.826.0.1.3680043.8.498.15575148791689906749155515639520370018","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11057883757434391643878168168845592475","series":"1.2.826.0.1.3680043.8.498.42336455299410288049765595714665192973","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11052656375391331522834364601050945860","series":"1.2.826.0.1.3680043.8.498.19413229478592056223228550937553961120","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11070900122260301212131296598983372089","series":"1.2.826.0.1.3680043.8.498.54453466763416462206996335236143653002","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11084010833221855396433326769153950681","series":"1.2.826.0.1.3680043.8.498.18489043351575480905003206899953535754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11079556243110299049578981210837702995","series":"1.2.826.0.1.3680043.8.498.82657823019792009830186985948277013186","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11091958567356981151034727948765010881","series":"1.2.826.0.1.3680043.8.498.10834164999259035141561523209903199560","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11064977170395595339882840534354229547","series":"1.2.826.0.1.3680043.8.498.95899025873769620503579364238048246532","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11111022632180183697497694737350901427","series":"1.2.826.0.1.3680043.8.498.12806007387283483960089642565569078735","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11119324966854127911591363740049886324","series":"1.2.826.0.1.3680043.8.498.73470752669789179862757403155304721822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11126383719852491587081347886000714259","series":"1.2.826.0.1.3680043.8.498.51050160313385167183212336737677950491","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11129751904857491327872353316480050290","series":"1.2.826.0.1.3680043.8.498.52933658544984806032332845440403330923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11123446342116837882811834332727990911","series":"1.2.826.0.1.3680043.8.498.11415951244431742755894570586986414642","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11135115047200279264914390528987153419","series":"1.2.826.0.1.3680043.8.498.10341114575649140662688968041392278844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11149530279799223497509928655010867960","series":"1.2.826.0.1.3680043.8.498.10749223286829806719253927186524623097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11145632111083286399683467139301564425","series":"1.2.826.0.1.3680043.8.498.21627738508661115912350573245413168879","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11150597567791052354257165699079348456","series":"1.2.826.0.1.3680043.8.498.12322997316383985816050208157048115259","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11103894265181149878864749992746236768","series":"1.2.826.0.1.3680043.8.498.98869393187117886929302736287522036752","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11152585968424897353533125085307104033","series":"1.2.826.0.1.3680043.8.498.27354094571597281567661550362861625952","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11156884483511690263238401598901614096","series":"1.2.826.0.1.3680043.8.498.46208887186138795149206878273927492535","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11183915112115196028799391678356464392","series":"1.2.826.0.1.3680043.8.498.70457245305382710844626032511291571927","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11192287547210248515805146368662883131","series":"1.2.826.0.1.3680043.8.498.12258672811607867127738435450432797701","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11173951743276817877673759975853741079","series":"1.2.826.0.1.3680043.8.498.12749801957372974323495869318158185422","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11201741528059502111642265469347871878","series":"1.2.826.0.1.3680043.8.498.12667198095796557437928234362388060048","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11208107516750574657838495761140109258","series":"1.2.826.0.1.3680043.8.498.11736508912380007255276556356590136998","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11210361780779474361817208167009080083","series":"1.2.826.0.1.3680043.8.498.11874005327436999463204950933612097829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11212667888429022863778740306223585731","series":"1.2.826.0.1.3680043.8.498.73069648459520276783474359846653342459","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11223691171233118900160265287691267884","series":"1.2.826.0.1.3680043.8.498.10473540547191433687694601151903395338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11219813299300915347039206446596901205","series":"1.2.826.0.1.3680043.8.498.11844014608624775794519273449658118385","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11190292536746235063218740991532759863","series":"1.2.826.0.1.3680043.8.498.22139886809385623191975162211071150721","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11229566047228169850473554486278603754","series":"1.2.826.0.1.3680043.8.498.83448427760610848396038387142125548246","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11232674915186872899189440191805868376","series":"1.2.826.0.1.3680043.8.498.31173607033908303580716566098811920607","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11218351123386367961865331646414911627","series":"1.2.826.0.1.3680043.8.498.43785816289831494855680741236068312120","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11257168136762043265165109052826442974","series":"1.2.826.0.1.3680043.8.498.89002777709198463092083185859369969570","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11249981834802444119817546155435759007","series":"1.2.826.0.1.3680043.8.498.33768242062565931148491037346265910420","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11260248089108026217929330361529065683","series":"1.2.826.0.1.3680043.8.498.13350918879216090241304662816359514795","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11264140845869814478794346490273061499","series":"1.2.826.0.1.3680043.8.498.47957804427037022298837789578316558338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11270845320057097450047912107461684885","series":"1.2.826.0.1.3680043.8.498.49230156672091147857185307812943623674","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11277078989123345865112116288626381353","series":"1.2.826.0.1.3680043.8.498.10644381080495292668789644476916293728","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11287173314232027941677698384842764593","series":"1.2.826.0.1.3680043.8.498.97727977007706440977998910031122085491","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11291875173369366854889507729635968509","series":"1.2.826.0.1.3680043.8.498.80282726740800135557585073407167462736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11295899132697862632905938037090529644","series":"1.2.826.0.1.3680043.8.498.79061417113084232928104592725683475721","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11298099571581989006135838942599921393","series":"1.2.826.0.1.3680043.8.498.13164727315204712699467894629040467681","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11282162920338602001456704018617788412","series":"1.2.826.0.1.3680043.8.498.10096877589530261930201157322108976095","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11307870660160563329996058267938398411","series":"1.2.826.0.1.3680043.8.498.47000828425417658135203368252638873842","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11334486185870324597335039163116965763","series":"1.2.826.0.1.3680043.8.498.21419291584723346033444350816579736828","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11314687816975146963598226343179928616","series":"1.2.826.0.1.3680043.8.498.29393702732276002945262706598485766066","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11276674006815021828943434205349220551","series":"1.2.826.0.1.3680043.8.498.71531833462787446311717090953342441749","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11316091466685605758061778045311969672","series":"1.2.826.0.1.3680043.8.498.54867374618942840055090661031449521891","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11337616353935430160375524217870968575","series":"1.2.826.0.1.3680043.8.498.97407224638418001514936053961805787673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11340429352365972769264064439788186260","series":"1.2.826.0.1.3680043.8.498.19356776260199146204837330613155326278","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11347466940974045182979684641300336177","series":"1.2.826.0.1.3680043.8.498.49052384713593462925981483279472545822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11353976221682532422630231842008146722","series":"1.2.826.0.1.3680043.8.498.53691548752849718343100880011565958153","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11369844885381156110080055326139875897","series":"1.2.826.0.1.3680043.8.498.26704922539513080082675845588574338137","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11385845157446557781575804962818981994","series":"1.2.826.0.1.3680043.8.498.40578720090280485592794345356779543757","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238","series":"1.2.826.0.1.3680043.8.498.38450243001981446118642153746249112051","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11395349736725035422736613041202251830","series":"1.2.826.0.1.3680043.8.498.12430834926865897110723738398925233805","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11402325536521359105285323485008939602","series":"1.2.826.0.1.3680043.8.498.61725663844680992053964431506647935219","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11391095418955817124325938917797687542","series":"1.2.826.0.1.3680043.8.498.84631574742267159099180444272229944285","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11406381777983128446583783158882827226","series":"1.2.826.0.1.3680043.8.498.82658517802894400230895366622562091255","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11425019165071014217290822852865654127","series":"1.2.826.0.1.3680043.8.498.11466737744680233493064074769420938642","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11414215447313658600502835138277811260","series":"1.2.826.0.1.3680043.8.498.51280662836201853898250384089387648730","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11428482061412578159115168314865188597","series":"1.2.826.0.1.3680043.8.498.15193333061892769887718612159505533288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11440214477653629899108986507154583794","series":"1.2.826.0.1.3680043.8.498.45901428374073786927873108127500246829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11441489276425334980205891153179893797","series":"1.2.826.0.1.3680043.8.498.33660368055709040321305120506164889439","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11448326350892044215419380818679679579","series":"1.2.826.0.1.3680043.8.498.16677935992453446443072656572484976779","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11467955500991327547315970867313333592","series":"1.2.826.0.1.3680043.8.498.31840433025096098808733103347248551915","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11455388290952699607973723923118848769","series":"1.2.826.0.1.3680043.8.498.55616261330690566232810198938234119112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11451448731901996559834481853733404099","series":"1.2.826.0.1.3680043.8.498.13209595503436144849689636548046859626","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197","series":"1.2.826.0.1.3680043.8.498.12914861826968650398682345293674689557","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11471148332557464549579838595592736047","series":"1.2.826.0.1.3680043.8.498.34513696711272547263323702215340162367","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11489881286208501558797333161632486099","series":"1.2.826.0.1.3680043.8.498.56019961138123934300068753549912844044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11497761063752461132736045914751707046","series":"1.2.826.0.1.3680043.8.498.11828227454221266337124062658451875174","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11477000011453869751231224714283666904","series":"1.2.826.0.1.3680043.8.498.79080307107333842850552427612247623274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11501070108039994411346060488666341726","series":"1.2.826.0.1.3680043.8.498.10272233408824746822209114908680885633","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11506642732332295874076832453198901115","series":"1.2.826.0.1.3680043.8.498.23012285169187003454581778719951147121","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11482325994550095339373157256174349858","series":"1.2.826.0.1.3680043.8.498.56007409742460730757596880793004969471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11516498203727058146656616413699979433","series":"1.2.826.0.1.3680043.8.498.40676256367536615983903698604743421040","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11508340006652282767645009271305921426","series":"1.2.826.0.1.3680043.8.498.10028023470330364812890471248551225306","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11529405479134457001825038259936796388","series":"1.2.826.0.1.3680043.8.498.73752551051086590118640024543721862523","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11532517984322509215429681648257299355","series":"1.2.826.0.1.3680043.8.498.10104470825042832140370495772349619437","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11520508017843738484108845144418061602","series":"1.2.826.0.1.3680043.8.498.10537905482904599135655877222272214934","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11498505068787084440256890724559761815","series":"1.2.826.0.1.3680043.8.498.37470739625349293073696680428925155237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11545541207342029419461179316053148974","series":"1.2.826.0.1.3680043.8.498.13245654908835864829797563038401066486","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11542298537902846156898976374522470463","series":"1.2.826.0.1.3680043.8.498.10717477909616914451702415154648326728","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11547192037231064935759879673500408149","series":"1.2.826.0.1.3680043.8.498.53477775631504123867430384868070061643","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11551770225828320378339882849245348148","series":"1.2.826.0.1.3680043.8.498.62542041665006433620920946905852480986","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11548839502674800051973767797346682106","series":"1.2.826.0.1.3680043.8.498.36372604547672946956177617144321969338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11569601172991867969672947467467392425","series":"1.2.826.0.1.3680043.8.498.10013168686774212100388186060914556274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11574480940878399452466699512074769686","series":"1.2.826.0.1.3680043.8.498.86887228898645329518407883003443158371","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11561519302314146020052258325459371962","series":"1.2.826.0.1.3680043.8.498.61924256142390193669938993812485050327","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11583098598455610459898849451117846552","series":"1.2.826.0.1.3680043.8.498.12731052053608982650396044637572604459","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11587989960888698769912899464470116193","series":"1.2.826.0.1.3680043.8.498.67599935957622051533829293650249121345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11577630811395858629711985689258647804","series":"1.2.826.0.1.3680043.8.498.17217908849136537290448622733260703778","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11592056376890319271545710970155310824","series":"1.2.826.0.1.3680043.8.498.11983365070209498572245326199477945537","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11596866241403598097171345671773992202","series":"1.2.826.0.1.3680043.8.498.63714359393851835865126331689434668921","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11595338216297251382188195835520492789","series":"1.2.826.0.1.3680043.8.498.17505962601960015648185436568973236643","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11599387556469808488160280438305325860","series":"1.2.826.0.1.3680043.8.498.78388115348069006526210891710350949884","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11604298807403466096392316117803661479","series":"1.2.826.0.1.3680043.8.498.59353327480788491364091219525597036743","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11610176134207131408691324676304640501","series":"1.2.826.0.1.3680043.8.498.66427064110530916302188286838899384231","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11613637647831801354398273334552399874","series":"1.2.826.0.1.3680043.8.498.11453584867368351773197729939361077885","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11641274367861998748475337224297076397","series":"1.2.826.0.1.3680043.8.498.22761904023226593518646746145132829238","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11642228346306175407288158335830376197","series":"1.2.826.0.1.3680043.8.498.74509109297784341428294589321161791869","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11625598825018422177242262569319579378","series":"1.2.826.0.1.3680043.8.498.10958828373915910081624857291676385831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11646465814751330102194657154003685957","series":"1.2.826.0.1.3680043.8.498.11813444535260394738083629164638342706","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11651755040149549966484720727140190853","series":"1.2.826.0.1.3680043.8.498.92635077996293735124876819360660219982","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11659861228342609979638095779959188093","series":"1.2.826.0.1.3680043.8.498.13270709852735850013894611059961663138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11669120316759278775792190599036288128","series":"1.2.826.0.1.3680043.8.498.26488967990512921120133618718595473915","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11654064547593365398103107850228629066","series":"1.2.826.0.1.3680043.8.498.60298992056961218190928129503624287804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11679044101697033975478370934058085847","series":"1.2.826.0.1.3680043.8.498.32682907972361553476494033976263816809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11682243736539231948790371952356165171","series":"1.2.826.0.1.3680043.8.498.86786423842981088369173119201432799188","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11689820638581431730501617718741779835","series":"1.2.826.0.1.3680043.8.498.30156440426404040405021093797787083907","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11706378290185110677863653788588307917","series":"1.2.826.0.1.3680043.8.498.60770792538816188656064595920924631901","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11719502398697390809777045891822044637","series":"1.2.826.0.1.3680043.8.498.12199571219898345032499941911016057026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11720664166420613534218395854543939928","series":"1.2.826.0.1.3680043.8.498.75614659816322192393677752654313364147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11723286647024326363349453996818420374","series":"1.2.826.0.1.3680043.8.498.11466631424956940350107476245680565815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11716605112302071535035323666308656381","series":"1.2.826.0.1.3680043.8.498.35410598338450519952118463454877564309","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11725843001854130477287484865097535511","series":"1.2.826.0.1.3680043.8.498.13135723618235592724757253241546766753","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11738872955471637985947490962985266753","series":"1.2.826.0.1.3680043.8.498.26287728381094133720832434033501259444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11695097367686121784104505410611821860","series":"1.2.826.0.1.3680043.8.498.24869180595160857654818968574082954311","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11752091432368750506312921028559788225","series":"1.2.826.0.1.3680043.8.498.35333870836159296429634806417957802551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11770690655120964721059321709525979950","series":"1.2.826.0.1.3680043.8.498.61931553595536312431382776495464220628","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11762352496600661264881556836248251400","series":"1.2.826.0.1.3680043.8.498.54142552344503333276654461308359618417","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11775352515236862473778507708524782015","series":"1.2.826.0.1.3680043.8.498.50134022120446753229708547785929280886","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11785138333495400389510795613252286069","series":"1.2.826.0.1.3680043.8.498.73920437843460669737953963005012193157","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11786582806577871618290342375465595269","series":"1.2.826.0.1.3680043.8.498.47020386999226105716312355687540241246","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11791567830371683660578052737702299124","series":"1.2.826.0.1.3680043.8.498.42149203628164150568880839675609919499","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11804799183537166337413388415687675758","series":"1.2.826.0.1.3680043.8.498.20482180143728693770410526665771513819","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11799272823743145498601173910069737855","series":"1.2.826.0.1.3680043.8.498.28915416749927980184608336323050983756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11814923363609140180293604689814785969","series":"1.2.826.0.1.3680043.8.498.90229942792317759958593938609531070850","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11807494462283790976935891042637502434","series":"1.2.826.0.1.3680043.8.498.12652200606231849842121225232760031272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11779387602988880691008088661075117406","series":"1.2.826.0.1.3680043.8.498.80226968363695214745776470997017181649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11830915612184936576804778995164042200","series":"1.2.826.0.1.3680043.8.498.36898043451662084138153405348162227501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11796547950032543541071896156852778697","series":"1.2.826.0.1.3680043.8.498.10069513479481433132489554141100027770","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11848828525319749834876185415845381479","series":"1.2.826.0.1.3680043.8.498.13218610088686100205117198051027201191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604","series":"1.2.826.0.1.3680043.8.498.96460311887938476084320882783854369448","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11863890747360341056211511874563792862","series":"1.2.826.0.1.3680043.8.498.47241939890604341530405125019748105372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11868154141292577776796381772987549697","series":"1.2.826.0.1.3680043.8.498.43180105778486656621656592937308956717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11880385031047315524154761508473409762","series":"1.2.826.0.1.3680043.8.498.34484325801268820618795461261029216978","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11871142388984150456249558619289234011","series":"1.2.826.0.1.3680043.8.498.35637777476707659871659169588184134922","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11897083656099489353496670273111097794","series":"1.2.826.0.1.3680043.8.498.11492946892066555765464590760279522708","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11887143891500181344364122992066593229","series":"1.2.826.0.1.3680043.8.498.36236879841460975226271585200937983981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11906266535122698149967624908585010734","series":"1.2.826.0.1.3680043.8.498.12707869102152332708128286340532048587","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11911688823908554151914734938172012708","series":"1.2.826.0.1.3680043.8.498.10001344172545964778532427094848882081","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11901474252960403779405194385221103673","series":"1.2.826.0.1.3680043.8.498.87713280404445747051838844877603533299","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11904156585987383639134967886361403369","series":"1.2.826.0.1.3680043.8.498.33231897399023621668807688285779070948","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581","series":"1.2.826.0.1.3680043.8.498.71284507741504130254010428167357512908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11912618147674655465071922267060032851","series":"1.2.826.0.1.3680043.8.498.43973876237487556863208472794813969162","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11923218013055841952315996726408812182","series":"1.2.826.0.1.3680043.8.498.52761229604020904804130025429657006566","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11926412946593048423086380882538180028","series":"1.2.826.0.1.3680043.8.498.64358086893036637274674116716627614756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11934483804567594363179973225449686890","series":"1.2.826.0.1.3680043.8.498.11217055425962450114317672996174603331","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11935893582662912138522962760478513770","series":"1.2.826.0.1.3680043.8.498.13314661652844499619208264014218803660","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11938864543152001817866013174907274014","series":"1.2.826.0.1.3680043.8.498.10851217338035309822470765102634698843","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11920026019322352546372153111621676066","series":"1.2.826.0.1.3680043.8.498.62437327371717523941799697041368675265","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11954711639198972958656966617027374769","series":"1.2.826.0.1.3680043.8.498.12574186653905857344225897523856059460","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11964826300307458382604479841253457304","series":"1.2.826.0.1.3680043.8.498.22855982724051643919486009774830601373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11977465062890611952998635094033472451","series":"1.2.826.0.1.3680043.8.498.12129679189683182783588630115465658804","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11979906027101464432999416052202616662","series":"1.2.826.0.1.3680043.8.498.90652865350281159438710695675546067649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11980805687949950504930975019161587970","series":"1.2.826.0.1.3680043.8.498.88508209423883286521220167855731363979","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11988551795896617815207749665102049499","series":"1.2.826.0.1.3680043.8.498.13378295628452891721504175223732364160","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11992172807590757302730513420827032389","series":"1.2.826.0.1.3680043.8.498.31445224797992729756505560973797274677","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12005327370242618323737652892230673473","series":"1.2.826.0.1.3680043.8.498.12556033271812146203169386429759673696","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12007335335641704192494936946829650792","series":"1.2.826.0.1.3680043.8.498.13153837254067338229673605822327557581","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12015406188790105426257083591807220760","series":"1.2.826.0.1.3680043.8.498.99491484753875155708841518917210962011","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11957476321850340968244508976719889277","series":"1.2.826.0.1.3680043.8.498.59448122125493399897964180771803432364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12010150865087936209475730225377000084","series":"1.2.826.0.1.3680043.8.498.95872619895796427149919187015465293317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12031315219947933845048665397149229969","series":"1.2.826.0.1.3680043.8.498.53727037480715534914377133980120699612","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12036404333388388978317335770611404933","series":"1.2.826.0.1.3680043.8.498.27534058922725893623273561492253123848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12041491173612341571223481616114183729","series":"1.2.826.0.1.3680043.8.498.12022579093849429431766089821664550054","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12054099347723802635102026371218576198","series":"1.2.826.0.1.3680043.8.498.10560148514295836473281434254568913454","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12064822036518837577612019712902340953","series":"1.2.826.0.1.3680043.8.498.99077228020457212663308522930032277692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12050415099056648944934039580585859179","series":"1.2.826.0.1.3680043.8.498.12211076836574088212391995778520843321","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12073401429398042797505820525732905336","series":"1.2.826.0.1.3680043.8.498.58321122336212662809258856781916120744","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12095779329738382018467257299597656491","series":"1.2.826.0.1.3680043.8.498.85455570635861546075428779379900607944","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12106775587428595108771304868101362618","series":"1.2.826.0.1.3680043.8.498.11808750556072786023523762150189850552","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12100254887364913022980951672127944515","series":"1.2.826.0.1.3680043.8.498.84839789536079013731735167942481940814","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12110510829608190094905588610131531103","series":"1.2.826.0.1.3680043.8.498.82489281899440205334065589455264631718","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12130605331118953242364684736005454581","series":"1.2.826.0.1.3680043.8.498.47221704273177959416320662780796940655","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12068674252637046261029360607215822017","series":"1.2.826.0.1.3680043.8.498.61928105796790378289636235386111839682","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12131866860554871145836544571379633595","series":"1.2.826.0.1.3680043.8.498.96195328937473777043009281743778643703","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12139829070290803654867114374799823711","series":"1.2.826.0.1.3680043.8.498.10532254454926395448894865395739820280","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12150502202634860887314001818431257267","series":"1.2.826.0.1.3680043.8.498.45715861120990015689567726343201056804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12135456587022560662140325717612726990","series":"1.2.826.0.1.3680043.8.498.17306288816329005490547175759554574305","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12166081590548185636776183599317715045","series":"1.2.826.0.1.3680043.8.498.50932285257124209622887179608479515207","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12168045829469202472124733625621943469","series":"1.2.826.0.1.3680043.8.498.91283996788859384654547158187372776901","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12171469604213841534511019872648327851","series":"1.2.826.0.1.3680043.8.498.23702354555155913228311034352790474089","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12154188739734505793699974626991872621","series":"1.2.826.0.1.3680043.8.498.74809988971135517791309813219752918243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12160065317167766993207036411434801297","series":"1.2.826.0.1.3680043.8.498.67490878230022012800529257936017364028","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12176466551281285838907773521868954581","series":"1.2.826.0.1.3680043.8.498.98971547363766327477578333905041086202","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12178921459424917661210327999672203211","series":"1.2.826.0.1.3680043.8.498.11095676664461968627994537962137028040","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12186494601308291174548814564387764515","series":"1.2.826.0.1.3680043.8.498.11280063003956449674385599980582240272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12192455583120681639582915365754780698","series":"1.2.826.0.1.3680043.8.498.17482732808116238190223515042208453297","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12201598444731259697380878112783653796","series":"1.2.826.0.1.3680043.8.498.13146690352677049123645196993014010887","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12182309401920763498667366352681074302","series":"1.2.826.0.1.3680043.8.498.95646735930361799495526781780146009946","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12207279366369605175255203969558548753","series":"1.2.826.0.1.3680043.8.498.12665019234626340890985583341522104817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12215902620611618183321984433114460986","series":"1.2.826.0.1.3680043.8.498.20177697011920872965734146207374555895","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12221892384948220776502422531414659289","series":"1.2.826.0.1.3680043.8.498.52882037988679812102173383325914666004","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12229628534316014336864328244217875465","series":"1.2.826.0.1.3680043.8.498.99790176167836919302394043265462933942","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12243764506921404196245986681613075136","series":"1.2.826.0.1.3680043.8.498.99985587288484035337375475542090751233","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12195090472992304712555822541659656360","series":"1.2.826.0.1.3680043.8.498.12682525940036380732605091818935114871","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12241413500128553133597113709525575494","series":"1.2.826.0.1.3680043.8.498.12479207878945563513431080447285049395","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12238753439953831519247789198174756615","series":"1.2.826.0.1.3680043.8.498.31791541341122039812964918816329369071","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12246086238999179953298261549112671014","series":"1.2.826.0.1.3680043.8.498.97360740996837418843857551456639667621","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12257484608743859456600273083454447341","series":"1.2.826.0.1.3680043.8.498.11062380002379767131614317198539036810","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12260661730837055872432103268905182876","series":"1.2.826.0.1.3680043.8.498.48600589356178590922880553961979229314","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12267944258766605501055560486655358039","series":"1.2.826.0.1.3680043.8.498.10242120377092561768083778499470457344","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12266133889557145601371064833389800065","series":"1.2.826.0.1.3680043.8.498.41704270097079303381219311983652426246","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12273356591019341319494035708698721367","series":"1.2.826.0.1.3680043.8.498.98612858546581216405641163370200426456","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12256365473385257798561212457071712327","series":"1.2.826.0.1.3680043.8.498.11622244427668017926253508131840064660","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12280407492991430179266369843997653205","series":"1.2.826.0.1.3680043.8.498.31963879241611751549090182800025823575","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12274801503555668613400469130773239683","series":"1.2.826.0.1.3680043.8.498.37091287689764523127897264524393147100","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12264607993101717201407955391437731134","series":"1.2.826.0.1.3680043.8.498.65287066394973657808617314266381023251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12286894057773964720268768529280193085","series":"1.2.826.0.1.3680043.8.498.13380335532128404452665739508984932272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12296624595750602443033689366743784387","series":"1.2.826.0.1.3680043.8.498.41656996891388396574093093245744071846","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12285177271228289867211553575739205132","series":"1.2.826.0.1.3680043.8.498.13661421102001431981857407141453627040","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12295284437770380354725533247473017729","series":"1.2.826.0.1.3680043.8.498.39614418745356717231605133083393759599","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12314729913655626716039376606973474150","series":"1.2.826.0.1.3680043.8.498.25521359311635144562628899143377322773","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12301945990161670321760708884248055936","series":"1.2.826.0.1.3680043.8.498.36806949566714966033522555357592448321","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12305281328019552390691110900015627668","series":"1.2.826.0.1.3680043.8.498.13034441341147837590851333927188302472","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12307356432736970350794033171174403632","series":"1.2.826.0.1.3680043.8.498.11281601290439367917286029508098881881","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12331841096593105748526634055968654914","series":"1.2.826.0.1.3680043.8.498.12263681946668052482489440799400000530","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12333996498811456012744466468462396081","series":"1.2.826.0.1.3680043.8.498.32376822456909015666885360106431481179","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12348352904452887226227118450588511578","series":"1.2.826.0.1.3680043.8.498.80380541195740357933440904517351617271","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12326930847104022196329390608780066947","series":"1.2.826.0.1.3680043.8.498.12193567937367537454939621656609592416","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12355011301785011370279005103251815187","series":"1.2.826.0.1.3680043.8.498.12087797894403626673894728406073856001","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12348960754791091583201828167607825033","series":"1.2.826.0.1.3680043.8.498.90189491461842325448376265371615017969","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12345064031074704673835101507426902246","series":"1.2.826.0.1.3680043.8.498.82232643011613587776822463517223473663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12352132666379687393012255589355771387","series":"1.2.826.0.1.3680043.8.498.91589349405870709360051259677617120452","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12361144153852256276011948786791830435","series":"1.2.826.0.1.3680043.8.498.61393663114382588886557487613261740928","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12374183329974116022606010915169117790","series":"1.2.826.0.1.3680043.8.498.58245619998479745866904330780307604445","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12366238437963147962850341556508931973","series":"1.2.826.0.1.3680043.8.498.44142214287079494960741947214503494009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12386255059403233724886571725482313876","series":"1.2.826.0.1.3680043.8.498.13190848404463831037321369683329807672","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12381909868368799280462187344958674397","series":"1.2.826.0.1.3680043.8.498.53927282299890996683318603925211102998","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12379570725733005682132316461218878636","series":"1.2.826.0.1.3680043.8.498.98386972751180337251355880451728780753","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12390186846317493876612228407734068726","series":"1.2.826.0.1.3680043.8.498.13041006473096998362541393711271551419","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12406043058701411628555010402350032739","series":"1.2.826.0.1.3680043.8.498.92393028266532455533765289460805789323","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408314059030096510457440879771212020","series":"1.2.826.0.1.3680043.8.498.52163717785860879123446817962657370191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12411396124304166984011280826756258628","series":"1.2.826.0.1.3680043.8.498.11041626577212498076180529946170291437","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12426381920136706157912507136730458010","series":"1.2.826.0.1.3680043.8.498.74076122648921937894848517916369154647","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12387241932914995417493041212958089910","series":"1.2.826.0.1.3680043.8.498.98939755784986100778677781047172526465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12437154858167700438432674983971817217","series":"1.2.826.0.1.3680043.8.498.13190130174417231523654547881587195562","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12432974885645948103184031984290883090","series":"1.2.826.0.1.3680043.8.498.44667901025473459103853743161401571081","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12438775479300781065075835270622538230","series":"1.2.826.0.1.3680043.8.498.57115378898402656841938682840785769009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12462703906850759490077163612851243080","series":"1.2.826.0.1.3680043.8.498.11978315170964233536520130318258876503","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12450858400030959723767712312663176092","series":"1.2.826.0.1.3680043.8.498.11526961444506381543155555046798334450","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408945963789246646229147558387793270","series":"1.2.826.0.1.3680043.8.498.11294879991391997344789265876107393644","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12481913908649439189738319460342358360","series":"1.2.826.0.1.3680043.8.498.99689985593412192688376667949616844599","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12473533314363654795650919604551685896","series":"1.2.826.0.1.3680043.8.498.10662999273921684120276899114140417448","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12484700311993345120015691857089576898","series":"1.2.826.0.1.3680043.8.498.75940149982100115964289949729201591790","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12489752598467506094826904655029553699","series":"1.2.826.0.1.3680043.8.498.77007147729918924925066163932947995536","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12491796764641306046176352066215240074","series":"1.2.826.0.1.3680043.8.498.53915388308848728651396649604946112901","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12491395125580387847235487841392723348","series":"1.2.826.0.1.3680043.8.498.62633647128517686732526942255662697405","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12501925425699863373513819814520995919","series":"1.2.826.0.1.3680043.8.498.12320101971011590040393044930016612133","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12497144849433082154223897356471317072","series":"1.2.826.0.1.3680043.8.498.87266952665219795800797785421356279112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12507957850103916926719601413279798696","series":"1.2.826.0.1.3680043.8.498.64009054822190539876510980455349282019","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12514357232100339485873453417784586592","series":"1.2.826.0.1.3680043.8.498.11050298643629265286087857184983253027","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12518297594498173981937681848554641851","series":"1.2.826.0.1.3680043.8.498.51210166181139095673054571177832306363","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12509944331769453135724043962410129357","series":"1.2.826.0.1.3680043.8.498.10508656100735574847807155638319534645","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12522620957277632939731546099134770342","series":"1.2.826.0.1.3680043.8.498.64496758779659273553525337994601403663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12533105585687352326867082376911240942","series":"1.2.826.0.1.3680043.8.498.76451732602010115046316815859970940697","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12526876471494728147635430816990398219","series":"1.2.826.0.1.3680043.8.498.17454881464018934436760011965566212823","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12525752631743097782762806859112218620","series":"1.2.826.0.1.3680043.8.498.55488397086375850552533323684561870317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12547308734950397389859712060653254111","series":"1.2.826.0.1.3680043.8.498.47337290545922212670144401860662493428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12568337973304376222940287737691898395","series":"1.2.826.0.1.3680043.8.498.11453474609693491411481319484170329879","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12553455350653672209409408595948190982","series":"1.2.826.0.1.3680043.8.498.57406702078601051782864779664353318922","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12559687473583953266143897736224215680","series":"1.2.826.0.1.3680043.8.498.80403581402260561238459876130882112678","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12570576337098834311171883034999327454","series":"1.2.826.0.1.3680043.8.498.95820781154415891769997810889894942600","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12577828789206467631166875572423539298","series":"1.2.826.0.1.3680043.8.498.12032186960327537843100962756039637340","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12552701911592685137269544109687472988","series":"1.2.826.0.1.3680043.8.498.88603941294997627371423929656153015565","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12580232188757225504172709417466515184","series":"1.2.826.0.1.3680043.8.498.22134575882004981137927401588931449167","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12574270269124650675934073761741661292","series":"1.2.826.0.1.3680043.8.498.13434230692165812727012540776257380073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12598627296066406747921546933391681918","series":"1.2.826.0.1.3680043.8.498.59474694716257683823663043288849543326","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973","series":"1.2.826.0.1.3680043.8.498.11119341733412446515084456525158577841","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12614097867016983076979065551985251584","series":"1.2.826.0.1.3680043.8.498.39265300686467901282056463671053712863","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12607659092357834912029375690348886159","series":"1.2.826.0.1.3680043.8.498.75297579259988973493932554142852438862","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12629372332421954368597330267888483345","series":"1.2.826.0.1.3680043.8.498.15350996715244077434881612364565108405","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12625250919597111075072308824921101918","series":"1.2.826.0.1.3680043.8.498.74196124277568210323283203867551418706","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12632084649033412400763943977604348172","series":"1.2.826.0.1.3680043.8.498.73325704415302645341730089845030935591","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12637591861595747322913829625646935491","series":"1.2.826.0.1.3680043.8.498.85256535738935965826359715886372724874","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12619895582462244920124420363095282842","series":"1.2.826.0.1.3680043.8.498.18755445171771574642694282278497530105","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12644400073937544865441360809610886635","series":"1.2.826.0.1.3680043.8.498.11045576656606769904599839922474016717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12653936535855858693516515692319733265","series":"1.2.826.0.1.3680043.8.498.61555237284369263110039698566846262118","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12658572494577556884626613972035441625","series":"1.2.826.0.1.3680043.8.498.10833991747666315114829377886113887764","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12668708646283106269005674386248135220","series":"1.2.826.0.1.3680043.8.498.18280374154128929454282810621768235404","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12674069717230126880288353271062347095","series":"1.2.826.0.1.3680043.8.498.81443531263342136772832402936706514240","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12640439197386154384526090653248054582","series":"1.2.826.0.1.3680043.8.498.95133801532870744073742578788133604274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12689874339887047433232628649886917274","series":"1.2.826.0.1.3680043.8.498.96890652071244179096837879553120405908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12676359576477452816896379524598090410","series":"1.2.826.0.1.3680043.8.498.32070626227873150622142355930905577089","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12697859103448296208701081683487338111","series":"1.2.826.0.1.3680043.8.498.10097242914443185772462346268255956363","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12690803821768205116702487219908820630","series":"1.2.826.0.1.3680043.8.498.22753445121041999578522834735708064403","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12701097867143790763644826659974337046","series":"1.2.826.0.1.3680043.8.498.97201305231344305809875745446705934057","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12715130507546263968763238702574191950","series":"1.2.826.0.1.3680043.8.498.10476052978860576366566708670392542276","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12709854414521475738435319009767427434","series":"1.2.826.0.1.3680043.8.498.12819671759571399478050654645611362386","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12700591955254168977239658917260628326","series":"1.2.826.0.1.3680043.8.498.42068331116015651482318423716122554633","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12727488524631885827798129418558343386","series":"1.2.826.0.1.3680043.8.498.10936627572040502009538635675572049963","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12719397512102459853075227679498459688","series":"1.2.826.0.1.3680043.8.498.32193924410879680452748154173544096594","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12717728703860353055063551162640833849","series":"1.2.826.0.1.3680043.8.498.12372304330687294354508640690740909628","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12724250000565012779853884787450109956","series":"1.2.826.0.1.3680043.8.498.26153385546262102991587610924301027743","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12731497178056855493473947157407264492","series":"1.2.826.0.1.3680043.8.498.69865184058804971315815142769051482779","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12754381146006839341912662615844567067","series":"1.2.826.0.1.3680043.8.498.16112135157393511305118403387719043388","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12762900657007480138202359874575403361","series":"1.2.826.0.1.3680043.8.498.41199897675090436072101942501555563986","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12769221625850663248477857087347890908","series":"1.2.826.0.1.3680043.8.498.19261027124134078986327642344057207120","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12775105605170844918620305197829216665","series":"1.2.826.0.1.3680043.8.498.88111636932053938014202881225522068446","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12771448321084229688831202344007889535","series":"1.2.826.0.1.3680043.8.498.25302986085127885822753226806459036993","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12761906443846433581503942530016459978","series":"1.2.826.0.1.3680043.8.498.83032560068042571487018844921524471473","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12785620442256422697216192462703273802","series":"1.2.826.0.1.3680043.8.498.74017352951013577673716137517663100325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12790542912560396272981117250277568532","series":"1.2.826.0.1.3680043.8.498.12151380225317193011096785072795607515","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12778920060586364664953350981285170949","series":"1.2.826.0.1.3680043.8.498.14893215073648912285752339211244297784","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12798419553759200632295966180228523212","series":"1.2.826.0.1.3680043.8.498.10896891967508619885496049345358439491","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12800130397185143653180509040746675358","series":"1.2.826.0.1.3680043.8.498.11805779128990728018332761183126635613","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12815132582382825980088023336277934996","series":"1.2.826.0.1.3680043.8.498.21244034621683285947090805668526677117","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12801643268939133682891906705186571010","series":"1.2.826.0.1.3680043.8.498.38948437851580840845426108750128759451","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12818010040786942613541254450332754046","series":"1.2.826.0.1.3680043.8.498.12377743860741679018567382576593930796","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12810074901642984174782853829149619043","series":"1.2.826.0.1.3680043.8.498.91805621513263013803427329028671197766","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12826828527176716109949704557150950675","series":"1.2.826.0.1.3680043.8.498.12524278821142005824207710125231941484","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12831769902826114242555323443809541052","series":"1.2.826.0.1.3680043.8.498.99713925352460368985647536465365184520","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12820051821104182459960036809430047569","series":"1.2.826.0.1.3680043.8.498.11412863545281292301039139501274452935","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12862782592303338109631225504988756108","series":"1.2.826.0.1.3680043.8.498.31586891647666321280994253136213123025","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12866937766198610064474201436305532400","series":"1.2.826.0.1.3680043.8.498.61765595978853136883068225840230704610","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12896740277483734090633132494285615502","series":"1.2.826.0.1.3680043.8.498.12230897518262674859174705692547158175","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12904190317996738774518258536997285832","series":"1.2.826.0.1.3680043.8.498.11504586581861092781000309621637542600","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12837030643153472711780894512565629507","series":"1.2.826.0.1.3680043.8.498.12170097656847802717226906986835232684","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12912436233176216254115594260643600155","series":"1.2.826.0.1.3680043.8.498.89728513777475526676213642603012503104","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12917504242141830416624247604384969664","series":"1.2.826.0.1.3680043.8.498.64012849777551751894813341846458320668","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12929915324982999692857962099866616265","series":"1.2.826.0.1.3680043.8.498.12337656742326571576598459960057461199","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12946163460972553911280889205685745877","series":"1.2.826.0.1.3680043.8.498.56670882570338095409312541663790860915","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12938500386086171684983091332136233036","series":"1.2.826.0.1.3680043.8.498.64378235991462857289117855137078342893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12949322114105642048256996295428966924","series":"1.2.826.0.1.3680043.8.498.12603542347674812451301900557498572669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12957752106697316015996777476239762111","series":"1.2.826.0.1.3680043.8.498.19539108257598770084498890598864085375","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12954163390367261226110152166469167935","series":"1.2.826.0.1.3680043.8.498.56838922864992268158939592583010631805","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12978172997053500029791786614906570285","series":"1.2.826.0.1.3680043.8.498.35072817235108436209429595800955663975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12968535455086298222684461157825070395","series":"1.2.826.0.1.3680043.8.498.89259340599455154858465090590047033149","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12986028147682493711912242081052787346","series":"1.2.826.0.1.3680043.8.498.76889086689139179678902315420468394203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12982826264059547618124186257414818172","series":"1.2.826.0.1.3680043.8.498.89467679544746244770666086086174221303","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12980448935316870673341494036873313299","series":"1.2.826.0.1.3680043.8.498.11702937130191473140486762779513429126","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12987428318535758055829100395092244687","series":"1.2.826.0.1.3680043.8.498.10719311413560785992590737290801120809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13009271274636013067662786774946029285","series":"1.2.826.0.1.3680043.8.498.63603923571849050581553056440305019414","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13003520962744192762179786723724282013","series":"1.2.826.0.1.3680043.8.498.82954383484719465147116864341867601855","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12961116776122838195555132309258963046","series":"1.2.826.0.1.3680043.8.498.12006172447578015165768631603173370859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13024250150960117282590397343184827466","series":"1.2.826.0.1.3680043.8.498.19324553511060818126216345662852719923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13022266764623819477228425331705640702","series":"1.2.826.0.1.3680043.8.498.11989572844548733116298385962081630171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13016684730504329799460698013153308088","series":"1.2.826.0.1.3680043.8.498.55116087119309805586712312179575108914","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13027024577729659065187419038453181852","series":"1.2.826.0.1.3680043.8.498.92224062944715430670668365578972932958","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13026024768908635083636086393662414870","series":"1.2.826.0.1.3680043.8.498.41078411673776413821928162861976678974","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13028226019192925363616760332442456953","series":"1.2.826.0.1.3680043.8.498.21159904756356784281128612320515918410","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13034169434311535671044673510644711229","series":"1.2.826.0.1.3680043.8.498.88738672058635335006195497475417398947","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13062438493893996542058869514390908492","series":"1.2.826.0.1.3680043.8.498.99171418684229083157858371154787401414","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13042847643086819857036458569632034999","series":"1.2.826.0.1.3680043.8.498.88024536087331137445546377833764880721","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13065120731888368919957856813755701101","series":"1.2.826.0.1.3680043.8.498.20009363618206079505967417106678315673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13069081594164738319108129224400751974","series":"1.2.826.0.1.3680043.8.498.11678157124387490478707033370518164878","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13073387272228394257811861905056479438","series":"1.2.826.0.1.3680043.8.498.60893370578618234818013890971652440165","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13078009378379325641560321462188164769","series":"1.2.826.0.1.3680043.8.498.73757298296230701186112457142479849650","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13088550794087945662322609097092458487","series":"1.2.826.0.1.3680043.8.498.11969995319806431790246568809953466487","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13086260989634665810181105278930088749","series":"1.2.826.0.1.3680043.8.498.81321625024074362713260922734897134028","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13078834627392793401892111941627273421","series":"1.2.826.0.1.3680043.8.498.57283704638178551711162894709349326824","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13096288495834275184930201615129174987","series":"1.2.826.0.1.3680043.8.498.12500753101296511813293218651834252884","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13093528938282977263747792690249036456","series":"1.2.826.0.1.3680043.8.498.98098538398813067689620821965125962097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13102058401501684064017656674827527261","series":"1.2.826.0.1.3680043.8.498.12186172648842188822439642474804030051","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13114228294359876639109410575915394712","series":"1.2.826.0.1.3680043.8.498.13031296107901096787808216982348364219","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13116941594523500503401720636978356127","series":"1.2.826.0.1.3680043.8.498.88738795779538454915877747469190474657","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13118532243906060645114391749986421707","series":"1.2.826.0.1.3680043.8.498.12245541631528026565989521201533552722","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13128352910661672966143173193241171100","series":"1.2.826.0.1.3680043.8.498.25058843802429773001936603809663490981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13126070481622692281046898445918832841","series":"1.2.826.0.1.3680043.8.498.19610019955974063557448246438051187023","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13131420451380633781538717117200087258","series":"1.2.826.0.1.3680043.8.498.92174663229727921407801397244970826904","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13108275880471592702987713766776683590","series":"1.2.826.0.1.3680043.8.498.27535139479026182224816448702604623225","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13135567399913958681613600650440761032","series":"1.2.826.0.1.3680043.8.498.29800664003909507426181862873343585115","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13136392756016978765204420377500607545","series":"1.2.826.0.1.3680043.8.498.11750428867198073970737667410261445943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13129258242739880102472095242703169773","series":"1.2.826.0.1.3680043.8.498.62066293818217769382216184726327355632","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13143446880514535170343243531129864116","series":"1.2.826.0.1.3680043.8.498.12597809182422921706503466922417803251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13147249459968468500804358644901642270","series":"1.2.826.0.1.3680043.8.498.27402348357955636493759174879530526976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13151845892175420505816340845155354956","series":"1.2.826.0.1.3680043.8.498.82021453544799579721293714933489234699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13150780733560495803861509587935093776","series":"1.2.826.0.1.3680043.8.498.75996334913980370088534013586239214447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13137549927896953291645459480590055925","series":"1.2.826.0.1.3680043.8.498.73285583644971341314608763147408941283","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13163727085668231782866823012535268245","series":"1.2.826.0.1.3680043.8.498.71761806120847368021682684017862086841","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13158227087287104386790063637592969032","series":"1.2.826.0.1.3680043.8.498.37866985410064639471312717437014615339","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13167727300538353052482338786183678684","series":"1.2.826.0.1.3680043.8.498.34271655265798169250066220343283406660","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13155572156594271049520969168607160312","series":"1.2.826.0.1.3680043.8.498.11010836760325649087593876758273433991","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13173050135651736408720687361834045815","series":"1.2.826.0.1.3680043.8.498.56608850020810698913728649702337829851","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13173926977743814814616127342102020710","series":"1.2.826.0.1.3680043.8.498.61734927384950452652249555130093136532","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13197409499722328227876152940338160369","series":"1.2.826.0.1.3680043.8.498.55046450590441167179627693419353258034","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13179169600887013946657470830682778717","series":"1.2.826.0.1.3680043.8.498.32826629687404447834994516959490350712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13162284470388627411596358366248494046","series":"1.2.826.0.1.3680043.8.498.58493843372764224740666920143593944649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13211838518836535713919214396985811434","series":"1.2.826.0.1.3680043.8.498.10201180786107092525596230046660080602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13207489724395276840990928372610055495","series":"1.2.826.0.1.3680043.8.498.44224441873122068267896000657789592068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13214267416673058581197714011913363750","series":"1.2.826.0.1.3680043.8.498.90481900496778587123354074405427736038","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13224933005220329091792052916905776921","series":"1.2.826.0.1.3680043.8.498.11878355545043443182668702760306480364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13200225436827306542001129042739578441","series":"1.2.826.0.1.3680043.8.498.67178758847369958866404222955109202971","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13234278816788324761159326624155958655","series":"1.2.826.0.1.3680043.8.498.12999885135467675044392619983853852057","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13229563888378977350299319159748823477","series":"1.2.826.0.1.3680043.8.498.43353043189005297693739187999034033656","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13246496699722524712272293738359002501","series":"1.2.826.0.1.3680043.8.498.42710485567519166667618072850701868646","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13261595972961358158990410740461308054","series":"1.2.826.0.1.3680043.8.498.35806914259774676120803858151219541655","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13262473718294425031790270527413368245","series":"1.2.826.0.1.3680043.8.498.39782508957378881979972581093349987497","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13242536312535334908597493894251783308","series":"1.2.826.0.1.3680043.8.498.11362645941562353234152333408026635306","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13263128568157735797391439005687112733","series":"1.2.826.0.1.3680043.8.498.97027152166685982651733802471903266334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891","series":"1.2.826.0.1.3680043.8.498.60329250734334275308110817172897714281","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13273890409523605456488117581706115075","series":"1.2.826.0.1.3680043.8.498.48302873127639605262688812585368358409","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13290698873574296175460151724606672982","series":"1.2.826.0.1.3680043.8.498.43792449251722232927072959376323974045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13283411144589724334502427926341646116","series":"1.2.826.0.1.3680043.8.498.22528665468538717486315623484744421866","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13286048014183811835847662577023535662","series":"1.2.826.0.1.3680043.8.498.82017207776080441435115348665764657413","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13312139694018789912028129989483016875","series":"1.2.826.0.1.3680043.8.498.72776038008406453005836722041262601605","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13319886927127591971322856062062654146","series":"1.2.826.0.1.3680043.8.498.14731492060028210399097630663165677021","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13323187196249612936800308722675884644","series":"1.2.826.0.1.3680043.8.498.66862429273085349590085143151280455774","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13330694332190836651711643932953274573","series":"1.2.826.0.1.3680043.8.498.11669477324170542272080418595626823189","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13295231176137292542631413614945667314","series":"1.2.826.0.1.3680043.8.498.13109132308452580784591382528380628441","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13341507176140340924941654608476073030","series":"1.2.826.0.1.3680043.8.498.52571108109766329747936233936395753931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903","series":"1.2.826.0.1.3680043.8.498.32243142107334417600044875429817861685","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13353409340862821090624127486679097099","series":"1.2.826.0.1.3680043.8.498.21725903491789634003545009659912809793","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13297973188397662905314777604089528518","series":"1.2.826.0.1.3680043.8.498.45689743229914325224944955050735546295","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13357923062946813820390731946535198170","series":"1.2.826.0.1.3680043.8.498.12543820180205307415494391796554677425","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13361877959630120404875545203408927433","series":"1.2.826.0.1.3680043.8.498.30311101257419081202695987252648057357","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13367740087756379568826046059252678700","series":"1.2.826.0.1.3680043.8.498.30714271932798221997116625515992741439","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13363856251607349805111249441972460575","series":"1.2.826.0.1.3680043.8.498.11305649453613435427270352530719007207","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13369443704765101449274062863864122737","series":"1.2.826.0.1.3680043.8.498.34135020123377088433756212491044296399","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13391735462825209934041441767886593395","series":"1.2.826.0.1.3680043.8.498.89777955969898815239332321340508076335","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13383972749798909973472489456126521194","series":"1.2.826.0.1.3680043.8.498.10172096675742181949603061448097219152","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13377426362348493372112241208607685717","series":"1.2.826.0.1.3680043.8.498.80531221482059316948077072839139840934","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13440909104569690641314836192184434461","series":"1.2.826.0.1.3680043.8.498.10197408536472958680584857503808570567","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13399254446600380420635633478666850540","series":"1.2.826.0.1.3680043.8.498.43954172762427362769898771921347945985","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13459473184684263663596121399737033933","series":"1.2.826.0.1.3680043.8.498.10038965291494057560598082657384946898","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13405120554981098042415743645998258385","series":"1.2.826.0.1.3680043.8.498.11121266312276659683468880037211880848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13460268079835549295338279322112693785","series":"1.2.826.0.1.3680043.8.498.66386686775701547745538907266652789270","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13463672277039459152171414002273376185","series":"1.2.826.0.1.3680043.8.498.16023888630574555554946417102233270325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13616982533395291590655841043722813973","series":"1.2.826.0.1.3680043.8.498.11777157303005168483947648715317310210","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13575953449958751965232580597585303226","series":"1.2.826.0.1.3680043.8.498.10072118276867498117556502528770891174","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13695989890112408005495264068064110179","series":"1.2.826.0.1.3680043.8.498.74054904766766646021456028951089876610","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13762716088334302039143651507177887969","series":"1.2.826.0.1.3680043.8.498.54059918329460660895236762513494833829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13843368750899904654034226115826130084","series":"1.2.826.0.1.3680043.8.498.12447356330970976484925653534860110100","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13972325309264527611989585341636140156","series":"1.2.826.0.1.3680043.8.498.16489270590958729940569364251650449707","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13915367834258386094001181500560177470","series":"1.2.826.0.1.3680043.8.498.32118559313451399836691814908124865513","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14024129019390041106306062001538710978","series":"1.2.826.0.1.3680043.8.498.11383743949272989213711679118934560170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14188540532926067083914263496492255704","series":"1.2.826.0.1.3680043.8.498.10453101273293799976518158592893169210","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14061227335828384505403558788548348446","series":"1.2.826.0.1.3680043.8.498.49802424352484133244312423826903268244","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14117402902169664329719927863268363596","series":"1.2.826.0.1.3680043.8.498.11449458565026710704146920510416786457","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14229598279617168024212382336641104878","series":"1.2.826.0.1.3680043.8.498.12263433546723299406376553280363164132","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14300107110014472682373589824643789093","series":"1.2.826.0.1.3680043.8.498.10338385710790860548561110088160690528","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14400292895187181989838122627394761532","series":"1.2.826.0.1.3680043.8.498.11374002494404109454300818188155187830","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14479913217228604289584726096114526268","series":"1.2.826.0.1.3680043.8.498.10223200602511958865858437348031321954","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14347036698014475959973071477777915575","series":"1.2.826.0.1.3680043.8.498.12354441417681753107432814809457588236","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14495028403937132710814780922642512791","series":"1.2.826.0.1.3680043.8.498.56416928425843114538681629780343009591","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14472261871417480346162193127520918127","series":"1.2.826.0.1.3680043.8.498.35308045346715338082472139092061131103","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14728068688166107022558957899468489735","series":"1.2.826.0.1.3680043.8.498.76310028721186419699875866622664293236","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14770617961376953596780345652892307570","series":"1.2.826.0.1.3680043.8.498.16022199894736210710440883453196631893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14558486491688263090686005096911018438","series":"1.2.826.0.1.3680043.8.498.11526830812053522286770824095987282332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14806659119591146961784818305031900863","series":"1.2.826.0.1.3680043.8.498.36576398008482202836742702151087685754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14963472809950919852176661973255059069","series":"1.2.826.0.1.3680043.8.498.12218370167225090088219312257519288525","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14959561028578335448289993045623217861","series":"1.2.826.0.1.3680043.8.498.23523112619804934219057592655797714731","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15025957400098336605043184930980421418","series":"1.2.826.0.1.3680043.8.498.89105738709335100634649688811562086779","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14850534390650184292244380816212794996","series":"1.2.826.0.1.3680043.8.498.99761905817970565369973711918757417871","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15219824433491621460467738389345697247","series":"1.2.826.0.1.3680043.8.498.34440766542095340943601371010038306304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15090991392324602559982474708414427621","series":"1.2.826.0.1.3680043.8.498.70154063977187292401100334347539112807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15133818248365103660456294775831248835","series":"1.2.826.0.1.3680043.8.498.11956737332045131535346761143920883710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15360150088598749206650025637914821738","series":"1.2.826.0.1.3680043.8.498.12328220757960100225176727103072589880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15399233309269178192583032682669258885","series":"1.2.826.0.1.3680043.8.498.69110703196177779434140057045207572323","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15326951535563258414956521742264374802","series":"1.2.826.0.1.3680043.8.498.82181185528698350195557802576037140585","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15444316810805746453392291780593934328","series":"1.2.826.0.1.3680043.8.498.36789968901856227949370186210551992272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15486463042236628073590077209793828847","series":"1.2.826.0.1.3680043.8.498.18112221917094571272442260135827560234","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15630146025588580188363070220779173742","series":"1.2.826.0.1.3680043.8.498.52488771343947083455025863502380951661","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15595595490910196952568459534537629865","series":"1.2.826.0.1.3680043.8.498.10555392099828549633905944339837148770","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15744573360087157139672078937679122840","series":"1.2.826.0.1.3680043.8.498.19968217427183064866595312515172820518","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15539098084847502447238233198010208924","series":"1.2.826.0.1.3680043.8.498.20314738764751634145330365014872345179","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15863325224755490561125835019647303556","series":"1.2.826.0.1.3680043.8.498.11444028267275645733208144447071593915","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16189833935983604051983090678102023832","series":"1.2.826.0.1.3680043.8.498.78075198920291459358291861510652705867","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16158812105029662172517682642229718094","series":"1.2.826.0.1.3680043.8.498.32304046180778387620456466563628026282","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192","series":"1.2.826.0.1.3680043.8.498.11844032892710836502823802081213979899","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943","series":"1.2.826.0.1.3680043.8.498.75278479092582319503064396031082719193","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16318078762749420171833135145622117452","series":"1.2.826.0.1.3680043.8.498.99994312049398560882626966131758380565","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16364873390453802195568603605191835957","series":"1.2.826.0.1.3680043.8.498.97613402841128611037029030780225742771","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16046164723877343553376490423345098256","series":"1.2.826.0.1.3680043.8.498.86074561706437498460000633575977811428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16397020531009156965821242117417547331","series":"1.2.826.0.1.3680043.8.498.12106594461614209740814162844238669020","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16534078612126477228267765508485297367","series":"1.2.826.0.1.3680043.8.498.20815875871582615712806510216364429280","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16550793987525699514242241133391355835","series":"1.2.826.0.1.3680043.8.498.96199537833610289211554206557511886297","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16589468854653713035210251187012168350","series":"1.2.826.0.1.3680043.8.498.55353410913842133598590861262929312964","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16483470188406187000599644150600279596","series":"1.2.826.0.1.3680043.8.498.22897286622904227722680581663007665880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16666020201961191623151386398626511849","series":"1.2.826.0.1.3680043.8.498.78908142632251992260651644141150959253","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16691383858003633107641297162554868325","series":"1.2.826.0.1.3680043.8.498.67335441164844698451715816282952347740","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16709696133689803501344038926512501779","series":"1.2.826.0.1.3680043.8.498.45183914324618822777911577223854321279","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16845451648760595922758166878121325861","series":"1.2.826.0.1.3680043.8.498.46297175789577846950046172944705142424","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16724013341359693140331158148551346369","series":"1.2.826.0.1.3680043.8.498.78528441236681678502197554054132671413","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16864649493021092944977766326505053153","series":"1.2.826.0.1.3680043.8.498.94160070623200539636420206772509316048","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17049315826418217889213226790962592213","series":"1.2.826.0.1.3680043.8.498.92991063145772262679524280827617643931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17070139245390396445395414066427036835","series":"1.2.826.0.1.3680043.8.498.64093540392311076190498704490417306533","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17107696939614797679012793547913769707","series":"1.2.826.0.1.3680043.8.498.61245332578805908949031381626575572558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17092147498062993594620309531155267119","series":"1.2.826.0.1.3680043.8.498.88593448085647941227670430637249380422","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17146868851921828837871671912815843012","series":"1.2.826.0.1.3680043.8.498.85297535217843228765328146948973015558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17241550277273207658303011467222594165","series":"1.2.826.0.1.3680043.8.498.53686120654406537336016944388907154138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16812524557513419866531695739014681414","series":"1.2.826.0.1.3680043.8.498.52588485467989835018516090707006064483","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17274490471295558668613733667390457509","series":"1.2.826.0.1.3680043.8.498.10196860452519015050280231146565470277","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17389120886685633448235711042721669823","series":"1.2.826.0.1.3680043.8.498.10327837334716075578292438576117163289","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17662880041629379035049367252086125592","series":"1.2.826.0.1.3680043.8.498.11225532576317555678326430491222394299","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17420673628956367425739018933968443620","series":"1.2.826.0.1.3680043.8.498.86768988848763601804646633826371604087","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17517950899674150719043097786728121608","series":"1.2.826.0.1.3680043.8.498.10318025045146320773196757695566786442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17596657287050550465463272509976106854","series":"1.2.826.0.1.3680043.8.498.51269758557856502958616579858316847371","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17768582402712001917772688462080201650","series":"1.2.826.0.1.3680043.8.498.12023525097305968650090822258728072936","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17813878802252660525798053577039470087","series":"1.2.826.0.1.3680043.8.498.11067351060599717646765478308674261078","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17931021655186901394796168474942485728","series":"1.2.826.0.1.3680043.8.498.54783523305456002398679325819694820326","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17985756708438762670829959560049370730","series":"1.2.826.0.1.3680043.8.498.72043274517048913626950162982553609232","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17691941055331862184525411493110144433","series":"1.2.826.0.1.3680043.8.498.11732967989180760655680098537090895223","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18113123512696698441392990570166201548","series":"1.2.826.0.1.3680043.8.498.48133754883459093795140678383758807762","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18090147030578547920423840630147682826","series":"1.2.826.0.1.3680043.8.498.79454006727337376915948969157375062011","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17865613300681542407687000381594067366","series":"1.2.826.0.1.3680043.8.498.11696399044106161138066918328501448254","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18199465156572252531439880765733639584","series":"1.2.826.0.1.3680043.8.498.56806124599210622308553163270409047857","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711","series":"1.2.826.0.1.3680043.8.498.12647596472796287171744368208436290003","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18245500498843512203508664874652003124","series":"1.2.826.0.1.3680043.8.498.10762203843506972411800239191793929665","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18344563362014614046776037921812502569","series":"1.2.826.0.1.3680043.8.498.77894591706974627283467527327745950493","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18389676218363070462526838642303976332","series":"1.2.826.0.1.3680043.8.498.11534431000967639894324482743361512739","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18175450083338316799857698898702333009","series":"1.2.826.0.1.3680043.8.498.61789785747378194262221905090501252454","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18518393141873346710197645305647124682","series":"1.2.826.0.1.3680043.8.498.10869310324248929747525445715785270678","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18592535981451766048952107991674227570","series":"1.2.826.0.1.3680043.8.498.85245730776669964879258137451674525818","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18625377919309205857120774497713011583","series":"1.2.826.0.1.3680043.8.498.54689841097404830358618014382962870719","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18671006890719350655167441375718348829","series":"1.2.826.0.1.3680043.8.498.57525924902205150544248843434006557401","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18758199166360428760124471581842191566","series":"1.2.826.0.1.3680043.8.498.75103383895801352736726531448271814063","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18848481788456572234982139368351538716","series":"1.2.826.0.1.3680043.8.498.22347627497092961764059556693467758173","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18807952338149288346040972005636698334","series":"1.2.826.0.1.3680043.8.498.58303087844085789339254873594303304975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18899625751876258520147203904985857313","series":"1.2.826.0.1.3680043.8.498.85362301305559162166027020467893662856","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18403627643718880914843209837672242908","series":"1.2.826.0.1.3680043.8.498.12873181211839997553833496606491963666","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18968967345606818331567480514437075646","series":"1.2.826.0.1.3680043.8.498.86767382328069826065343657236383367794","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18695727379446620258431748063749176639","series":"1.2.826.0.1.3680043.8.498.61225035056450823345903426141309961899","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19034825848229751701363461935217290295","series":"1.2.826.0.1.3680043.8.498.11467017173325648867256917416001020709","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19197037365998103318720742154440921358","series":"1.2.826.0.1.3680043.8.498.13230956512357507492284164573171043729","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19291375342005405073321673638971145777","series":"1.2.826.0.1.3680043.8.498.30957353422002248521481632673974913924","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19124879088354547836118593585646705198","series":"1.2.826.0.1.3680043.8.498.10929190237748516864924559465797531629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19374331384306309057311171252852774841","series":"1.2.826.0.1.3680043.8.498.12554153563601248448420871595479748354","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19183011851352129901149443549646899548","series":"1.2.826.0.1.3680043.8.498.52674808923448056777079677011903656417","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19439950168933299231685815631786334293","series":"1.2.826.0.1.3680043.8.498.95734032623208705512255917022892820712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19516545000256238042886469152307943182","series":"1.2.826.0.1.3680043.8.498.91878618850101117240256101717136256336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19489875090732464985703624680756212359","series":"1.2.826.0.1.3680043.8.498.77703733877193279265246773795281258502","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19624674714110931897302481276137067638","series":"1.2.826.0.1.3680043.8.498.41312420245936010059177261952201627358","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19813267030974842239752491935957206955","series":"1.2.826.0.1.3680043.8.498.54494516519889180804460372904088862999","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19722687746450343001601428638894755031","series":"1.2.826.0.1.3680043.8.498.40343874840444687228101563005549690874","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19830663206351645385520268848711462246","series":"1.2.826.0.1.3680043.8.498.29448896805415956194464239028139528518","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19845711038895023426012115550237130530","series":"1.2.826.0.1.3680043.8.498.23385529225779993097024888805820360059","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19592989024994864431870593504844309885","series":"1.2.826.0.1.3680043.8.498.10999243104292802162073528803301708691","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19925337578871911327225897926650188274","series":"1.2.826.0.1.3680043.8.498.30476174448276845160808424010689496237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19887850741193535685529812263489892001","series":"1.2.826.0.1.3680043.8.498.10311986159399473114717199847258404840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19977273190231806393528180968361858122","series":"1.2.826.0.1.3680043.8.498.90379137953625930379424850396961647525","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20063306448698185569553477478719533390","series":"1.2.826.0.1.3680043.8.498.12172494624534065850930467525454219800","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20184541870535223885310764061146422870","series":"1.2.826.0.1.3680043.8.498.70521700752910770270351352831225078813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20196165276643741937915346646793735100","series":"1.2.826.0.1.3680043.8.498.93502209050112873906516217092073630527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20269296702024202621734017056347194499","series":"1.2.826.0.1.3680043.8.498.94284416663605591206297395908976513563","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20134168618929769962391688049013971830","series":"1.2.826.0.1.3680043.8.498.74746365934563371258223236405420153893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20375739629244657719385962474248679850","series":"1.2.826.0.1.3680043.8.498.10742380850219837245211460396921572708","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20521417572710861366819886462837836465","series":"1.2.826.0.1.3680043.8.498.78697720036035740968256663977621693963","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20427402498958253193571475926580142499","series":"1.2.826.0.1.3680043.8.498.11152730140409758884686569292075924051","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20488337421252584047930242186072775007","series":"1.2.826.0.1.3680043.8.498.26202680422718921893885592726506572254","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20587514285427671071934631922521946816","series":"1.2.826.0.1.3680043.8.498.11523217572126152962160921544165440166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20640858366001462094044810443654811785","series":"1.2.826.0.1.3680043.8.498.31650080465618438430523212806679583621","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20232953944759707220316171133595554284","series":"1.2.826.0.1.3680043.8.498.22132631360759187874046056033023076506","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20733151481921261415295204209910573639","series":"1.2.826.0.1.3680043.8.498.26118892537419085557301539679548395641","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20953536352252551462901259297974956430","series":"1.2.826.0.1.3680043.8.498.83171250629713292101967120126611100629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20852351697358589606507109300359572624","series":"1.2.826.0.1.3680043.8.498.55297135162261229046028360762788497600","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20933151653103548000061448393771614270","series":"1.2.826.0.1.3680043.8.498.56844460485952340556089632287467034173","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21060274255072711241653174133146945101","series":"1.2.826.0.1.3680043.8.498.27357833280430216720732476939774633429","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21239630702461934662855021164119114203","series":"1.2.826.0.1.3680043.8.498.87505716596937779484047808618604555708","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21312254534068992054745674608909722454","series":"1.2.826.0.1.3680043.8.498.11814564742376856923567906401193955690","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20801986011906346655458394198951999413","series":"1.2.826.0.1.3680043.8.498.33580467286243249295830435016521033051","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21135201439483157418374033420156525658","series":"1.2.826.0.1.3680043.8.498.11155341686101066666030988292166106417","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21389336408905524316915532623385847283","series":"1.2.826.0.1.3680043.8.498.80351487496445035957021274802393258166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21413285047211074156709106352750752275","series":"1.2.826.0.1.3680043.8.498.11652508826162764717405568023693331243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21012635480218712816069264326295410954","series":"1.2.826.0.1.3680043.8.498.33362799521324252504525584889432002824","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21478660341431677113136859217606773871","series":"1.2.826.0.1.3680043.8.498.27306369991938040307328367086662980366","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21556073772857867149576001337847113661","series":"1.2.826.0.1.3680043.8.498.10308612330982861991719143549300155186","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21584859831565112159503971937497384030","series":"1.2.826.0.1.3680043.8.498.12116409567314195410921370804153769386","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21606196343708804451390109412118036330","series":"1.2.826.0.1.3680043.8.498.69758444945480502033227478842724221746","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21636337451527049130190928852193267826","series":"1.2.826.0.1.3680043.8.498.14557080101775734982689537862191843000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21769352980698937182060933429354012104","series":"1.2.826.0.1.3680043.8.498.12373081538281234525464917373363268914","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21862609534273524003153225756948913437","series":"1.2.826.0.1.3680043.8.498.97423979213378037024610108638471347728","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21937000442536121262190458446148124910","series":"1.2.826.0.1.3680043.8.498.11503697292988964850537522482109804415","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22025729757015485203756049440674914537","series":"1.2.826.0.1.3680043.8.498.71004682170600229100910805159512490956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22053727963380087706894183799325067544","series":"1.2.826.0.1.3680043.8.498.30312748347182078813153788630418377687","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22088734384869383587217556574382319654","series":"1.2.826.0.1.3680043.8.498.72210395231783858262319112994061554540","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21874562538764236819894456417694074673","series":"1.2.826.0.1.3680043.8.498.99268646549647053641058841619189449843","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22135282054716841452811687331325271982","series":"1.2.826.0.1.3680043.8.498.23094642084293848573222438499281135361","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22119720417177816110156744770171496965","series":"1.2.826.0.1.3680043.8.498.12642689937308707950933301984030640737","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22249763791584161324425922920269052844","series":"1.2.826.0.1.3680043.8.498.10314834889213432594346486361416312305","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22357754074308826479280023393343731000","series":"1.2.826.0.1.3680043.8.498.62731812544988454420374535124433881817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21991252201684723900045301917670383228","series":"1.2.826.0.1.3680043.8.498.58090393006569101765204380641303603581","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22398041676780235097751294563297953841","series":"1.2.826.0.1.3680043.8.498.40440916678162390567628275100674845572","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22456525904575663955604077506103354060","series":"1.2.826.0.1.3680043.8.498.86677045942694059277557242033992930194","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22508124641661887430614105328316053644","series":"1.2.826.0.1.3680043.8.498.10935417237564335277934850728569356860","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22407175411826914385667795929161924645","series":"1.2.826.0.1.3680043.8.498.74368606250009360116849917558220814456","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22594249996322566888700984133307337128","series":"1.2.826.0.1.3680043.8.498.79719234863479159186696953984193889484","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22596004572382994883130705870731193046","series":"1.2.826.0.1.3680043.8.498.62543423010562898734410287052184934618","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22275537949387641831010030150398063643","series":"1.2.826.0.1.3680043.8.498.10273498432404468154456110886328752651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22678957404909201599340396039237525550","series":"1.2.826.0.1.3680043.8.498.52306955606281620616666095177360439658","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22632919018724241015271844108954804020","series":"1.2.826.0.1.3680043.8.498.95074300241248665038905223977203651351","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22803020644232321685575897661362804423","series":"1.2.826.0.1.3680043.8.498.11012066421729811002231381529887461526","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23043120269252616884307662769766701449","series":"1.2.826.0.1.3680043.8.498.12108183385863448431870286695679152134","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23063675880057833741455102729816364091","series":"1.2.826.0.1.3680043.8.498.91267745773964880080001564953586315443","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23073694267101759922738121732233014980","series":"1.2.826.0.1.3680043.8.498.34092593612358507820310596296054779946","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22985467719058223198642920273233975053","series":"1.2.826.0.1.3680043.8.498.93949955567631417046934223350082114251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23094733099955217022059584418468076048","series":"1.2.826.0.1.3680043.8.498.36296029079957551969013423439568103249","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22886251629508830700774506697670139848","series":"1.2.826.0.1.3680043.8.498.92548634661520652237930619732014106768","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23409201792139701921731647470152180932","series":"1.2.826.0.1.3680043.8.498.12789026972720181021327531822855395162","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23166300530486600338569917595744444657","series":"1.2.826.0.1.3680043.8.498.10851559814373512534138964577209130637","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23427198594727947883953289189443006288","series":"1.2.826.0.1.3680043.8.498.11409597031553463185211699463755861920","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23378338408981699466914366379590099738","series":"1.2.826.0.1.3680043.8.498.92896822608634432460865099108134139868","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23523176367321534630723586470190430254","series":"1.2.826.0.1.3680043.8.498.86651125750476640600458564882684251082","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23686732555936573712275826587678502899","series":"1.2.826.0.1.3680043.8.498.38309977987610271114638343915985175494","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23713320796530316370039202981138376061","series":"1.2.826.0.1.3680043.8.498.87374717944355067239306326032174645442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23733147518259581649562651915997449088","series":"1.2.826.0.1.3680043.8.498.42188426405570945267869255262009336931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23849672490426589132734390202638283865","series":"1.2.826.0.1.3680043.8.498.77874791346378819343113775945914406827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23986051274386630509877205174270047762","series":"1.2.826.0.1.3680043.8.498.11889274897805361052463444823097842994","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23939313916974849526459717576344063277","series":"1.2.826.0.1.3680043.8.498.64377746410115163216126275635225648263","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24126422042626612740219325272793737656","series":"1.2.826.0.1.3680043.8.498.19160237476840923062962792900430241091","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24267153760943721146187182369810590631","series":"1.2.826.0.1.3680043.8.498.19120978724590476107906487185191676717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23822313132245448173452915228147497501","series":"1.2.826.0.1.3680043.8.498.36706497608271950515608592380361786282","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24068239310155798723127213793623172530","series":"1.2.826.0.1.3680043.8.498.10973306044747059940542641637515070681","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24219103446788192052503003052696523200","series":"1.2.826.0.1.3680043.8.498.29962407440283384875746703986222582171","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24313647347273496316728514161733307557","series":"1.2.826.0.1.3680043.8.498.67954101476322784463095191846767762397","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24432243084670627958434695475653885175","series":"1.2.826.0.1.3680043.8.498.14487114335891648434539378322233699747","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24502332819381293694020055836255028614","series":"1.2.826.0.1.3680043.8.498.71121256822437936986967711178028788097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24291465113360632954006568021942978765","series":"1.2.826.0.1.3680043.8.498.60824386064264777694144543143335563617","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24661137747502154216858169258339731263","series":"1.2.826.0.1.3680043.8.498.37737970288850893679744003793649380532","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24588524807799377432185260702152371503","series":"1.2.826.0.1.3680043.8.498.83725936957424306595093730826762381061","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24680736086205072406257935876858402593","series":"1.2.826.0.1.3680043.8.498.93262057968373100540166431453530390210","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24758830794084473905762034032291806038","series":"1.2.826.0.1.3680043.8.498.20791398796465200779169043225755258477","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24826041178550854190310664677274404684","series":"1.2.826.0.1.3680043.8.498.83325981508119308742242504805976381421","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24872746733270791851241663716306993824","series":"1.2.826.0.1.3680043.8.498.11469401967247091624769873573071113084","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24555528844026121007453886375651403314","series":"1.2.826.0.1.3680043.8.498.87869236251023237433175498895287268149","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24940437326555494150792221007907527983","series":"1.2.826.0.1.3680043.8.498.10503581053680883556156547705280226043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25010661650105533302140829168731585935","series":"1.2.826.0.1.3680043.8.498.32916724983246481099074088426518794990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25103708168569906591706406794385334861","series":"1.2.826.0.1.3680043.8.498.10675114465276172491992868074541088569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25145588904873072866960435474159832047","series":"1.2.826.0.1.3680043.8.498.12354753594609904252041979450256296318","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25167060436407065195851060078175404789","series":"1.2.826.0.1.3680043.8.498.39311761807960026618721385472839681713","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25252553477605568387898480443616118724","series":"1.2.826.0.1.3680043.8.498.28435334945646042857631401526218099698","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25196393144875332073285250936506555534","series":"1.2.826.0.1.3680043.8.498.10206706303591790916343722924639349073","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25359770839263995354733835365510047109","series":"1.2.826.0.1.3680043.8.498.11305807499339011780757582719118860130","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25266231062774974303723359806535863810","series":"1.2.826.0.1.3680043.8.498.11301436726729180709633752945537122135","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25431207706481141354761246438973384670","series":"1.2.826.0.1.3680043.8.498.11061835977317429793697356754847694880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25505153402978560400776672230220616863","series":"1.2.826.0.1.3680043.8.498.44921373983462834706134769006158985681","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25549974159486441938843396591725289882","series":"1.2.826.0.1.3680043.8.498.12771549602237763789841828877947665531","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25480883220477368504575353972328124891","series":"1.2.826.0.1.3680043.8.498.60135955330238585386626945153258246974","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25558526735614916746398919603493693652","series":"1.2.826.0.1.3680043.8.498.35782351866367011994814907245015720951","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25616615589791001678067099907823838797","series":"1.2.826.0.1.3680043.8.498.71267934639404970881744880271151396705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25663557443141366332431300959089922567","series":"1.2.826.0.1.3680043.8.498.10390250789019313817895217781864415749","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25643897330283020467388814112594686690","series":"1.2.826.0.1.3680043.8.498.16706284490885452348854061880220974496","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25679930222284447338484621953345520487","series":"1.2.826.0.1.3680043.8.498.37209366517316832302582735089595101573","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25697539332996211770844878649053316629","series":"1.2.826.0.1.3680043.8.498.82368703707891418350906120412087770429","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25815857653380672532983223088817283976","series":"1.2.826.0.1.3680043.8.498.23964233906613968970767373741405791297","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25765711614507755527825654507920391096","series":"1.2.826.0.1.3680043.8.498.11499992333589408206255768377124855013","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25904473139226718487560978103077541725","series":"1.2.826.0.1.3680043.8.498.12068607763887646715186810798378965240","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25928779639851562853500186715690156268","series":"1.2.826.0.1.3680043.8.498.87392387749408229209028993215825433146","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25766673238950703352855821505938941878","series":"1.2.826.0.1.3680043.8.498.49313136217818031568849467094110837120","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25856396695552621816673203639618612020","series":"1.2.826.0.1.3680043.8.498.50955808200217567441645214375194012497","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26023120405608635089544039694812003604","series":"1.2.826.0.1.3680043.8.498.10466202330722238070326359452168824240","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26004572902429209855865822977387283547","series":"1.2.826.0.1.3680043.8.498.13128465680021306032582879320554446369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25977532612504997807641730439567378227","series":"1.2.826.0.1.3680043.8.498.82159736662506622201456314598753020859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26266119962108543565575361589492142091","series":"1.2.826.0.1.3680043.8.498.69469841769883262260276339601367576945","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26072622476167889001255150030772906081","series":"1.2.826.0.1.3680043.8.498.80178119670542339882399667981171343759","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26431974442245797095978351976773354669","series":"1.2.826.0.1.3680043.8.498.28761268253720255800239982364106618996","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26333849754345698117396119042432557445","series":"1.2.826.0.1.3680043.8.498.85537931925911391167271099481172811399","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26201626427712146573573952548179938304","series":"1.2.826.0.1.3680043.8.498.97516787416941343033289650450765456022","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26536992666456596813876340849399448897","series":"1.2.826.0.1.3680043.8.498.95316512663931345042478038972442465824","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26485916396240515006042297316506505324","series":"1.2.826.0.1.3680043.8.498.10662579038093138779984220387670806412","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26619973716086136503589916868066348957","series":"1.2.826.0.1.3680043.8.498.10453415806397984113277370622414174033","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26373657290121957625263322898744712730","series":"1.2.826.0.1.3680043.8.498.92703988147781206971094781896415900808","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26698126402727061400851465377574886112","series":"1.2.826.0.1.3680043.8.498.42079706488655999241887378983689047215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465","series":"1.2.826.0.1.3680043.8.498.76639110060009477040774057211843609294","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26842258746390750314059176647398369242","series":"1.2.826.0.1.3680043.8.498.10662461909695175516192593157177068816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26848412380762311150802301982233072694","series":"1.2.826.0.1.3680043.8.498.70521424925419104507233916431321910314","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26880562799522930458745136318439622601","series":"1.2.826.0.1.3680043.8.498.45994383167323604703135183740637817315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27020884753639915245324093232218617967","series":"1.2.826.0.1.3680043.8.498.58165474059845580604384008338135663485","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26978027400525885904983020900996228004","series":"1.2.826.0.1.3680043.8.498.11744911536645495385787524557440429934","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27059975738703546675398620281849285863","series":"1.2.826.0.1.3680043.8.498.60942571591166063726009849325077602471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27161418589407291713644384596541786246","series":"1.2.826.0.1.3680043.8.498.12605449158214266473904999297796168729","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27186001621007168581611489439038285508","series":"1.2.826.0.1.3680043.8.498.84393876088808274463002503414958988504","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27192678730060203972082511503480566620","series":"1.2.826.0.1.3680043.8.498.17396428235108103979562208846344519258","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27025989808002468335294585261291982661","series":"1.2.826.0.1.3680043.8.498.63060356447163260689527522763070958275","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27224327142578332905227939015479767285","series":"1.2.826.0.1.3680043.8.498.11612119772984227014971167956922181157","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27306039618857661522863167157208122018","series":"1.2.826.0.1.3680043.8.498.71494760316702866130184064221358903816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27339999385861467858422021400023533486","series":"1.2.826.0.1.3680043.8.498.81981702592814127817584517116104250261","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27319908980978184820438244981035567211","series":"1.2.826.0.1.3680043.8.498.10099231781303174010006408777590646905","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27422742748182689233651649793019570511","series":"1.2.826.0.1.3680043.8.498.16088601401847679112156385593642021058","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27456622705361831116491505615162380604","series":"1.2.826.0.1.3680043.8.498.12298676920246788266998876587310951308","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27653320597564308144389724321115208375","series":"1.2.826.0.1.3680043.8.498.55956317947207026587272182453333447170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27656830051511853680197276412365020780","series":"1.2.826.0.1.3680043.8.498.23711895519764225069822163214978457199","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27627190971637735537007595480396264063","series":"1.2.826.0.1.3680043.8.498.11423795381631475167935075394289680738","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27723413941379681806317930349613970721","series":"1.2.826.0.1.3680043.8.498.10421323280640142846460024157322746944","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27737101979486411869208236811885597351","series":"1.2.826.0.1.3680043.8.498.62347154379215784513497657217587682026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27796604203156439069742212039368371002","series":"1.2.826.0.1.3680043.8.498.30520697695445514238394213496537055392","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27890860883826681731748268459485259935","series":"1.2.826.0.1.3680043.8.498.45972846249153383456968422639063567238","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27925145307825795804342879375835424449","series":"1.2.826.0.1.3680043.8.498.38872410222075350193301684118821124270","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27474440682441854303485403899867525864","series":"1.2.826.0.1.3680043.8.498.12546153241462180982793601369689398805","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27946605663086243142356841956045514227","series":"1.2.826.0.1.3680043.8.498.10865787499123358201070338768720097973","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28011057282528460306257420499560926354","series":"1.2.826.0.1.3680043.8.498.15767148471223425241708395850301477365","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28028890493668487238845659163252231337","series":"1.2.826.0.1.3680043.8.498.57063452162924679989798166251180468331","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27681651277422558344520227187021583924","series":"1.2.826.0.1.3680043.8.498.75965197563925741625221726837516783109","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28218476802410349760103243990109474475","series":"1.2.826.0.1.3680043.8.498.11858328475198032741362701797811211913","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28134711422142706763867845558971485241","series":"1.2.826.0.1.3680043.8.498.29436645078339287703086080601909444262","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28237068093947300427040925444622840795","series":"1.2.826.0.1.3680043.8.498.55309159342678412834689609531200990471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28045606887691550130031344985830975325","series":"1.2.826.0.1.3680043.8.498.11064109764685084786946639764892086098","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28289838818003822677994912735532050038","series":"1.2.826.0.1.3680043.8.498.66468758562523185533468123248213455817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28309626953500461628112343997868050028","series":"1.2.826.0.1.3680043.8.498.82677472434340750413499745348432339513","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28351884945918062351436698411322252525","series":"1.2.826.0.1.3680043.8.498.61375876835664325480148275674555062840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28488216314048990109149780234417422948","series":"1.2.826.0.1.3680043.8.498.15678097195616186042985594807934527349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28425280718252632810092859612928047098","series":"1.2.826.0.1.3680043.8.498.10996320913706997581656197411371765557","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28595729988867300278425411189173775382","series":"1.2.826.0.1.3680043.8.498.45762806670293118643294465887256498725","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28628472249424790112957421215774036156","series":"1.2.826.0.1.3680043.8.498.12427732169235811881932174579250457921","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28539130184243787339297053654373783542","series":"1.2.826.0.1.3680043.8.498.67927523134797075657123165809736504442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28638490997780173004029312240334483157","series":"1.2.826.0.1.3680043.8.498.97264523884261393571466097336765023863","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28655385212214986812679045835025619123","series":"1.2.826.0.1.3680043.8.498.39823041224413277480268573209405009630","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28746954364434312909368235768936980607","series":"1.2.826.0.1.3680043.8.498.78775065568289180100033531284212751033","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28779021651975021236045277212517629154","series":"1.2.826.0.1.3680043.8.498.10118611894766940897517672723885257772","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28873180831637763124652074775631752918","series":"1.2.826.0.1.3680043.8.498.89772547104367607993513204003291374103","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608","series":"1.2.826.0.1.3680043.8.498.88416925277894891378549723681135749833","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28686498221359515429184385385077600687","series":"1.2.826.0.1.3680043.8.498.31118817986070899822975521351771519650","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29227028592609694700827329867343028944","series":"1.2.826.0.1.3680043.8.498.52263141810725736651291764469557977429","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29055892482211392922525222851162308510","series":"1.2.826.0.1.3680043.8.498.92339134310169992597613891203115704821","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28954361146470748868500417103935640742","series":"1.2.826.0.1.3680043.8.498.20653982341147774399009386107897501573","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28946722216615442736212359468410891872","series":"1.2.826.0.1.3680043.8.498.57590514577223763321310593464320724533","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29292398668403593100514493322849770181","series":"1.2.826.0.1.3680043.8.498.70910712265247387623888775767971449501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29343735048224321966824641432873034291","series":"1.2.826.0.1.3680043.8.498.11821834434707907257306071283143861806","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29364640434226980751405780979697632001","series":"1.2.826.0.1.3680043.8.498.69950374033910117930696342597225384094","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29375052467745327893128894062568944496","series":"1.2.826.0.1.3680043.8.498.10461632583436213318621688189541850748","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29473278891551292295080887778714088878","series":"1.2.826.0.1.3680043.8.498.12172716706688803307920696109196387240","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29418130858182890216867847991592582370","series":"1.2.826.0.1.3680043.8.498.12966533137847532533432901189975987189","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29579560656789178333629609998252952856","series":"1.2.826.0.1.3680043.8.498.11463466628492424245314612665650892748","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29739171278960083313048266628194182569","series":"1.2.826.0.1.3680043.8.498.11664247862831881323395155411890212447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29758331841591711520514486913006094054","series":"1.2.826.0.1.3680043.8.498.30684611705269449263210719764419969292","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29866827083314645739444078187995496049","series":"1.2.826.0.1.3680043.8.498.54972033214920802299712022344975725315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29971686675947075024168289004002445393","series":"1.2.826.0.1.3680043.8.498.13187856990960059279704742615377977376","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30058585660043363937345050478072453854","series":"1.2.826.0.1.3680043.8.498.23480217721204609780806379699607382745","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30104491808697888642424160917715722238","series":"1.2.826.0.1.3680043.8.498.49021516346135628319874997983043239313","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30174064884443577177838026250637902388","series":"1.2.826.0.1.3680043.8.498.68327849870342597394320344963311540175","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30150613585152527569197215147135001922","series":"1.2.826.0.1.3680043.8.498.12394471865050070118255267538894423043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911","series":"1.2.826.0.1.3680043.8.498.13364655615747190896475164649441911015","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30222189977108980260768952691472020722","series":"1.2.826.0.1.3680043.8.498.31484617641218130862047660690246746083","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29794301133358685635777478470865282350","series":"1.2.826.0.1.3680043.8.498.21161397102713509132001544590737564927","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30381005430261460472880465705496199540","series":"1.2.826.0.1.3680043.8.498.37418784542545257274714661627978038997","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30282025087399390929963075109487390046","series":"1.2.826.0.1.3680043.8.498.52823494043529682189038107782018201848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30423643741419627789701272644909134762","series":"1.2.826.0.1.3680043.8.498.12392133833947198191848529542568039442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30490857770169699336468319880130884989","series":"1.2.826.0.1.3680043.8.498.11483539117590170354706106131069786357","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30557810085867943803354292590079995622","series":"1.2.826.0.1.3680043.8.498.24077785817362409098999952415943378232","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30520444048368652235940346583890702451","series":"1.2.826.0.1.3680043.8.498.79839955925555385862489786610386324420","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30662407196303880392459287553277761223","series":"1.2.826.0.1.3680043.8.498.89729442768468456054114970702387954524","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30252501257451878373370333329266264057","series":"1.2.826.0.1.3680043.8.498.10685896341445645681996674725447643663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30673411238679998084995204090753297949","series":"1.2.826.0.1.3680043.8.498.12599750580812755805445726215183947350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30746167840159843733412534591988948270","series":"1.2.826.0.1.3680043.8.498.72986544783094840909494432855642348980","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30838586189496669675465778955893367497","series":"1.2.826.0.1.3680043.8.498.85053425602417621848409573427036745347","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30763696628968291934469221453461209579","series":"1.2.826.0.1.3680043.8.498.10552235174703550393451849162388769721","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30947630416867960170317316514422933066","series":"1.2.826.0.1.3680043.8.498.57058083148303659570170144172751657918","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30888379005655662168586864728687777425","series":"1.2.826.0.1.3680043.8.498.77592910892876839006375268431620482339","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31080452089421812812019413302222437434","series":"1.2.826.0.1.3680043.8.498.18657547827515406204383520751565437583","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31145453746697192222724674918300432873","series":"1.2.826.0.1.3680043.8.498.36529212585052191105691171332876785077","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30832771419882354941430609179589560374","series":"1.2.826.0.1.3680043.8.498.18998758027097646382710114175513785621","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31099878224304681441631176766255029271","series":"1.2.826.0.1.3680043.8.498.24718845335586396638422314959291258939","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31175171249406459042393597291141275669","series":"1.2.826.0.1.3680043.8.498.79564398001802877300133053291158893196","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30998941785989496591079894962766360976","series":"1.2.826.0.1.3680043.8.498.12894328905885541290514240965979994834","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31193772347652083986607183406944630716","series":"1.2.826.0.1.3680043.8.498.26150829645641947248841990605285832198","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31373064397225665483187291274194812486","series":"1.2.826.0.1.3680043.8.498.78309727791151614659143832636960265777","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31220970065722563280311764269386457090","series":"1.2.826.0.1.3680043.8.498.41658661181313939050182618326068905407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31280487369922340170695453174997912050","series":"1.2.826.0.1.3680043.8.498.11753709004566234898338787712410640602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31284572943302383501947308874558551301","series":"1.2.826.0.1.3680043.8.498.22730240003047429734884863015136291135","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31555333355990829772911515362502000209","series":"1.2.826.0.1.3680043.8.498.10531545665261055833730518433910793948","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31461240229123701082444627629841504099","series":"1.2.826.0.1.3680043.8.498.64146847745325599700142998289794744620","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31610133883321894510649078615659708431","series":"1.2.826.0.1.3680043.8.498.70232897578699202788218895629154247993","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31792234576444017394182974951507838927","series":"1.2.826.0.1.3680043.8.498.53636821062436630092720753976480414899","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31745644522310786426860265012373331979","series":"1.2.826.0.1.3680043.8.498.12548594251388596898054752478266769796","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31851627014639252866084081561126592250","series":"1.2.826.0.1.3680043.8.498.51288997156339120720955190187792117113","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31827987476038668074615045841102711882","series":"1.2.826.0.1.3680043.8.498.11663456935307515342346667410140230433","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32043012602421532910420313983803503778","series":"1.2.826.0.1.3680043.8.498.65582622051030114319388100877996526361","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32172672595181417103935593872811182816","series":"1.2.826.0.1.3680043.8.498.81845363596462298600837184216412830297","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31899121807187535405063131351086765296","series":"1.2.826.0.1.3680043.8.498.60842651076064986690489009739987259383","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32265622493980668732317541941196894795","series":"1.2.826.0.1.3680043.8.498.33301963288916587176468030754122994608","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945","series":"1.2.826.0.1.3680043.8.498.49657487656121528921826247613085032325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32226253011052584155674401943276961907","series":"1.2.826.0.1.3680043.8.498.50253959585692500999605462999378120072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32329374180424300100441619205766536205","series":"1.2.826.0.1.3680043.8.498.27442467542624186606510507013334763019","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32109377199577933060124357239881959602","series":"1.2.826.0.1.3680043.8.498.62665295568889577240908291213976166319","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32488933488399304214586978746578986823","series":"1.2.826.0.1.3680043.8.498.29193633516554644378719792003009074845","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32593583441236180433075435541777683664","series":"1.2.826.0.1.3680043.8.498.50139683263246658337264433994274670388","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32521102053110469817073126425955562934","series":"1.2.826.0.1.3680043.8.498.10452377348451577391643835327018772872","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32624291809937483520460431326945616461","series":"1.2.826.0.1.3680043.8.498.11479498991501258480514559974879681783","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32504785212530110880087745488871181111","series":"1.2.826.0.1.3680043.8.498.56789355613584332423216192400516933593","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32742383730417896527915284404039812924","series":"1.2.826.0.1.3680043.8.498.32445896157027226467480295515746588108","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32835080036019877751472792740274297358","series":"1.2.826.0.1.3680043.8.498.38515043759289363988119652366708954375","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32704159300269838731722725240928883385","series":"1.2.826.0.1.3680043.8.498.19410523687045198998847435653347510007","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32910372391370203605482772577821971133","series":"1.2.826.0.1.3680043.8.498.11317636104982470341075768297472156394","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32417506901165787885027994386927321654","series":"1.2.826.0.1.3680043.8.498.47185743718542085765053851188147469104","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32996271790302034744448837505729856569","series":"1.2.826.0.1.3680043.8.498.37103463075961455113324720996560122768","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33085434860684110795966817803985794300","series":"1.2.826.0.1.3680043.8.498.76040949107742016817507659429000038092","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33215387283623093392227819856970670071","series":"1.2.826.0.1.3680043.8.498.11663780407215559444735901554395501037","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33324606756355240783076971858231422510","series":"1.2.826.0.1.3680043.8.498.13291573991407864263836444114134503536","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33339945632086976351786752793064764443","series":"1.2.826.0.1.3680043.8.498.11819169239048940562521527736325192554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33427080769740419588952225424281396684","series":"1.2.826.0.1.3680043.8.498.29368340292805976536611392301620887778","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33273874534046658504935877940537027702","series":"1.2.826.0.1.3680043.8.498.51382813780046235806619310472261837984","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33393833442961030521286257650328223086","series":"1.2.826.0.1.3680043.8.498.11922712187427140270615515885066348147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33709759248055631658322056320849494088","series":"1.2.826.0.1.3680043.8.498.77269786049211763934643141515950935289","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33614319611050071354692274770073371314","series":"1.2.826.0.1.3680043.8.498.12781753622990062596540104980788460124","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33657688895130083119886800524324572941","series":"1.2.826.0.1.3680043.8.498.12370069882523579113298044669959275135","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33834227140201599591317512187502720859","series":"1.2.826.0.1.3680043.8.498.39754014732866667213169077656627170017","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33479806419966566464727310013949313521","series":"1.2.826.0.1.3680043.8.498.35697967323951117172955835449540732394","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33957165685646407808982253731894148028","series":"1.2.826.0.1.3680043.8.498.95798182876179338941802715006334442225","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33899056854391617633929258302459140830","series":"1.2.826.0.1.3680043.8.498.68839805364998796277344976842497193957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33880013318702285028756482543453984598","series":"1.2.826.0.1.3680043.8.498.68633105141236050965957455528612424146","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34040631818813472592501398076075999835","series":"1.2.826.0.1.3680043.8.498.13092435215559461478334909327855839293","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33997019840409023169542161093833920630","series":"1.2.826.0.1.3680043.8.498.11769637216355690545766976680153701931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34206043272866138622475050809785944941","series":"1.2.826.0.1.3680043.8.498.11091661603319742987606631497522385371","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34294522885703132478119231619149118745","series":"1.2.826.0.1.3680043.8.498.64418367554756344022703749575767442868","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34076627216941578548083498328399240999","series":"1.2.826.0.1.3680043.8.498.11903773933137778219749691463973495613","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34433343440016591348979084714300062954","series":"1.2.826.0.1.3680043.8.498.10517520196509105745861272651855351136","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34379812274299658721362253287768041613","series":"1.2.826.0.1.3680043.8.498.81685730186520461556502150660203967147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34461302710296907289543656738174044465","series":"1.2.826.0.1.3680043.8.498.35173248906102070841454753868707117458","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34109091435857467600499949506940613037","series":"1.2.826.0.1.3680043.8.498.97938180139627115854434120431738071351","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34690162612328981644511154212357885455","series":"1.2.826.0.1.3680043.8.498.50938947752533500979747897270386543391","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34546106114963066554367298449299214677","series":"1.2.826.0.1.3680043.8.498.37764118424218311727028289598391093245","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34654275504962399799861007974465776328","series":"1.2.826.0.1.3680043.8.498.25366793924198908014635813558011177871","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34740783941206992242367609204835771599","series":"1.2.826.0.1.3680043.8.498.87244101764847209485337045242619185008","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34574046861696910828478657346725687542","series":"1.2.826.0.1.3680043.8.498.73520121452697220711760029756363139482","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34826100703045042103752460066224521343","series":"1.2.826.0.1.3680043.8.498.38306391410221340200183992809680611435","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34791780913322886484171421115861604853","series":"1.2.826.0.1.3680043.8.498.58495393402659035444112940594786767289","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34861721301001052975626513770908147451","series":"1.2.826.0.1.3680043.8.498.43897738235626779920657399502788626559","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35026052487843947230089350615730580521","series":"1.2.826.0.1.3680043.8.498.52521355191860194286857109699895371041","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35015675197378142698488944856288804597","series":"1.2.826.0.1.3680043.8.498.44775076637146015471048040406178171002","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35121262802959257590413582180924083425","series":"1.2.826.0.1.3680043.8.498.11644757024158116383086845674659343352","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35257158249961136373696255071228322165","series":"1.2.826.0.1.3680043.8.498.37062642474787867958991379014412576870","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35210451107308788915675910467662426761","series":"1.2.826.0.1.3680043.8.498.26321586476495595675596093299881564931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35526385429946467546636621603749416429","series":"1.2.826.0.1.3680043.8.498.46272407953527446422420930984860569492","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35330802038595377723999341992570259909","series":"1.2.826.0.1.3680043.8.498.44858952835471288494114935421227434310","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35549628666222168741574331855838621990","series":"1.2.826.0.1.3680043.8.498.10183169116260843226207305334567719143","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35633121799529901420381085923067525212","series":"1.2.826.0.1.3680043.8.498.33692677204812643572456293236780345165","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35664741425496814074030858561751287211","series":"1.2.826.0.1.3680043.8.498.73460077712551068068138880415935496483","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35064504053970754752690367203246822139","series":"1.2.826.0.1.3680043.8.498.72661974064666938777339996418447456231","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35777075794856195065665011443485613681","series":"1.2.826.0.1.3680043.8.498.69481714009370371017076668391219611897","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35820650938063925991582813178954225720","series":"1.2.826.0.1.3680043.8.498.57273179488527852592295208019957212676","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35892665011493087075493127252825546314","series":"1.2.826.0.1.3680043.8.498.99138049163872461567609682467144873068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35717294165699019392845830077937768611","series":"1.2.826.0.1.3680043.8.498.27033554724494805150051601440225345358","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35835033603115368194824573270370351403","series":"1.2.826.0.1.3680043.8.498.56983001858921358322058403964910466145","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35853404993418331599021266761280794201","series":"1.2.826.0.1.3680043.8.498.63637996182953115208018242954847239898","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35923146679164443588381195543533271939","series":"1.2.826.0.1.3680043.8.498.23995491990008360127618975285175857846","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35923753382244056317223397206343376734","series":"1.2.826.0.1.3680043.8.498.12816980805622072670533382587033385708","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35965770865730278787229040370963322662","series":"1.2.826.0.1.3680043.8.498.15755898326727804563960998133984841638","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35991881811942962264382690224767291006","series":"1.2.826.0.1.3680043.8.498.18999594658804197255126812494445050530","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36090989845682961399757203739039806565","series":"1.2.826.0.1.3680043.8.498.10528024145341623403106667815977434538","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36035264348264378094569428634302238394","series":"1.2.826.0.1.3680043.8.498.67264241031203544525737013977579197940","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36185754164795053628650116141620545139","series":"1.2.826.0.1.3680043.8.498.62866412471357540900715837583278255158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36158956431673597472934120620368656246","series":"1.2.826.0.1.3680043.8.498.35384541181388836783503715255340813172","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36293583160636884934622362743062825202","series":"1.2.826.0.1.3680043.8.498.12020690197798945699452991496732363958","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36221680997794815863088530960610069278","series":"1.2.826.0.1.3680043.8.498.33050277184766172534340834160143013972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36315811661139684771871696847516662339","series":"1.2.826.0.1.3680043.8.498.46969620048753616814121615771588422379","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35942316720546803304976641081186518384","series":"1.2.826.0.1.3680043.8.498.86455955191431936699966775415654073943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36648377536792309029975432419163533533","series":"1.2.826.0.1.3680043.8.498.46603460332195920659758441063118318551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36521162836904095186654423648490092035","series":"1.2.826.0.1.3680043.8.498.11062891739421655203119402405090469704","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36599566036331923007586730435849419272","series":"1.2.826.0.1.3680043.8.498.12934378810049214494396945865285643577","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36825274340853796392762195174336452919","series":"1.2.826.0.1.3680043.8.498.22669877583859522490061754950669599237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36470828078108352859868482351276699300","series":"1.2.826.0.1.3680043.8.498.89252997423388045143398863865069719380","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36757476214794170118868510630678488185","series":"1.2.826.0.1.3680043.8.498.17702057353200771926314077471785373602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36924091633099385100942228071545017565","series":"1.2.826.0.1.3680043.8.498.77263300413918280349751113580401850551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36891467514776207809384804066253854193","series":"1.2.826.0.1.3680043.8.498.12614022091570838129727813519177489368","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36995206854684735704099881847485073991","series":"1.2.826.0.1.3680043.8.498.12962549911879906038213918752154837541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36688959138536319644194458211515765203","series":"1.2.826.0.1.3680043.8.498.92050542523804105793866083492690576791","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742","series":"1.2.826.0.1.3680043.8.498.23174969622873816054521935098776352184","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37147839676713817130306478240352601535","series":"1.2.826.0.1.3680043.8.498.10901359455482219773843393108852767051","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37257728513136112802363958568529758895","series":"1.2.826.0.1.3680043.8.498.94208301767140589849083610851676702853","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37007302166102452703965686598524467188","series":"1.2.826.0.1.3680043.8.498.74982284925772426894631141971042226769","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37332768072749723117839591872457536315","series":"1.2.826.0.1.3680043.8.498.10944281190307064396601712592036009544","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37057548998853865829402084168010676727","series":"1.2.826.0.1.3680043.8.498.39060003989991136459403351584839828037","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683","series":"1.2.826.0.1.3680043.8.498.36923468181159701148339847602169721829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37427993203943111265113924891058049338","series":"1.2.826.0.1.3680043.8.498.47307356483280869056840524443594676638","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37438631765770807892914198396852544668","series":"1.2.826.0.1.3680043.8.498.36523088667957457301244679161051099964","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37508036912473928281276902883575142611","series":"1.2.826.0.1.3680043.8.498.50773273053517407621620396684232760415","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37572710131396388050286115883363501023","series":"1.2.826.0.1.3680043.8.498.24717616718499559248914182805066436976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37371365833775442014007234936409739360","series":"1.2.826.0.1.3680043.8.498.56917566336345063898367545413276766955","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37519350121906786941281770770693979271","series":"1.2.826.0.1.3680043.8.498.48657031384001598828512652137108630572","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37591186611460576939009950935820113903","series":"1.2.826.0.1.3680043.8.498.12758801145097983461045801008115363747","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37633035024065331400434438202028794597","series":"1.2.826.0.1.3680043.8.498.22068887316070570188227062451016278149","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37695832484726177391252504639155717084","series":"1.2.826.0.1.3680043.8.498.95266284460756026528941025031087660767","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37729566534337521443213426693965453255","series":"1.2.826.0.1.3680043.8.498.44915460737536039247206775697091309711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37833587429731221455928642963031995680","series":"1.2.826.0.1.3680043.8.498.33502970026548879077669041950507260816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37846940236952007502147453013489359606","series":"1.2.826.0.1.3680043.8.498.11292337056688932769833796228810491998","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37879265071349098151803248077034280617","series":"1.2.826.0.1.3680043.8.498.68828232655620055874109853252848614663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37463970179664003092832463732043124908","series":"1.2.826.0.1.3680043.8.498.73413009928147915260677164086026165738","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37921127120678939078868474340027123335","series":"1.2.826.0.1.3680043.8.498.93296394352435133152225703724099502373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37969790195791584440906553207150666923","series":"1.2.826.0.1.3680043.8.498.37088326434413359672683885464292947055","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38010459551726691601986492657296948013","series":"1.2.826.0.1.3680043.8.498.13075232786895046857981113440068012018","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38069035744880456154810472748123499221","series":"1.2.826.0.1.3680043.8.498.11244454995410171214652450632367189707","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38039631093067555454700266397073080430","series":"1.2.826.0.1.3680043.8.498.70073707803775716546213676851185016662","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38095098835657518803781210111463995296","series":"1.2.826.0.1.3680043.8.498.11948798440705762217496123680048541394","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38233845434092134418342290639986261345","series":"1.2.826.0.1.3680043.8.498.10308716421492398115622165016146518941","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38272345519163551901865251730721530169","series":"1.2.826.0.1.3680043.8.498.52068254480549997354787465024921026056","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38303307720827621886420004896097661262","series":"1.2.826.0.1.3680043.8.498.37708820621178246108295750570746422126","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38173249623713271734596884672770674290","series":"1.2.826.0.1.3680043.8.498.11364161624533341042650586367298779714","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38325494489236992123219618004392575979","series":"1.2.826.0.1.3680043.8.498.12789087629724310543219429675294822019","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38427680930662301114332563903957471851","series":"1.2.826.0.1.3680043.8.498.10686467261406494801445631247948799710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38373204312077453422262347769111605003","series":"1.2.826.0.1.3680043.8.498.14411142684494715965273920030404249637","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38361762550587464310603057450403132552","series":"1.2.826.0.1.3680043.8.498.11102612150853431038852466897191813553","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38485168845116671268554752030254772359","series":"1.2.826.0.1.3680043.8.498.12673314975077674643732942183077047192","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38528511947818361078957998546327084024","series":"1.2.826.0.1.3680043.8.498.63003634217601976738808993258889942994","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38655145317957160440692625317823254736","series":"1.2.826.0.1.3680043.8.498.86026565195151832959498872990178805673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38591915802992557518846862828618375715","series":"1.2.826.0.1.3680043.8.498.34219717417201853114137731199206568304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38617606118499305781729905815407382451","series":"1.2.826.0.1.3680043.8.498.82730936890028170315155653412071363142","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38702677049080452580748506476598606517","series":"1.2.826.0.1.3680043.8.498.98726920095602376402169723234889785811","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38795933423959135237935740989825843893","series":"1.2.826.0.1.3680043.8.498.71067050742571428933430484000115332729","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38844149478205202385909887748058004820","series":"1.2.826.0.1.3680043.8.498.39960454289580310407361733465088811603","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38750761816114659458563557493234655604","series":"1.2.826.0.1.3680043.8.498.12092852689210078304072738013346524852","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38898732985465285166803609252960353405","series":"1.2.826.0.1.3680043.8.498.12781084861184560853836091117866414206","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38567342184301320850258918666303107813","series":"1.2.826.0.1.3680043.8.498.10121662642504419892894935579682497587","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38870467957974446108337293912930572060","series":"1.2.826.0.1.3680043.8.498.72859942099489335639725461194121129042","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38912715505731269998129663552068223373","series":"1.2.826.0.1.3680043.8.498.14601432495901784810206395370007826385","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38981318408220047366318316620365040799","series":"1.2.826.0.1.3680043.8.498.16550537561954660595261771082511205744","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38940868544480800333574786964232944756","series":"1.2.826.0.1.3680043.8.498.29905086090585880938661496899167014043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39020232951771672617852805188845504161","series":"1.2.826.0.1.3680043.8.498.13383725820713325121266739450871062380","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38928323954764175259962816410233336718","series":"1.2.826.0.1.3680043.8.498.13373743097879238295343319947320595638","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39250844287482885227700969308333354206","series":"1.2.826.0.1.3680043.8.498.11554524531439141707983948605193611910","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39190090804401525573012570218955054016","series":"1.2.826.0.1.3680043.8.498.37707648632005820632011267755728776836","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39070169128115370701189151138307035053","series":"1.2.826.0.1.3680043.8.498.95417539995422221553409355320944708295","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39275838957651450909114440721012584183","series":"1.2.826.0.1.3680043.8.498.87423667004286956370813410258114526478","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39064406065534097226924475518005522427","series":"1.2.826.0.1.3680043.8.498.11270689472593996100454696434182679286","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39399668753650041254826892657562722764","series":"1.2.826.0.1.3680043.8.498.11677656723190179504584054516364428954","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39375796600811714870169173423682043709","series":"1.2.826.0.1.3680043.8.498.79604097480577899455257123157291489279","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500","series":"1.2.826.0.1.3680043.8.498.71452782593498755829744891501612633384","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39624624539323722688093816284210287611","series":"1.2.826.0.1.3680043.8.498.10250583930723173809154682754146927715","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39642276506855996181597769215091644995","series":"1.2.826.0.1.3680043.8.498.25897197573728223352799020378641508583","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39799597341675508044808282472428242917","series":"1.2.826.0.1.3680043.8.498.18686826006678133510693973875485801045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39875636232483792306109110284167439592","series":"1.2.826.0.1.3680043.8.498.11013862647945118995152311807233506975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39547752108509751520306190293259313911","series":"1.2.826.0.1.3680043.8.498.93618359083481863877804067877428893707","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40036526071307024601256787605725348112","series":"1.2.826.0.1.3680043.8.498.37292664067179917354354278582559110793","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40086005762552976451418579375198597540","series":"1.2.826.0.1.3680043.8.498.47861105241521745954023415414194659097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39959632509842689058626880477779808473","series":"1.2.826.0.1.3680043.8.498.10409579718387365869407238377696316835","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39932424305167611414527482641898907759","series":"1.2.826.0.1.3680043.8.498.65526307042725886629991487165971534833","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40154708689850542672289658249059660788","series":"1.2.826.0.1.3680043.8.498.18262270406427638912565391885299027831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40269807401373034066229810558300266830","series":"1.2.826.0.1.3680043.8.498.49800901200137871773894950020909193985","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40254123984175499564145728084464192446","series":"1.2.826.0.1.3680043.8.498.10310175537935591892484509015474430664","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40223239401390462911823193489179244754","series":"1.2.826.0.1.3680043.8.498.12784973474686413928963162620840307787","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40317527233615709136328116805254969837","series":"1.2.826.0.1.3680043.8.498.88996434018101409792291168903227608329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40441879276927383122480339472406105572","series":"1.2.826.0.1.3680043.8.498.82830577215880165883617810083259840751","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40386781168330493993852288092654487818","series":"1.2.826.0.1.3680043.8.498.12224357956587373596834905197001608976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40436886205373732563782477802719203679","series":"1.2.826.0.1.3680043.8.498.84786823210501751449532019160534682143","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40533893516114346078820881178450524201","series":"1.2.826.0.1.3680043.8.498.36064638631435791234876242094198393057","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40576954651310602807432132614823906315","series":"1.2.826.0.1.3680043.8.498.24995472757908429609094792398425977631","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40612173652156509478338094582396889655","series":"1.2.826.0.1.3680043.8.498.99009659675479925820730844991598711125","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40627043074325458726412346974457616938","series":"1.2.826.0.1.3680043.8.498.58865718343375516309320108798512146002","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40703974473944295875248011843170371394","series":"1.2.826.0.1.3680043.8.498.70427611448426663131001084053493803493","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40723835064354784133289444204866484476","series":"1.2.826.0.1.3680043.8.498.11280238824949571761802970991063413595","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40823842819404259265489417470859248075","series":"1.2.826.0.1.3680043.8.498.91494856702384212339401827785618009677","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40837296112545776586904280088902000849","series":"1.2.826.0.1.3680043.8.498.81033936903305426129789810324146165545","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40910368031948275476880475208016058754","series":"1.2.826.0.1.3680043.8.498.48014958400238537894997795114650658516","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40967435550546093348221485942169879116","series":"1.2.826.0.1.3680043.8.498.61802829446164939954356399113348919566","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40994616246264625946610553907227975893","series":"1.2.826.0.1.3680043.8.498.39626163178438023645080773465990705447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41124244151692842048993594448992046260","series":"1.2.826.0.1.3680043.8.498.10580735046872528058385237783612283047","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41057428195711190931490429649014776496","series":"1.2.826.0.1.3680043.8.498.65299757310885804324634129368291226912","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41197678541534079475667407055098243437","series":"1.2.826.0.1.3680043.8.498.83906280196662902178941315615477060896","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41366889271483886167284378157366992074","series":"1.2.826.0.1.3680043.8.498.96552084221529327972702114860465505155","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41326575785634778198143222323166756210","series":"1.2.826.0.1.3680043.8.498.33150118058569713040300577326827793578","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41254472410580125134596211010943842213","series":"1.2.826.0.1.3680043.8.498.33551558281661382017275298982420356816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41439889720274233955076963938741061358","series":"1.2.826.0.1.3680043.8.498.11341088182141161875054344787850820380","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41454762422897656673710145732036115764","series":"1.2.826.0.1.3680043.8.498.62869956929700621394915734143384384286","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41657361598083583047831423768034379414","series":"1.2.826.0.1.3680043.8.498.97332974008144584905635222628806391407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41587640208769641326416798167574152049","series":"1.2.826.0.1.3680043.8.498.47158657277873538940752615960387390178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41623512571866040054960002151226127775","series":"1.2.826.0.1.3680043.8.498.67661320429290212989445761026899428235","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41605033773891953796694219242501252567","series":"1.2.826.0.1.3680043.8.498.37391578895940181801495865849904375989","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41672145969790132883562432403529293545","series":"1.2.826.0.1.3680043.8.498.11847666308817702287808962478797661284","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41751049067043481681879071436091069601","series":"1.2.826.0.1.3680043.8.498.53853666157942525831542309619219610245","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41698150452794914665945455397334605778","series":"1.2.826.0.1.3680043.8.498.61632024638081900832246913333778939144","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41776233288569201013095816839442161537","series":"1.2.826.0.1.3680043.8.498.56380485730216550785108458691390410649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41837648239243141231469586963862388683","series":"1.2.826.0.1.3680043.8.498.67309568703184788857141501756251202462","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42018410640565840140895012243840256810","series":"1.2.826.0.1.3680043.8.498.15739488975683791426193116366206813767","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41854610297797849683195403204758363476","series":"1.2.826.0.1.3680043.8.498.21808505006242010596512009279448125521","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42057349781448469704191384429694247725","series":"1.2.826.0.1.3680043.8.498.65304877352852657349564423169901859775","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41939824564209194493855981805593285528","series":"1.2.826.0.1.3680043.8.498.84616875369835756802107761091818578246","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42130173624033946465704266507948122088","series":"1.2.826.0.1.3680043.8.498.11412114188991364342152465178757843746","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42221373360893091454504422964928099406","series":"1.2.826.0.1.3680043.8.498.76511823524387592146387475260080757518","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42287188181305579531212818123200473947","series":"1.2.826.0.1.3680043.8.498.38381649716183582882450994852297342688","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42023788139360998656558751921918860808","series":"1.2.826.0.1.3680043.8.498.65855057494860048812207293393177993890","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42476443240785038044326483794941265936","series":"1.2.826.0.1.3680043.8.498.10938175170104981833998866331085660186","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42396216793489016293032290339618185275","series":"1.2.826.0.1.3680043.8.498.87913976315251187607175617140718120170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42499092649567991851673654940784921876","series":"1.2.826.0.1.3680043.8.498.11146120919470998861440232300258166338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42580107838226807862930203257414265276","series":"1.2.826.0.1.3680043.8.498.71960465090376842290219404972189035509","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42598518026540237824479232797503218019","series":"1.2.826.0.1.3680043.8.498.91772367183191673434894356444082830372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42768603719459834345001093623865167866","series":"1.2.826.0.1.3680043.8.498.88779093891245670998884099602175810203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42634709955067416383713194799683605006","series":"1.2.826.0.1.3680043.8.498.26372804217768507853681254762469616051","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42778524243264061817587398263842942394","series":"1.2.826.0.1.3680043.8.498.25569737874638315783833942901481297372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42829490507052740062127365747846030214","series":"1.2.826.0.1.3680043.8.498.11752746995614411058737668001690335024","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42887028634948156598417204377884297339","series":"1.2.826.0.1.3680043.8.498.10242333489462409639931468061076863219","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42965031665568825650795822006244115609","series":"1.2.826.0.1.3680043.8.498.11909645000614859072661152742339904176","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42921593713351589642747633505679170597","series":"1.2.826.0.1.3680043.8.498.43209600700224981347005422372706678355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42990542863216044717944418712887800295","series":"1.2.826.0.1.3680043.8.498.45240005126882461974316801792740032103","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43046713539641995841029654207663845887","series":"1.2.826.0.1.3680043.8.498.46462143848599944195400469956234564374","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43117007553940146763245418710268028741","series":"1.2.826.0.1.3680043.8.498.11248293529657977508104948565598663692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43144544376687968624598661376472625623","series":"1.2.826.0.1.3680043.8.498.11418148177618967627189747964186786438","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43325226037168674077736336919956196684","series":"1.2.826.0.1.3680043.8.498.11530231876373091523110572131095761183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43351215447181735764667371462573550267","series":"1.2.826.0.1.3680043.8.498.10459360829898188957672817058277043084","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43256905750043156063416845850825787417","series":"1.2.826.0.1.3680043.8.498.10014196852555980013194055310655549742","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43412154740072739455825163326814980838","series":"1.2.826.0.1.3680043.8.498.91688502205805231746324292208084535052","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43445351322695981909807931874796758490","series":"1.2.826.0.1.3680043.8.498.28193929872138406180225173799726241332","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43587191909319947256183198986671106743","series":"1.2.826.0.1.3680043.8.498.70339529634787008447204466610926422273","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43692339773096477132070133629776394739","series":"1.2.826.0.1.3680043.8.498.34593814699378306819029114070519300446","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43439413368880313042951208701983487343","series":"1.2.826.0.1.3680043.8.498.66730912505808417168257915838368925920","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43779228281849457650343989788070421822","series":"1.2.826.0.1.3680043.8.498.12314521641474229177711529709912818696","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43832771919542075229536000564653592465","series":"1.2.826.0.1.3680043.8.498.42818529651136909191339331554344996305","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43967752580489865323542041346927244370","series":"1.2.826.0.1.3680043.8.498.11211466512337820733032613684207233436","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44239214982361627984688924889728201208","series":"1.2.826.0.1.3680043.8.498.37041800573492571824518905876032004722","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44024350801191362011463613321559949052","series":"1.2.826.0.1.3680043.8.498.49622014172976904703418228614287939737","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44323131330689365651417881352766725366","series":"1.2.826.0.1.3680043.8.498.67340940015635996196998489512487508398","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44350638600163778456163342454087669278","series":"1.2.826.0.1.3680043.8.498.23876817236344700315144156320383610551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44405554266141576380633014711246118209","series":"1.2.826.0.1.3680043.8.498.12813609970531935880870974030102323620","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44476475746458439091354236934188679569","series":"1.2.826.0.1.3680043.8.498.85669732293730958937797748903065291468","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44261256084247541720465769177956875383","series":"1.2.826.0.1.3680043.8.498.46031105625448843209596424295004681063","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44424196147366603683090692394096032727","series":"1.2.826.0.1.3680043.8.498.12432211029005585935535284454363352267","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44629238055006732101555855930071163428","series":"1.2.826.0.1.3680043.8.498.12634449308435120288192845638483377482","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44733200650377282012534389271385687852","series":"1.2.826.0.1.3680043.8.498.63294953327974762443398980661481732831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44792591585917525576461101442761037507","series":"1.2.826.0.1.3680043.8.498.26808274135574678180910653402264416815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44670636848627131733157841382853275817","series":"1.2.826.0.1.3680043.8.498.63623212065038299734930944150141057032","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44875593707781066742286566388388035791","series":"1.2.826.0.1.3680043.8.498.10813808541391234510347804590761802989","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44941174190375343987327566550204051010","series":"1.2.826.0.1.3680043.8.498.55679553696082108073087926877155693871","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44879493336213912007673875382158420811","series":"1.2.826.0.1.3680043.8.498.75602104737677180486775697803007448216","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45033326048521572717771708818514772861","series":"1.2.826.0.1.3680043.8.498.47415404664182899263803549378674925712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45072896942121803896760563154385918032","series":"1.2.826.0.1.3680043.8.498.45377129434639142257390125991722644142","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45207812477654339877341538582090218293","series":"1.2.826.0.1.3680043.8.498.11870819330895659838485694086349279513","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45118217412917282413100913670807141634","series":"1.2.826.0.1.3680043.8.498.22671011449335800095105347200192288812","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45367987334597713963053051542179369641","series":"1.2.826.0.1.3680043.8.498.18774352291692162618374228737624304642","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45326662718293919909188368237039419949","series":"1.2.826.0.1.3680043.8.498.50934920277802346564618860597788786439","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45304570280714567513866386380356988876","series":"1.2.826.0.1.3680043.8.498.16119541353240457512098376060532796704","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45273393220765922863913956738614783632","series":"1.2.826.0.1.3680043.8.498.41681971461305169339232132998123186995","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45406212529450909512649737062964080190","series":"1.2.826.0.1.3680043.8.498.74485896509928837768523391330719738546","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45491168638937605395466251659273753775","series":"1.2.826.0.1.3680043.8.498.84523037443942605383800219042173319110","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45462389009834095091353507592651342877","series":"1.2.826.0.1.3680043.8.498.67089387632615979198523119438719435543","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45519502487291848581539950159353124045","series":"1.2.826.0.1.3680043.8.498.13226766539418027648368523696684350727","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45380312679924593883892686293576260616","series":"1.2.826.0.1.3680043.8.498.70465950970550302715600339858606396072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45678969592219293243366784683191701674","series":"1.2.826.0.1.3680043.8.498.23828495233746103725927389344211870827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45768777378108699271714894225826640121","series":"1.2.826.0.1.3680043.8.498.15979542635244081240996274375123844554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45630051398709774279703407401383545642","series":"1.2.826.0.1.3680043.8.498.44412901608619173277718686478159675229","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45862157701704991290669186265816463426","series":"1.2.826.0.1.3680043.8.498.20452625936182345337794961902666496774","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46016776602328665980436110567854612595","series":"1.2.826.0.1.3680043.8.498.43573827710098200422611514599732533071","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45968957856865486437256472516713628614","series":"1.2.826.0.1.3680043.8.498.79682100121466279871907642104479636640","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46051119885630602872050329412008545893","series":"1.2.826.0.1.3680043.8.498.11878321781274509625920460709000314871","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46123761840464615814604226041496278995","series":"1.2.826.0.1.3680043.8.498.67023124743321172102836018847820895876","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46259494895672269475940484605821837768","series":"1.2.826.0.1.3680043.8.498.84271951417752430155883698568044421381","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45868806544202339567395643235910857549","series":"1.2.826.0.1.3680043.8.498.11271850341341224124461762254750898588","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46222691720198967161306520058359099856","series":"1.2.826.0.1.3680043.8.498.12113823066210990212493196377263012861","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46305694398420300356121078124030126469","series":"1.2.826.0.1.3680043.8.498.11047372638370056735794466428153179990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46427879286435198611282055405541286766","series":"1.2.826.0.1.3680043.8.498.43196298385659073184927516819770116237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46282234455466954264170065462242039568","series":"1.2.826.0.1.3680043.8.498.62555601011259795983364825754687726842","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46658713832425741743740258795376270265","series":"1.2.826.0.1.3680043.8.498.21611040016765592924854173612952056847","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46713978512945139851373191422321462143","series":"1.2.826.0.1.3680043.8.498.21281159070928517089572538261240352316","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46335849264221717416382765091672040855","series":"1.2.826.0.1.3680043.8.498.52973597934291874601079200754030025785","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46745172905542117506863575411125924241","series":"1.2.826.0.1.3680043.8.498.79376558575824485414324592009535389094","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46491051379327985199309365945879841064","series":"1.2.826.0.1.3680043.8.498.10962015803858024802094074984979633504","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46825768767498441994570613185981335949","series":"1.2.826.0.1.3680043.8.498.64403274006390026720135268266782369357","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46909729013880708074310736136355969956","series":"1.2.826.0.1.3680043.8.498.11257721613073637523317832018091193380","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46776367027887983105817631582518625639","series":"1.2.826.0.1.3680043.8.498.51170803576419176243354568428883605520","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47061252152535181371772020997295000029","series":"1.2.826.0.1.3680043.8.498.83055380866142244102336620095225212158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47080537724499166054609508776778189902","series":"1.2.826.0.1.3680043.8.498.18259829811596124361148635120228492385","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431","series":"1.2.826.0.1.3680043.8.498.10632531192811899276510557763004424886","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47268599732083036147289040424392838620","series":"1.2.826.0.1.3680043.8.498.51747954664454310413324813235861297047","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47255564202539432049700324486197024855","series":"1.2.826.0.1.3680043.8.498.83331010660039849502380127709886784616","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47449893776425045020340604662487159684","series":"1.2.826.0.1.3680043.8.498.11916006522500570183747265982000158430","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46884541791777826238103426426267571657","series":"1.2.826.0.1.3680043.8.498.28437182087896443584321597116360338205","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47339093185444044578247608022398494955","series":"1.2.826.0.1.3680043.8.498.19280031721707746582650786478763752982","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47495385306095610608327984787779011133","series":"1.2.826.0.1.3680043.8.498.12412539547593641673985309308488364836","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47666607805434428477774854287399691400","series":"1.2.826.0.1.3680043.8.498.18728301867243170650298718200439958539","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47707670074001059720671838493302874710","series":"1.2.826.0.1.3680043.8.498.16651761456914811889696402586204574804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47651715105501347255071941051553742767","series":"1.2.826.0.1.3680043.8.498.59680999638825047371946289358267805323","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47784065888386216103165400551121771174","series":"1.2.826.0.1.3680043.8.498.47138658445090985884214102064084236350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47130872195336642107576758658144949813","series":"1.2.826.0.1.3680043.8.498.37044297722480265629717237838090266975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47864423097056282241181943425451597067","series":"1.2.826.0.1.3680043.8.498.14887148912290217410835331780602303115","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187","series":"1.2.826.0.1.3680043.8.498.69542398721543718229644391277314854342","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47949251060660360953300340380324578835","series":"1.2.826.0.1.3680043.8.498.53423098088174479480562356174820339737","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48160569779413970442472500116176876660","series":"1.2.826.0.1.3680043.8.498.52842432862031892477596696350328156269","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48068031587855513429324313927339963541","series":"1.2.826.0.1.3680043.8.498.18426004358039155482770204957622047940","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274","series":"1.2.826.0.1.3680043.8.498.76119594349644984197513470829331200414","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48435973333068748044485658920315789799","series":"1.2.826.0.1.3680043.8.498.96211909765176921244224688695324865278","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48531746264504523370547117933271161633","series":"1.2.826.0.1.3680043.8.498.54768629829189679029279045086896157944","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48658459018915173972212724570862424208","series":"1.2.826.0.1.3680043.8.498.73185015553836734640407117245642883427","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48368929760320103375578806071106317953","series":"1.2.826.0.1.3680043.8.498.74180541198861428297338973872296978112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48750912584073496502856009873745362236","series":"1.2.826.0.1.3680043.8.498.10527372249218195962305749228173275945","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48780598610511011488578963416186905560","series":"1.2.826.0.1.3680043.8.498.12422818164319649865913946366507108072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48818931767017230678539144135307491171","series":"1.2.826.0.1.3680043.8.498.36744020597283645288197728112470981487","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48520238688122533624681796433258211236","series":"1.2.826.0.1.3680043.8.498.17404727553361386264373847245843664551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48855562507887692822284056171390459619","series":"1.2.826.0.1.3680043.8.498.22711502467844567048975634006083779337","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49025249926859976858200443939208207124","series":"1.2.826.0.1.3680043.8.498.84250424290209744431201057465672524135","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132","series":"1.2.826.0.1.3680043.8.498.71137653521806972309735604555464864164","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49048739684287632681242054218807611262","series":"1.2.826.0.1.3680043.8.498.45797037706269478415415830230138676456","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49094276335794811195117175603556929507","series":"1.2.826.0.1.3680043.8.498.32272180637884212608343547266133429981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48918090379630186340047006216221526186","series":"1.2.826.0.1.3680043.8.498.64867153997540181282377059135231393379","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49287143478474218875661109488367501329","series":"1.2.826.0.1.3680043.8.498.52351258951764795464803876798412708534","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49441835811239898312243678447796777305","series":"1.2.826.0.1.3680043.8.498.62057904362113150644539753685442148765","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49379034023506626029648394840207360925","series":"1.2.826.0.1.3680043.8.498.13114880851576137559449532162041050050","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49186859342644783675130322288585477727","series":"1.2.826.0.1.3680043.8.498.64336887286026816100748763171355654787","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49495644703047055149965226028097259207","series":"1.2.826.0.1.3680043.8.498.56338127657109507002211746533833691648","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49481811111224238761880576486796603139","series":"1.2.826.0.1.3680043.8.498.42494308275565791127296148882195214160","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49818874332345742217581267858748819921","series":"1.2.826.0.1.3680043.8.498.44352556073252636373656389476736278822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49603554452147117475589797389254733654","series":"1.2.826.0.1.3680043.8.498.12339236012351991087811177600357917477","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49679916686449433962263937489400526112","series":"1.2.826.0.1.3680043.8.498.47238200259005827026679449390227965808","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49575149159425506211571562732929634662","series":"1.2.826.0.1.3680043.8.498.15966069115711994596266576338059597502","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49866252311077397175807461727885923657","series":"1.2.826.0.1.3680043.8.498.95647887673248837374073341036050591673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49850672111501136862025523789202033809","series":"1.2.826.0.1.3680043.8.498.54314843332224250605242567210536967329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49884243466758407760126403944842240776","series":"1.2.826.0.1.3680043.8.498.56958173450301824914389366788316639339","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49926524898703962863714803838421958913","series":"1.2.826.0.1.3680043.8.498.10683197776482518663080857392391065047","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49984662916977328110175445895515541267","series":"1.2.826.0.1.3680043.8.498.68599807230663939721427271281798832075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50089015780087008332657793589955395952","series":"1.2.826.0.1.3680043.8.498.68890713806740299232749171555089841045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50182378358892377789975570268512056553","series":"1.2.826.0.1.3680043.8.498.67587082438954037521961705234579379384","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50109255875403812845391217314726500341","series":"1.2.826.0.1.3680043.8.498.23036415776479210052149982839289795648","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50215409013618353346619153290049205718","series":"1.2.826.0.1.3680043.8.498.12628176075712516429651985414499707676","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50261788831539215224639813005933285052","series":"1.2.826.0.1.3680043.8.498.32069151288254303813745246581743069688","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50318760104506317341991314680640521601","series":"1.2.826.0.1.3680043.8.498.12053566315116574221917468139728402480","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50069224337753933811759198830086956116","series":"1.2.826.0.1.3680043.8.498.96711627810353112042032162256642799723","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50248717336781045512984550029203786086","series":"1.2.826.0.1.3680043.8.498.12618533659788697569268134592522883107","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50458745240631577290185672298591447264","series":"1.2.826.0.1.3680043.8.498.12329731090121469197061872962790358505","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50501020658943435430736717390350999395","series":"1.2.826.0.1.3680043.8.498.13065138465814951868935397318556890143","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50571692547974449396635351108344029536","series":"1.2.826.0.1.3680043.8.498.12497431191064628576251786354665859470","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50666048236278074641860077012751071758","series":"1.2.826.0.1.3680043.8.498.95500496670269849890981752046260012268","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50680622091377404463269575858175673158","series":"1.2.826.0.1.3680043.8.498.94837910259507808780855085777444944339","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50689256431939553935971505794681061679","series":"1.2.826.0.1.3680043.8.498.12544013351943942030727609430859035719","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50707407856055788475528375847872674241","series":"1.2.826.0.1.3680043.8.498.13339143510757473604428481078441579931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50813123035221058777203670043892828582","series":"1.2.826.0.1.3680043.8.498.11123721473140515656072730818953353924","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50780438941220579448947609376330661130","series":"1.2.826.0.1.3680043.8.498.44637539377912746699287484985091363744","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50392110746914984873009735588879049321","series":"1.2.826.0.1.3680043.8.498.10961393130652586253371912064444508918","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50823279853002320074080586693604976784","series":"1.2.826.0.1.3680043.8.498.12599913938282165902401840019682785301","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50848799900564667487859253004538870622","series":"1.2.826.0.1.3680043.8.498.29237526667222563892026973094890482474","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50882628969985977810341532747520504841","series":"1.2.826.0.1.3680043.8.498.16206651435564591742003363880445758684","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50969691234820675847736765735133138324","series":"1.2.826.0.1.3680043.8.498.14605291189704223749101725969956020809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51059575642903646008860869691477305022","series":"1.2.826.0.1.3680043.8.498.69330826409998184957038959306587850721","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51143994849141407980459861759302509053","series":"1.2.826.0.1.3680043.8.498.13013687791149465050047998632211985598","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51215847567976999838748705030071176367","series":"1.2.826.0.1.3680043.8.498.46053272074776573480388347763940491521","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51351386796317851725461803790273870995","series":"1.2.826.0.1.3680043.8.498.10316562902851956594043227459660665899","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51426696939581135481763379246368970360","series":"1.2.826.0.1.3680043.8.498.11893732387470280974464876066930374509","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51006813205605862503064447831381009425","series":"1.2.826.0.1.3680043.8.498.42038167025348624022730622708921580218","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51429653138043597611341459799295143598","series":"1.2.826.0.1.3680043.8.498.11572325954953082513736792653930830327","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51509214978951980397275258374070843967","series":"1.2.826.0.1.3680043.8.498.10769002615066788558001754451958070342","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51581188134109925758379838961317006116","series":"1.2.826.0.1.3680043.8.498.98889130881569759116298517963198446148","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51684117251092766010524179062007524257","series":"1.2.826.0.1.3680043.8.498.20120815116931082766720813121595457462","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51768733026033198941231586462443503827","series":"1.2.826.0.1.3680043.8.498.38683524561100870856992131429954837270","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51831304809871808790657397599860793132","series":"1.2.826.0.1.3680043.8.498.68408272106519202566600855372632214816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51813125531970590883098432751115674643","series":"1.2.826.0.1.3680043.8.498.10137691260535600008030582217198753711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51782185934115000642557689300687723099","series":"1.2.826.0.1.3680043.8.498.32653608869067183940622810464481248832","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51970198810758997428998886534742243215","series":"1.2.826.0.1.3680043.8.498.67829055696932741644916395299742952718","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51867133375333949773527521703402111601","series":"1.2.826.0.1.3680043.8.498.14890193958380391952407071618599707993","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52010826440192227660457107121818389761","series":"1.2.826.0.1.3680043.8.498.10170848734809310421229954287583294082","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52059522418267239652374290810994711032","series":"1.2.826.0.1.3680043.8.498.12970269837582163643258267604746024026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51910852739185117358682097116614556910","series":"1.2.826.0.1.3680043.8.498.11078243344151280200955920255746880192","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51985313438970133264846259111827877590","series":"1.2.826.0.1.3680043.8.498.80601749966737293126822961230184591647","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52098249444968803324423421624756966294","series":"1.2.826.0.1.3680043.8.498.93908436056411444186693339943723977228","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52132258638267172087240021957310219977","series":"1.2.826.0.1.3680043.8.498.84818849677404113446285675508135056295","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52330656126848153803758928167944040055","series":"1.2.826.0.1.3680043.8.498.39737117933080303812462019605440157701","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52390908285903728334239836699232920065","series":"1.2.826.0.1.3680043.8.498.11440709647104969966690033959412430943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52140911874747534807033188343849537207","series":"1.2.826.0.1.3680043.8.498.65384049635301144269159561453150002494","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52435631723314823200374985065756182593","series":"1.2.826.0.1.3680043.8.498.16189858906256822633822306034391637273","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52499379042723926379822784411787551701","series":"1.2.826.0.1.3680043.8.498.46021939857207805904632114321514636979","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52558822670236188507569589448789513993","series":"1.2.826.0.1.3680043.8.498.10250205817854620519563411448379373897","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52636981391578580436180193211113340410","series":"1.2.826.0.1.3680043.8.498.42308675079270614437622063177767949057","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52456272643822591755257802747431239152","series":"1.2.826.0.1.3680043.8.498.83323328564104911658335475989003883277","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52745530063404662086111315503240395535","series":"1.2.826.0.1.3680043.8.498.99167880137217114161995231517127545439","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52846236545327527967419125989598675101","series":"1.2.826.0.1.3680043.8.498.89051172122271832128897566569996831827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52844966504845792334740560021996858939","series":"1.2.826.0.1.3680043.8.498.74032084237784812771690418539772317052","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53025489296610571044246428509154950855","series":"1.2.826.0.1.3680043.8.498.90707909662265171042146354215402405481","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52937412433120578046842194545406797236","series":"1.2.826.0.1.3680043.8.498.92204926883223207616638287125167373538","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53094450151502907325041400000717725711","series":"1.2.826.0.1.3680043.8.498.11959360677387766153956951509499371016","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53156118232564605912649705207438612782","series":"1.2.826.0.1.3680043.8.498.13143288708576903121337331136279204555","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53131430415327479511358970615140982847","series":"1.2.826.0.1.3680043.8.498.38429240521392438292783897633933138800","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53174810138171412044410050984734412146","series":"1.2.826.0.1.3680043.8.498.86933529573392632516575863771248775592","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53078461307849490495536549806091931824","series":"1.2.826.0.1.3680043.8.498.54043093137449909397253705485766891219","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53331626867618182966740022261110760133","series":"1.2.826.0.1.3680043.8.498.67377713619104366528757124137570016916","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53343379040681745197301575417064236501","series":"1.2.826.0.1.3680043.8.498.13343682341871016534458114897824337251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53361558447470435467999344560545063264","series":"1.2.826.0.1.3680043.8.498.13157258277738941835157388553110862312","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53393741562201846141435914434030220216","series":"1.2.826.0.1.3680043.8.498.11347857453911464043261797811326343183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53490080963614272234072703760363052476","series":"1.2.826.0.1.3680043.8.498.12161311395632993293125729526134485579","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53737349894831365159124471910871800583","series":"1.2.826.0.1.3680043.8.498.10610164835215280190897757349629241615","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53579416035647672067965892020505016454","series":"1.2.826.0.1.3680043.8.498.49422652592385966719705189641408974106","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53262958748075737242820629127001253146","series":"1.2.826.0.1.3680043.8.498.55361082686815836481833699128493681355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53628897373416863107623485630264501760","series":"1.2.826.0.1.3680043.8.498.87452799272932473646208614256848400410","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53792159922324410857755054939229546555","series":"1.2.826.0.1.3680043.8.498.11309993371646943970624784857279720880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53884715696186989798696544661396487883","series":"1.2.826.0.1.3680043.8.498.14921322393910937247312585249703109618","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53968943793571323467397128102531786394","series":"1.2.826.0.1.3680043.8.498.31568774426265484835559853532424851832","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53901077733518036755308692304268815965","series":"1.2.826.0.1.3680043.8.498.27706960862546627780791178076944673197","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54030363830629950591221870797594277092","series":"1.2.826.0.1.3680043.8.498.82366608131282271044299469149685051883","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54099199514076260519579096546600505763","series":"1.2.826.0.1.3680043.8.498.72781641234111085946942218587234767734","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54164004416963296439322252627774426375","series":"1.2.826.0.1.3680043.8.498.10391827751217329197423351364975724681","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54270039578867689669466339980682684126","series":"1.2.826.0.1.3680043.8.498.49724356452476726386330211236910087941","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54306467780023777062920737767581126383","series":"1.2.826.0.1.3680043.8.498.72267086364052333440899680309602663844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54336962625483046366337626878609850219","series":"1.2.826.0.1.3680043.8.498.23837340455610604725977598722018939302","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54138747504377266740838999528964336002","series":"1.2.826.0.1.3680043.8.498.59810640292708757714261450814133744148","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54394066317295451515247748115888976860","series":"1.2.826.0.1.3680043.8.498.90460618934739910206089272981524548487","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54448516923758211534546401490623824012","series":"1.2.826.0.1.3680043.8.498.11888706160097250105897088149797666413","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54482848766921329667469114743547203929","series":"1.2.826.0.1.3680043.8.498.16899123316756782260037079782036205955","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54101830029742852876183926998159899424","series":"1.2.826.0.1.3680043.8.498.81705913316809115103256314755856652611","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54662745951809363774369202018332309887","series":"1.2.826.0.1.3680043.8.498.10479250996018507035661694922366154044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54689086194769265957947122616025330642","series":"1.2.826.0.1.3680043.8.498.27049325265859195570787268198743211281","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54699196872730350880681018674893216245","series":"1.2.826.0.1.3680043.8.498.85893952380944272627508776194716336520","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54755249425443469939742671497042113665","series":"1.2.826.0.1.3680043.8.498.10460750967890626240281590186349431211","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54539703415639765348239790227512479583","series":"1.2.826.0.1.3680043.8.498.86413821224936741428149115251333046119","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54619696938940618755334224550745735815","series":"1.2.826.0.1.3680043.8.498.11708370949774373476519107193591427712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54792838438079861454532183998999249213","series":"1.2.826.0.1.3680043.8.498.53782289927457682890211035001030975281","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55008913229055942530957507434366806883","series":"1.2.826.0.1.3680043.8.498.98610684208637789220593581007706033851","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55022631713888448062201243978634363406","series":"1.2.826.0.1.3680043.8.498.68390584919630341246382655832014279440","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55065105486648830377200570585789726800","series":"1.2.826.0.1.3680043.8.498.24603382767187312184777903055916095392","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55202459816190607876303419067810637245","series":"1.2.826.0.1.3680043.8.498.98076695332199576629949971998323339809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55105303802677863700162285680213496441","series":"1.2.826.0.1.3680043.8.498.12294777983211804487691656639447315572","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54866613282412053616997257524944654864","series":"1.2.826.0.1.3680043.8.498.89114959842603719507913617062220976584","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55332521421041708438748125666498042763","series":"1.2.826.0.1.3680043.8.498.12026086828420700139629793624876033332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55397602116058381866334455456729470963","series":"1.2.826.0.1.3680043.8.498.13748681839649771437058470808590397034","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55344353927048912237233441917779875587","series":"1.2.826.0.1.3680043.8.498.61145034196151112324670459796775330598","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55687452977471862155042703116192295932","series":"1.2.826.0.1.3680043.8.498.69970939909242676583180874885149695875","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55713753582074871470943388423906540792","series":"1.2.826.0.1.3680043.8.498.76792245602347626714433416064275398934","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55461938012492393262172182405932212246","series":"1.2.826.0.1.3680043.8.498.48114332935259415387216870376142715428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55586358833903787781826406120068237870","series":"1.2.826.0.1.3680043.8.498.35869994756981268206144609366370621803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55771181380353653474896884649540028856","series":"1.2.826.0.1.3680043.8.498.20745879292493257570524317788044767372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55850385284038070407841702672827533728","series":"1.2.826.0.1.3680043.8.498.32761503151252955560211310295181588965","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55971152627433130850649668964887611410","series":"1.2.826.0.1.3680043.8.498.26007583381951913382324640244418339569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55927741980493661714408715886641034285","series":"1.2.826.0.1.3680043.8.498.91017183900480405698045978311425774622","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55991293001848799160041967098772087659","series":"1.2.826.0.1.3680043.8.498.35101531992558756682156916285996043350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56061505201159269536816810086342635391","series":"1.2.826.0.1.3680043.8.498.32961253014558149634475770125613235212","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56045687004558569005240775213510192805","series":"1.2.826.0.1.3680043.8.498.20301629736893509311089167842862415972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56146582420561901778012523946835033628","series":"1.2.826.0.1.3680043.8.498.57860423285952094209241463687946583492","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56170570792418244380186761574367557535","series":"1.2.826.0.1.3680043.8.498.58993443376892706842582527942491320544","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55861544093014499478739864837675161978","series":"1.2.826.0.1.3680043.8.498.79056741344306722263101991941557723481","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56474364950531483257947155855279443851","series":"1.2.826.0.1.3680043.8.498.10201315045472865274359404567701425059","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56385473079196655120993854770623075553","series":"1.2.826.0.1.3680043.8.498.11611380071666893601563322094429653848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56269695574661037525483516531194514456","series":"1.2.826.0.1.3680043.8.498.23532154834739061161839143937140291276","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56216873861595017403772627709751840588","series":"1.2.826.0.1.3680043.8.498.13686533335945342736950568283496974266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56513276339820682030546599321915304265","series":"1.2.826.0.1.3680043.8.498.11378430477818803105723310530334159775","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56686055146646210144964207684044815006","series":"1.2.826.0.1.3680043.8.498.26406603940966961850060023141520095717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56751453017492477106384704540803963879","series":"1.2.826.0.1.3680043.8.498.50479554017979142568333956632004212317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56506415277488978301157490514745400410","series":"1.2.826.0.1.3680043.8.498.74263210512629823470200700151567153399","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56786556618312090924315296993756843070","series":"1.2.826.0.1.3680043.8.498.49676844442758217441794514786654499115","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56798070549054124854685511360777559033","series":"1.2.826.0.1.3680043.8.498.50143747924391806772136799943283820319","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56575942633137496743464648679775042979","series":"1.2.826.0.1.3680043.8.498.26503815104360800716291148508955030635","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56839374698050163738039760889156066445","series":"1.2.826.0.1.3680043.8.498.86737986652358173105378075127122665109","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56939860329243258146102948466697018543","series":"1.2.826.0.1.3680043.8.498.73109770830119434650749028595624835233","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56894990686683651282564584582335346350","series":"1.2.826.0.1.3680043.8.498.93090265109038916688426081577619178759","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56993108275828399240508695615212151102","series":"1.2.826.0.1.3680043.8.498.24981336315328645089927905067337895557","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57064618560116981041722203028857007300","series":"1.2.826.0.1.3680043.8.498.13377701463417985398191636717670014325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56989113744241846544086409786820246754","series":"1.2.826.0.1.3680043.8.498.45280926821295529845245434579291689799","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56882015310689536014061315110969323429","series":"1.2.826.0.1.3680043.8.498.44128429354954482613507041984206303477","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57084445429567494093960261980941917499","series":"1.2.826.0.1.3680043.8.498.95833655757099904116388739000851388346","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57316522861709598765962486949846744486","series":"1.2.826.0.1.3680043.8.498.94116341866881606394553776570106411945","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57348281565196196401055821640494904475","series":"1.2.826.0.1.3680043.8.498.21923465625371278001852985143049688041","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57491375928585600499323136123872536597","series":"1.2.826.0.1.3680043.8.498.24361189682595621485649381877074516653","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57615268093937994500995086275007694564","series":"1.2.826.0.1.3680043.8.498.26882092756480966836711727868210745164","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57768539239475064336346600805236336044","series":"1.2.826.0.1.3680043.8.498.95060502356655803972080421786454211739","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57789603189240866041812385320399435864","series":"1.2.826.0.1.3680043.8.498.21334161432591836671614912014209776355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57811199787891897795884073594875380873","series":"1.2.826.0.1.3680043.8.498.10789368838532827352087886587417883995","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57160448024145680660335079050450776852","series":"1.2.826.0.1.3680043.8.498.10965728973823842729329728143245853060","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57859543314667538612269642333387175763","series":"1.2.826.0.1.3680043.8.498.35551111035013694638274142009895572997","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57917634151851433037090858498504057826","series":"1.2.826.0.1.3680043.8.498.17560593655813087967593361309485431904","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57960501042895683580303516749157258348","series":"1.2.826.0.1.3680043.8.498.68842580432922220973928646025565997716","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57836524675365700674244122764523161699","series":"1.2.826.0.1.3680043.8.498.12243302333248604116573935512731637540","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58250779958866595505585700517927225230","series":"1.2.826.0.1.3680043.8.498.11215038533992311378784326533352116232","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58143227319316521737029358930629086887","series":"1.2.826.0.1.3680043.8.498.13254590511368053923505203806464809205","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58280111882249056715877316026191844548","series":"1.2.826.0.1.3680043.8.498.62993034304453904604924573677330911785","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58298767414933858487578597918936525600","series":"1.2.826.0.1.3680043.8.498.11946206612867658626828435340152101452","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58366682083795592678613986311570432419","series":"1.2.826.0.1.3680043.8.498.51407765261734109803107246259349234179","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58475208045272914292874190001248518687","series":"1.2.826.0.1.3680043.8.498.45352378446514244026997960769353257788","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58482779112389764165599701941310919660","series":"1.2.826.0.1.3680043.8.498.57811601768163097267680186536018093748","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58404766256608483217059706069436050741","series":"1.2.826.0.1.3680043.8.498.76632328607370511017971106164062213622","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58550899000434548218916895907938175917","series":"1.2.826.0.1.3680043.8.498.58908325074851361276367209960983268047","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58685941646495891824559391997783302656","series":"1.2.826.0.1.3680043.8.498.46828273462983774432213928747616250692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58728013757221328590898166513995249334","series":"1.2.826.0.1.3680043.8.498.15266936718114844854269085675099770015","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58634494571833713296574414948455471101","series":"1.2.826.0.1.3680043.8.498.41572322207263987710164039534834136009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58604600600114899516493668210902251434","series":"1.2.826.0.1.3680043.8.498.30832971802893214065005563825537473306","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58749835013162544157291112101941849646","series":"1.2.826.0.1.3680043.8.498.40341845907912808638974490704325283043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58776844157027040145262759105410792687","series":"1.2.826.0.1.3680043.8.498.77999189350117397330034453063000661698","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58888406752590566453001353089284177456","series":"1.2.826.0.1.3680043.8.498.15479079348221458031709585524202508142","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58835863392949696556026907211309159182","series":"1.2.826.0.1.3680043.8.498.48887764538068524196133512289684069217","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58965370475364247222892582428665460530","series":"1.2.826.0.1.3680043.8.498.37888308032129682077291982644115157349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59047534248122071167153110802170658177","series":"1.2.826.0.1.3680043.8.498.57956649314340183532737577282032222705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59073122184069676919857712290159511305","series":"1.2.826.0.1.3680043.8.498.28109800190578422428891621550315961261","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59254379036462832374262221063106167294","series":"1.2.826.0.1.3680043.8.498.79610553512157262730804158555226545649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59126324546257448818397467078224944768","series":"1.2.826.0.1.3680043.8.498.13017867667165553396007947401145482044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59420153636832938628088507776889305823","series":"1.2.826.0.1.3680043.8.498.68030106922047213116121905498528365257","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59309239681970150083311665704264518844","series":"1.2.826.0.1.3680043.8.498.62216716655496311194584067055159855724","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59449702487184611096113620550399616423","series":"1.2.826.0.1.3680043.8.498.48032426515418196953907042356021491364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59511171909586924989766685687032462369","series":"1.2.826.0.1.3680043.8.498.63668821436547068997041603451748634417","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59680220055968160813801840287391339651","series":"1.2.826.0.1.3680043.8.498.73944641199331487153960010681768736672","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59476657510212258499047356518058209126","series":"1.2.826.0.1.3680043.8.498.44295136673260745222663041783084644605","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59533384841491000088839469462294817273","series":"1.2.826.0.1.3680043.8.498.21807466302426369408412242062422132517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59758232061190668535098343218115599228","series":"1.2.826.0.1.3680043.8.498.10422902385610242288188587675578950158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59828592894392286103154555572376381359","series":"1.2.826.0.1.3680043.8.498.36803858689170976119414140517446821335","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59932425615255727459402684084295081504","series":"1.2.826.0.1.3680043.8.498.64485586160736929498373406137918588666","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59939977812549570931951749553467050226","series":"1.2.826.0.1.3680043.8.498.14391911196181091117953811386967857735","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60064080350033586611530748685370040101","series":"1.2.826.0.1.3680043.8.498.11792698483374486361887296963224032133","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59864697179957009967756900038846073439","series":"1.2.826.0.1.3680043.8.498.11694048493199836318512873941399815316","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60094182309575937331152278560900767484","series":"1.2.826.0.1.3680043.8.498.48041494380797778614949854362512909815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59954207362160100069708115719895503197","series":"1.2.826.0.1.3680043.8.498.10062461076000933362424628266183945811","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60339775486026531260673620094004244044","series":"1.2.826.0.1.3680043.8.498.12422709895393720912365220719469647266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60240929196665639608259748424475337443","series":"1.2.826.0.1.3680043.8.498.24447589533454430351399731611224717337","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60435128564448668334758790827765960420","series":"1.2.826.0.1.3680043.8.498.93404403371491291777323011162839233565","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60394858470917657607846893765818799720","series":"1.2.826.0.1.3680043.8.498.18858834741538714886853289900785219432","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60189166269847256202786309096136242746","series":"1.2.826.0.1.3680043.8.498.93380653077299808265826603294820973084","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60446085094710623303973373214794179184","series":"1.2.826.0.1.3680043.8.498.12150133794865325756617147261652643750","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60473167624534682552272469123806503959","series":"1.2.826.0.1.3680043.8.498.70362335324085859456988139860894374180","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60492772672040498730937577391401159109","series":"1.2.826.0.1.3680043.8.498.15745254194521430617835664195446498999","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60614975796208617794180432816390721331","series":"1.2.826.0.1.3680043.8.498.89571444230437089902602667810354520241","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60736996513214411991331917490659995815","series":"1.2.826.0.1.3680043.8.498.12213215582407804179798415030232348858","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60582762894748063612925320003677861384","series":"1.2.826.0.1.3680043.8.498.36482269933485664656396950795771610779","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60661676782543347731000874708337132276","series":"1.2.826.0.1.3680043.8.498.79440727097090710561312035866280336824","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60924385750386872880015870524131415592","series":"1.2.826.0.1.3680043.8.498.11779996984418741531394915038014816826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61043690805522768840878574610198420558","series":"1.2.826.0.1.3680043.8.498.45011939244154780930592418743327137113","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60965286164497194740320124904525623873","series":"1.2.826.0.1.3680043.8.498.46646668038861367920478478967478379416","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61148667330113929729568621580194520169","series":"1.2.826.0.1.3680043.8.498.73134083170249017428901714517783505077","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61313243050348988917987796330691448770","series":"1.2.826.0.1.3680043.8.498.12449651313152665198030327940351019024","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61227644940421894951950605627016308160","series":"1.2.826.0.1.3680043.8.498.11855225305003373570139538149523199206","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61140619849833544411463750382539885141","series":"1.2.826.0.1.3680043.8.498.12692471264687564175358312216352843340","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61354742313056287432707914716598361679","series":"1.2.826.0.1.3680043.8.498.81091982680737283075534875473435688837","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61456481310419978302697858056095086811","series":"1.2.826.0.1.3680043.8.498.73216875745600090396583622788253264039","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61518377417979517706237284901970943470","series":"1.2.826.0.1.3680043.8.498.26607829166183712241249013494031931067","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61554384977916135091630433680780524386","series":"1.2.826.0.1.3680043.8.498.32377952812776135942021080232010442744","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61715630583919056026446344483871058799","series":"1.2.826.0.1.3680043.8.498.11531355268162358798591343963628948954","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61956863718716027369068406125965902569","series":"1.2.826.0.1.3680043.8.498.12478071169947807265248987898567947665","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61812343111344551704536863711676179812","series":"1.2.826.0.1.3680043.8.498.11796023675360144113342156315288266068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61904856234837992279531381374350177464","series":"1.2.826.0.1.3680043.8.498.97227983694932612085666776733447825783","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61581513083114643407028382481113097015","series":"1.2.826.0.1.3680043.8.498.10323156271931360840362270736561928157","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62025036385410820041698409408092924893","series":"1.2.826.0.1.3680043.8.498.72015515817970790437360033062987259774","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62192347880785177732202565414591408086","series":"1.2.826.0.1.3680043.8.498.26037824433265788283196386462146627537","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62109244272539959226824769201542205950","series":"1.2.826.0.1.3680043.8.498.49358655354820452142772024536324743425","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62251470468640645490157592046417355534","series":"1.2.826.0.1.3680043.8.498.51270451167641406073939343777167797229","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62406588098851994968957074186291720431","series":"1.2.826.0.1.3680043.8.498.60213942949826616025874340178256499349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62217447760619326492634156203608889396","series":"1.2.826.0.1.3680043.8.498.10415750892381709193896824650854791466","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62329283530383340787096585492217632179","series":"1.2.826.0.1.3680043.8.498.41839050988006475172075754307922525719","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62437671010172828772205139640126159272","series":"1.2.826.0.1.3680043.8.498.81174073741195925874373291151789643073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62400371770962739002393364782172386752","series":"1.2.826.0.1.3680043.8.498.10820091427350505719536158686351199630","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236","series":"1.2.826.0.1.3680043.8.498.81892732695419468989322862582884021558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62693151030819598514729637644804213983","series":"1.2.826.0.1.3680043.8.498.13028465092536707248894452561135886108","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62760659205261492447481830774781950875","series":"1.2.826.0.1.3680043.8.498.18081296927832776681748716952060200145","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62782163204193388751869649423542931657","series":"1.2.826.0.1.3680043.8.498.46537388809053163848321198186663856191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62928513795250694194589804784951257414","series":"1.2.826.0.1.3680043.8.498.58841061018874545455578167670723747217","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62983291359021407345388012674614537807","series":"1.2.826.0.1.3680043.8.498.10844087348354702494275597863077836876","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62918046703989259691005447883746340916","series":"1.2.826.0.1.3680043.8.498.11146853011455769002225072436737910628","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63069333313670617776227577192591993067","series":"1.2.826.0.1.3680043.8.498.10458808895697308503844112029725510210","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63105151375349194807502162946823966410","series":"1.2.826.0.1.3680043.8.498.68377837328795484211635001095304804162","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63005973438879239920248535292375481294","series":"1.2.826.0.1.3680043.8.498.81468967086770846320385553454453992604","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63151733210715530368323422018984705316","series":"1.2.826.0.1.3680043.8.498.16430620340935153791906404647159397908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63197503121546251937003511751049515360","series":"1.2.826.0.1.3680043.8.498.11860675779485900003061510778531494449","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63216944317982327305447295456609309952","series":"1.2.826.0.1.3680043.8.498.11656811838961437334533853863632271444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63229263722659785142967850542699676113","series":"1.2.826.0.1.3680043.8.498.10666011271293246197060517870905822367","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63377901538320536934403969965624202519","series":"1.2.826.0.1.3680043.8.498.78069691998995103433855879057128617147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63279205896049798083339413931561428540","series":"1.2.826.0.1.3680043.8.498.90097682845842928283332514828725011507","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63320397279666280420359587173167195895","series":"1.2.826.0.1.3680043.8.498.31317792374706468755947510841492326936","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63426442500798189991842988966227275166","series":"1.2.826.0.1.3680043.8.498.54435231701662767414172341059620987988","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63539809431911516912857504094611479480","series":"1.2.826.0.1.3680043.8.498.16753318058331431959100812015720085446","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63521033783841675104013677430088400343","series":"1.2.826.0.1.3680043.8.498.10983865213128582255170896597469916258","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63565089119380386830816381560603756258","series":"1.2.826.0.1.3680043.8.498.87569874613469191329718338899389367069","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63510378689078892579575716369639643386","series":"1.2.826.0.1.3680043.8.498.13246860191832543155011251377349384329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63556690547679257097888724096967682731","series":"1.2.826.0.1.3680043.8.498.87615717504747758787305706509827425412","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63849051301265505051181245915656194652","series":"1.2.826.0.1.3680043.8.498.38639091816096582739777967208449817760","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63894147741855851067411708837856142566","series":"1.2.826.0.1.3680043.8.498.77628058722119927425041626229361842698","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63673733836218886397484265807806548473","series":"1.2.826.0.1.3680043.8.498.12217735465436996918706697199149277619","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63726512375511246975989322174096818969","series":"1.2.826.0.1.3680043.8.498.24830026619245989120192410378102606807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64029768638431636860053484202080135316","series":"1.2.826.0.1.3680043.8.498.40763069999351114310749467550473737078","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64066465191710322298540005681534170036","series":"1.2.826.0.1.3680043.8.498.50355091183758585249254987579720349131","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64022112049842890696609934970222024490","series":"1.2.826.0.1.3680043.8.498.33182328540963800244465748903730501699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64119798231341423127851203797783576453","series":"1.2.826.0.1.3680043.8.498.13384458520869252098551861492013033867","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64186424577324542519674308830472181373","series":"1.2.826.0.1.3680043.8.498.12812562495853645000092843097965322581","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64283618749539713756220539395944025878","series":"1.2.826.0.1.3680043.8.498.83940835335605839007009645522186343158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64325734401816269429960464734504322970","series":"1.2.826.0.1.3680043.8.498.10664393847034029110537303841908286361","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64397747170052970974000650185716672477","series":"1.2.826.0.1.3680043.8.498.68919540392773289350667863799332365026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64443149121726771900576109439951701105","series":"1.2.826.0.1.3680043.8.498.13357026917304822103046333628335055127","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64408775259838960958354835366765941676","series":"1.2.826.0.1.3680043.8.498.11386792283600257049020166426940918699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64590392444259942263746282444188079724","series":"1.2.826.0.1.3680043.8.498.97176687980761205328653220930966315105","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64663285742626599570990862360902141188","series":"1.2.826.0.1.3680043.8.498.11810869323466957201411917434499364553","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64865300853258390816475799272221318005","series":"1.2.826.0.1.3680043.8.498.42994676746980871405084648595024651589","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64640435232894033317418319985727526223","series":"1.2.826.0.1.3680043.8.498.78066312822217701362909244324030093444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64526757783166918696475501252110447010","series":"1.2.826.0.1.3680043.8.498.11132651848598435765910491586608293225","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64950600385134699009390197520158979183","series":"1.2.826.0.1.3680043.8.498.47691155533480590944043770858592256561","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039","series":"1.2.826.0.1.3680043.8.498.11346458840980855602023845654340426652","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65077085081642080527129904737510276643","series":"1.2.826.0.1.3680043.8.498.12696627953245672082762230475561013788","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65123055741109319778634264103224173252","series":"1.2.826.0.1.3680043.8.498.29780894991873964576021375327681120998","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65160241567146620406193593170843451582","series":"1.2.826.0.1.3680043.8.498.10257125549551966981088322442531946982","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64917073507426524387269500871970032783","series":"1.2.826.0.1.3680043.8.498.71633769430706266279152741358388093384","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65322445533737673246312151513913460816","series":"1.2.826.0.1.3680043.8.498.55326519435926380235406538380353327407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65222333228514862688331581862087607325","series":"1.2.826.0.1.3680043.8.498.88388621331874734728400417717767727691","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65403984120528711726317048984986599537","series":"1.2.826.0.1.3680043.8.498.27555151041345516324135247398005699827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65547889329285432771702699287910809921","series":"1.2.826.0.1.3680043.8.498.34968892665783639573794420961598769062","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65244223225926362862970496735749098549","series":"1.2.826.0.1.3680043.8.498.82015422571326947421905515983932142932","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65515828911930515002046151253057175178","series":"1.2.826.0.1.3680043.8.498.12481000213342705174054770655578150316","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65607807464576681752987746458209389847","series":"1.2.826.0.1.3680043.8.498.27416101867555840463268928746427308428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65703265163052901613211590083675186626","series":"1.2.826.0.1.3680043.8.498.48482618291259542087011676396650838981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65530757406333318449497070048719890442","series":"1.2.826.0.1.3680043.8.498.74581647698111747364870962351435366103","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65715827585629489888518528736848969678","series":"1.2.826.0.1.3680043.8.498.69503193173086613330084336952508471362","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65763261177572072944828171262268039852","series":"1.2.826.0.1.3680043.8.498.87341450927401221245596755001127297607","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65809115193576546782858254430325338674","series":"1.2.826.0.1.3680043.8.498.56829627798653213410775896917882242068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65855247249608663104822603172047250057","series":"1.2.826.0.1.3680043.8.498.36287173784404727437618124509892547116","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65959182350599581687336889990076028593","series":"1.2.826.0.1.3680043.8.498.13228507435970096440464076629094416469","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66020213859389864645118554257523478921","series":"1.2.826.0.1.3680043.8.498.51658095084904833435596253880736906075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66031142975529812579758652269360365398","series":"1.2.826.0.1.3680043.8.498.14255000984769027187399566320730094166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65788206282522750507559682036750692505","series":"1.2.826.0.1.3680043.8.498.11090102870867460571881314219419785698","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66077420344931334696112616504179985591","series":"1.2.826.0.1.3680043.8.498.17481513835521392166442773678194277449","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66156015559325316834458423557757995415","series":"1.2.826.0.1.3680043.8.498.88591760755279570285392804224720257459","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66287049728466409037095442705228040084","series":"1.2.826.0.1.3680043.8.498.40538931074467395277510485970681736849","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66305771884507759587182077473863996177","series":"1.2.826.0.1.3680043.8.498.38315244217262258380294979870112927043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66210455934011506415244876770725282769","series":"1.2.826.0.1.3680043.8.498.12186284138382232794801177790233787165","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66334686028272068369345540785320291695","series":"1.2.826.0.1.3680043.8.498.83296331816405063154958678858867197758","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65909865035783831607773206653044421345","series":"1.2.826.0.1.3680043.8.498.28949963231070038078221580881570858343","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66321903250295261063309947910189882952","series":"1.2.826.0.1.3680043.8.498.13405930396603581664999447975066871037","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66401771232064170830621215861092532421","series":"1.2.826.0.1.3680043.8.498.11567570853102572218561203057453876327","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66489634638847834550694493585147280754","series":"1.2.826.0.1.3680043.8.498.11000326800806757682835283015118458314","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66596090141798159058634289358454183351","series":"1.2.826.0.1.3680043.8.498.48427518176664000264788648397666668523","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66690100241274246406145618641481787709","series":"1.2.826.0.1.3680043.8.498.12399926635427829954930055301271220175","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66737859593831436259089736126768464771","series":"1.2.826.0.1.3680043.8.498.56959561152557854973528935905694977045","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66993076561387452114335652021593805727","series":"1.2.826.0.1.3680043.8.498.72625006746207267234868428065517380344","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67153684605171746899496816770567866269","series":"1.2.826.0.1.3680043.8.498.11782024248868438037658951379160830622","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66841103065389483583108279900721522084","series":"1.2.826.0.1.3680043.8.498.57211812527316038983250261421623423769","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67189969532330535369638450306601659967","series":"1.2.826.0.1.3680043.8.498.30984200874962348797517899619151379673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67238774585857531338533947812568490258","series":"1.2.826.0.1.3680043.8.498.17655080056113700700710333302799713806","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66960891947495601371173426277430301127","series":"1.2.826.0.1.3680043.8.498.92851410459352681137187451033126265691","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67181088492292316990397177612918195713","series":"1.2.826.0.1.3680043.8.498.96655386006846869692900577526010356100","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67284681623851829916403311739987186159","series":"1.2.826.0.1.3680043.8.498.15423333056365846753634300181023910407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67404282038417089622852480023919627784","series":"1.2.826.0.1.3680043.8.498.28688257763144870121484968046787572114","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67386780309737226410965627751074124678","series":"1.2.826.0.1.3680043.8.498.39299993324033442890190617209469179112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67468397709097947104298147948850473142","series":"1.2.826.0.1.3680043.8.498.13131404906732628102839591109680642861","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67498362978754615554767658151787188674","series":"1.2.826.0.1.3680043.8.498.10434299174621715523658137213176582947","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67613388743126519966672593774754849330","series":"1.2.826.0.1.3680043.8.498.11889899538063904409669021253191915777","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67550262192397377344258276170992188234","series":"1.2.826.0.1.3680043.8.498.12537044509644757605595003497605900524","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67678576373934022409084801948795250361","series":"1.2.826.0.1.3680043.8.498.38703689805709163434065392515322660641","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67705068860320494485379949506807098172","series":"1.2.826.0.1.3680043.8.498.20532746716529135150810824071564289917","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67772676243889350087068112324015156848","series":"1.2.826.0.1.3680043.8.498.82075329312902559140507667379797678675","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67739091465243228820766982459725849814","series":"1.2.826.0.1.3680043.8.498.42713517202564814820019558942119140158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67860057199678546706175336855949799228","series":"1.2.826.0.1.3680043.8.498.95858235087208472983811986445765755363","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67788047649994083591931245362169694306","series":"1.2.826.0.1.3680043.8.498.53669264156952182878699812022686951163","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67901792488494108577057563458320460313","series":"1.2.826.0.1.3680043.8.498.13311901221572052140857223559263741465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67931702364673809781455566584550075686","series":"1.2.826.0.1.3680043.8.498.10007033060853101402914165412426592663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67958551704692248657133951054036903879","series":"1.2.826.0.1.3680043.8.498.27566092550582715220100986187512089331","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68079818232244983808025467536454890669","series":"1.2.826.0.1.3680043.8.498.11309803557147807579898350640387104929","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68106879690636836695681159513106553666","series":"1.2.826.0.1.3680043.8.498.34332324815499947514915273636657358733","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68158735450883179427440578190038002606","series":"1.2.826.0.1.3680043.8.498.13666733115662526243643968035475425416","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68131564130881160284492534203572971623","series":"1.2.826.0.1.3680043.8.498.12054792138620857474530849992289942198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68168197665046786209369894696801397179","series":"1.2.826.0.1.3680043.8.498.58446813329545488154455749742123242567","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68219942161081583831276769978472014317","series":"1.2.826.0.1.3680043.8.498.31449246226241939401705364671546291899","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68256141451764547710152696919981893952","series":"1.2.826.0.1.3680043.8.498.67155541936366969753081122157648454527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68264559730756309944466784364591509088","series":"1.2.826.0.1.3680043.8.498.13700537862993732876991435540356166382","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68328371080087298795966933114044350359","series":"1.2.826.0.1.3680043.8.498.92405892450464181435823282912945269838","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67981539513814867568631217960473023750","series":"1.2.826.0.1.3680043.8.498.83130986690483914017925862088783851579","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68434116759606759586218589635360502038","series":"1.2.826.0.1.3680043.8.498.16676263877661139413657746558499059461","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68576465611266620513305434082656807028","series":"1.2.826.0.1.3680043.8.498.14592434379858044502275797908573855488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68435927323644457286635872824563629385","series":"1.2.826.0.1.3680043.8.498.57395278214033364260511828049134372978","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68524920268469128133836299591793117756","series":"1.2.826.0.1.3680043.8.498.10386320144425632854985307489659574330","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68593524008570334427931338225473487643","series":"1.2.826.0.1.3680043.8.498.15359841889396539818549307849899008631","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68448811873610037553554213141537599150","series":"1.2.826.0.1.3680043.8.498.11561785794841837629390838659079485786","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68740466085414970518655922465908223436","series":"1.2.826.0.1.3680043.8.498.30596741791710556139477879259185063794","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68799355386870083063665227055340582612","series":"1.2.826.0.1.3680043.8.498.10572319209415949255234969765698294433","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68769474672202195228430674317444754835","series":"1.2.826.0.1.3680043.8.498.45359771479710709727301784738020420372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157","series":"1.2.826.0.1.3680043.8.498.10203353291428924408099807238468662354","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68668620295929607459355154952035421306","series":"1.2.826.0.1.3680043.8.498.91684029854315456129802836159002365779","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68931510756844060683643298583132884960","series":"1.2.826.0.1.3680043.8.498.13300483737661434464502084777711642093","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69156800026549034619316278954442147703","series":"1.2.826.0.1.3680043.8.498.44357996484223408424632186229569019769","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68971603226593154050514800137550171537","series":"1.2.826.0.1.3680043.8.498.63390024700729293956624486251817556804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69131433601281694165384475765166382765","series":"1.2.826.0.1.3680043.8.498.74319258473430289826703848450377515923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69236835611860340913671698884308946455","series":"1.2.826.0.1.3680043.8.498.12601349571192885112616803874110138652","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69267821440055811776827497076481587198","series":"1.2.826.0.1.3680043.8.498.12790264401692992085871757140455328710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69411539430142524612470544041263824359","series":"1.2.826.0.1.3680043.8.498.10641790822990412415774346494981677158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69429384137362531712760113705595820981","series":"1.2.826.0.1.3680043.8.498.58608663803055102707154472967119881271","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002","series":"1.2.826.0.1.3680043.8.498.79225807496183598139611165476901586908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69495014653333523580255444915082056825","series":"1.2.826.0.1.3680043.8.498.50096905271719452157530373405688077185","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69033651423155214548081618535669641812","series":"1.2.826.0.1.3680043.8.498.83708632885169417334157793180609088181","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69544085715595825083308172629360899339","series":"1.2.826.0.1.3680043.8.498.98616158961937147677194836650448187687","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69729903409900053570304330127687235813","series":"1.2.826.0.1.3680043.8.498.12529259750687501204315422143619660396","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69654898267139720912940475772303404772","series":"1.2.826.0.1.3680043.8.498.11722337243519359188271529022245204447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69632034454628807663252084651278084839","series":"1.2.826.0.1.3680043.8.498.98091908979238374466575522794375484942","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69755849784902271124335765927125722483","series":"1.2.826.0.1.3680043.8.498.89721658251537521534613858133914953954","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69846596572083911408018130524077120494","series":"1.2.826.0.1.3680043.8.498.10517324299134749383898739890805847981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69673283369917182409597968479300552579","series":"1.2.826.0.1.3680043.8.498.76708918360436892899346996346953494310","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69930569170269472474738549886709596023","series":"1.2.826.0.1.3680043.8.498.57215670838574526302120627442201120063","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70291739279544255146458190540233023337","series":"1.2.826.0.1.3680043.8.498.76235348580562598791926309137195697094","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70129413051088864999976563404818211137","series":"1.2.826.0.1.3680043.8.498.11212439766926866438626116423103089447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70212141059019236877414201247791694101","series":"1.2.826.0.1.3680043.8.498.25832639803733962355582245006542717884","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70124572708965486791612654433965515063","series":"1.2.826.0.1.3680043.8.498.65836375281148112535554945310071545145","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70375834273924102402706635540898552043","series":"1.2.826.0.1.3680043.8.498.97646367735711330982976230124941030314","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70310491987092072540988486213408328865","series":"1.2.826.0.1.3680043.8.498.12714386553342957069274785364046430266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70427947434642957729711329062828233047","series":"1.2.826.0.1.3680043.8.498.79143413803594135274752026879964441486","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70464717849928808075676221329034929056","series":"1.2.826.0.1.3680043.8.498.10865332009625927915428143800717235186","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70531311408047627903498280968772932092","series":"1.2.826.0.1.3680043.8.498.98414595323975784127069228468147724877","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70478068758081824347762712452252763894","series":"1.2.826.0.1.3680043.8.498.10164420145174100771789335128134336859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70609649448269244608860767754947143395","series":"1.2.826.0.1.3680043.8.498.83682396695752197243631832050920001582","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70873330457307886117949106432465978541","series":"1.2.826.0.1.3680043.8.498.11861176148907478147591964017998776610","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70831735578941318362881379802693293952","series":"1.2.826.0.1.3680043.8.498.52762354838848410441802175831955057536","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70691305452244378255646875206864340458","series":"1.2.826.0.1.3680043.8.498.43077138775719195355134064277352537333","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70931245733460964524244528230239226104","series":"1.2.826.0.1.3680043.8.498.12980945439042383133602083549164765029","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70913971933753517251340687745718346828","series":"1.2.826.0.1.3680043.8.498.10311693038864023859428477778277983540","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70948162965271404678418299669321564765","series":"1.2.826.0.1.3680043.8.498.48877756795986091604050800666699136570","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70796360440320543674384511859375295769","series":"1.2.826.0.1.3680043.8.498.86695390227239610003259591355900499901","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70966951020892708303866968206211604316","series":"1.2.826.0.1.3680043.8.498.12326496107887723587954327737654619408","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71063550263564056105844586167361750567","series":"1.2.826.0.1.3680043.8.498.70148514493589891032048586019990535064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71001339006150287453095055802545712130","series":"1.2.826.0.1.3680043.8.498.97304422562269897581968210898759028240","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71157888664602228768823145752952837565","series":"1.2.826.0.1.3680043.8.498.18840237195477521315952133725659991304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70880546533848103776823598726224805522","series":"1.2.826.0.1.3680043.8.498.10416648697509247566721762644203814620","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71262022207903748073532915013888861999","series":"1.2.826.0.1.3680043.8.498.73198426841800128878232049460564638109","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71355457409240695394093233905149541215","series":"1.2.826.0.1.3680043.8.498.87328590202657546136627154504879117511","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71622107620007706778976946784787365505","series":"1.2.826.0.1.3680043.8.498.19121917954307830798561661440731446444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71405492944866967669069984757332849184","series":"1.2.826.0.1.3680043.8.498.13278393236845377083646776622001976892","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71686717249431292334159336816742076478","series":"1.2.826.0.1.3680043.8.498.12535635941566437105949137018817750977","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71660070729774772658965427776586988585","series":"1.2.826.0.1.3680043.8.498.52816907244996120452188461997381295554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71709477515434076105531625950744135656","series":"1.2.826.0.1.3680043.8.498.13038976962248509093750392551875288577","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71753603955749189961638324806586359888","series":"1.2.826.0.1.3680043.8.498.10384075452197857357867761484406056527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71744523263023933066263506201344989825","series":"1.2.826.0.1.3680043.8.498.60346834278185967925257224018881381235","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71830371861099586573789550797712053914","series":"1.2.826.0.1.3680043.8.498.56258513288925128604003943112895213177","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71866327348775269043047475658220220525","series":"1.2.826.0.1.3680043.8.498.10860398989733582906550424233649729621","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71792468450491064417953341293663396190","series":"1.2.826.0.1.3680043.8.498.67460400431031664504392902773108896239","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72017096768214637927798648334301077145","series":"1.2.826.0.1.3680043.8.498.35679368128328302904752944261821888762","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72023797276837219152365428999317131110","series":"1.2.826.0.1.3680043.8.498.25973557273702253296238651273228484151","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71899370657875960028369229847153071577","series":"1.2.826.0.1.3680043.8.498.67262755438159843913745096661039095130","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71701484984624141610445774249629295610","series":"1.2.826.0.1.3680043.8.498.70770025630371983252935997483325257865","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72159028826938094045309841541333724597","series":"1.2.826.0.1.3680043.8.498.80919285853950399511510156487005803922","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71985417286226511458068110832784023627","series":"1.2.826.0.1.3680043.8.498.11662442678563405760122392692477318963","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72191353854496806844040272237700346774","series":"1.2.826.0.1.3680043.8.498.10445332488427565850266729803028989909","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72276631472244076773626607623783737687","series":"1.2.826.0.1.3680043.8.498.13334696318860076860244537719127952730","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72342425990274562150756501476989691572","series":"1.2.826.0.1.3680043.8.498.75517303748731445791260989633300329985","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72375103774239787721371778221019402834","series":"1.2.826.0.1.3680043.8.498.12482877642970985094139686846833486447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72182847112910012489603688436912469641","series":"1.2.826.0.1.3680043.8.498.99903185831825418113284047949375040183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72248728319398595146365363062799556637","series":"1.2.826.0.1.3680043.8.498.10185012997837128985732936735230480723","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72255181548595299889516400134053439401","series":"1.2.826.0.1.3680043.8.498.56314381170020160412533890160599592397","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72495590491484814220060275823761767516","series":"1.2.826.0.1.3680043.8.498.19530208440713101948880088825267824749","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72680485820462987585370556994612687540","series":"1.2.826.0.1.3680043.8.498.18150660771917756191498569315685995586","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72547312490317214326791189007411860203","series":"1.2.826.0.1.3680043.8.498.31400126119479025571974862490159949849","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72453426317225885428514595399496580896","series":"1.2.826.0.1.3680043.8.498.94742480575082698523805531710221925000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72740499664060575780267947543695605585","series":"1.2.826.0.1.3680043.8.498.10701488279248722839534322968647675542","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921","series":"1.2.826.0.1.3680043.8.498.41786504123190759621834893248893641957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72800659596099468282135738543971261904","series":"1.2.826.0.1.3680043.8.498.67284563393718563595511069556993280326","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72781608662300877077691853670862414309","series":"1.2.826.0.1.3680043.8.498.11384414575384375879609510799633231326","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72587440202779699193620339987034971895","series":"1.2.826.0.1.3680043.8.498.11843716960349643355959664065098902943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72896348955280307080916504120667484813","series":"1.2.826.0.1.3680043.8.498.45743193556197379313356833114655813246","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72931900540220826595003388066347326492","series":"1.2.826.0.1.3680043.8.498.14685399791106298213255568901915127199","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72943191222961132416678221885831944435","series":"1.2.826.0.1.3680043.8.498.12506834395354221152837263794562626502","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73037642285055457162691486258363101398","series":"1.2.826.0.1.3680043.8.498.10327361441658978145469298906168547711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73021428171736137311489171988512484948","series":"1.2.826.0.1.3680043.8.498.13311756035774080365694911549969306953","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73077380963620788952030569132380549826","series":"1.2.826.0.1.3680043.8.498.11145683616163255714407791539806330392","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73101997284755549124715401557235981642","series":"1.2.826.0.1.3680043.8.498.13134458982249989620346237046615986432","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73231153161312542696053060380619432126","series":"1.2.826.0.1.3680043.8.498.29775900101263865706808018203685297402","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73205916973275504240156555900460720571","series":"1.2.826.0.1.3680043.8.498.87929125724733479288292489657953342541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73242978867957397254276973409271434573","series":"1.2.826.0.1.3680043.8.498.13364247422680291432659520074475572985","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73257122462858353041066181004123253930","series":"1.2.826.0.1.3680043.8.498.33077713529712590859236571991193524951","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73314900301724532218347713512381622074","series":"1.2.826.0.1.3680043.8.498.84475987492464622019588339023784873363","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73448420644251136304786298254801920284","series":"1.2.826.0.1.3680043.8.498.79687589874326857915156562639454161457","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73356831368999268488355650751758108912","series":"1.2.826.0.1.3680043.8.498.73651767620997919036292383599407680898","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73194579460722181989151410198309642266","series":"1.2.826.0.1.3680043.8.498.41456013457932567959085421191712998624","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73478948866451045346677275674140339803","series":"1.2.826.0.1.3680043.8.498.50462935157298747214558265217646107712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220","series":"1.2.826.0.1.3680043.8.498.10348071327293290966523579991979381444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73545582819102316606081606511277579853","series":"1.2.826.0.1.3680043.8.498.10592251764601754820240215725156883248","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73748007182569068263280098473557128226","series":"1.2.826.0.1.3680043.8.498.12685314430773824448196862722668380823","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73511703934549662392765328175356230615","series":"1.2.826.0.1.3680043.8.498.99706895063734370111952508133877314975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73682608760611536443754406204042210115","series":"1.2.826.0.1.3680043.8.498.56632664393267341263458738880370575262","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73825252186234755453650759159564715478","series":"1.2.826.0.1.3680043.8.498.79623010268978043382386426192447144533","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73602922823389605007702521335136347605","series":"1.2.826.0.1.3680043.8.498.93247798304758718686949250787271765100","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73891117356545433634964184054304516063","series":"1.2.826.0.1.3680043.8.498.95530990272860605123183374516562182233","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73983081622010008601849687650985260980","series":"1.2.826.0.1.3680043.8.498.13382064066945413456089134478522178305","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74052473719853343027548507306186291134","series":"1.2.826.0.1.3680043.8.498.36837439447383005706042620964614416813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74095402538454760179339842746572802563","series":"1.2.826.0.1.3680043.8.498.22537317190739368825598872304676377224","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74140806279409854828032496578072554031","series":"1.2.826.0.1.3680043.8.498.11195764477495822110193359777420239748","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74222284584657051737421776919599206132","series":"1.2.826.0.1.3680043.8.498.30336284545027348260106715926414494762","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74214829456846525506232367308573550186","series":"1.2.826.0.1.3680043.8.498.83426444540589130629898425514067841431","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74243110959141459268838217468465880820","series":"1.2.826.0.1.3680043.8.498.26977274722728899649111534053702796957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74282280145383357786409764912398881272","series":"1.2.826.0.1.3680043.8.498.19869322524713756912503810431289771026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74247325306307303036016759070403973520","series":"1.2.826.0.1.3680043.8.498.10645632245634241856701769193358494740","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74177736542522869266952610634091628727","series":"1.2.826.0.1.3680043.8.498.52929831732799086188685642516927980046","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74362364640190722036251935797260325860","series":"1.2.826.0.1.3680043.8.498.19736922186838321245822290713631090027","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74394784271003789564066563841806954326","series":"1.2.826.0.1.3680043.8.498.87362573879657017242198301737188759523","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74461421748415064477531528884191890661","series":"1.2.826.0.1.3680043.8.498.10774027666193881155759398415390218393","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74471724874857406578316075456414902954","series":"1.2.826.0.1.3680043.8.498.68783667368216158234241966020678247289","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74622912009754642425258484521795653023","series":"1.2.826.0.1.3680043.8.498.11926060532881759233901588214389297689","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74714018385278253151175853998945654373","series":"1.2.826.0.1.3680043.8.498.53211051985426633949463280825085920960","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74749071555443582954158991248361308204","series":"1.2.826.0.1.3680043.8.498.46582862072353597965813760762140022350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74864145321761783409935288797327597738","series":"1.2.826.0.1.3680043.8.498.50523458396466286300281783993797071182","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74940598515597814600837126537274199312","series":"1.2.826.0.1.3680043.8.498.67168798528877486177186948363871075091","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74914646210033032631540802996162814268","series":"1.2.826.0.1.3680043.8.498.69227934226904137115183523764466785772","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75168480371573015758963049782687952716","series":"1.2.826.0.1.3680043.8.498.92727942743325700702271100407157210902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75067227246352035607675978826177487284","series":"1.2.826.0.1.3680043.8.498.10564280018678910314584588910195940596","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190","series":"1.2.826.0.1.3680043.8.498.11625457303004368642031400585358126215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75378323910171368568360074903297446681","series":"1.2.826.0.1.3680043.8.498.55582055687821275598906811447462062602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75418637314688036152637525301534342254","series":"1.2.826.0.1.3680043.8.498.16859416473048475187877507619333045350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75276563505367629340309025030148269450","series":"1.2.826.0.1.3680043.8.498.61638331043388244838865638530922399812","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75518253718111135068488417046278011303","series":"1.2.826.0.1.3680043.8.498.55179554405458535075390929522574563389","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75523541256169382504121567002441615798","series":"1.2.826.0.1.3680043.8.498.11718024462343690891574900055143140803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75605038901688937188489854019478082759","series":"1.2.826.0.1.3680043.8.498.10134254916109720422007293982954307609","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75639423607035582240174986752280730347","series":"1.2.826.0.1.3680043.8.498.11835261916733248605595822879405448345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75420186662442688955616630735303321846","series":"1.2.826.0.1.3680043.8.498.78198485875867543155368118449671719299","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75688484058613633923943882730405145535","series":"1.2.826.0.1.3680043.8.498.69379717597958849120291408391418152943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75548650589349966221875964542604002971","series":"1.2.826.0.1.3680043.8.498.64274814258415261102166989050240038629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75742195884147990442829552992649204078","series":"1.2.826.0.1.3680043.8.498.86629663978615250845226551144024835973","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75773447415943537938555952303103557864","series":"1.2.826.0.1.3680043.8.498.70181778953134739701515585227523559270","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75893915698353338311947268382839152877","series":"1.2.826.0.1.3680043.8.498.10369649134908761648651898882497266276","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75926011938952404257302292495731660728","series":"1.2.826.0.1.3680043.8.498.50682273162955015895248984346730155656","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76027799502777251766328820405966979386","series":"1.2.826.0.1.3680043.8.498.67081809714744987222299814204281428874","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75751784078102748563815298069799859542","series":"1.2.826.0.1.3680043.8.498.70764472115883295171770141491762153542","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76167153566894354739332364376483514559","series":"1.2.826.0.1.3680043.8.498.40762391324766802223300236939533931215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76093766204928582819565636658977147523","series":"1.2.826.0.1.3680043.8.498.11243566688247944462266385623246911957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76122127303562588545128053661609574837","series":"1.2.826.0.1.3680043.8.498.72658887322131335959033608678017568505","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76350662767360412185091180512006419379","series":"1.2.826.0.1.3680043.8.498.27796575097549552015009851265951807839","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76281930374588438789344170829280438697","series":"1.2.826.0.1.3680043.8.498.37972480232946231764103778080533922990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75980374267002793040832157122311396174","series":"1.2.826.0.1.3680043.8.498.10120851566934647467741443199105830080","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76222770668660274805993655800271653712","series":"1.2.826.0.1.3680043.8.498.12540303068436671410700465746415446923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76538737802277241976971950690663079376","series":"1.2.826.0.1.3680043.8.498.83739669276840253909276093827204578184","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76514504104121634569792846980451896030","series":"1.2.826.0.1.3680043.8.498.75918722840183673641552980818100790777","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76651080491383110752977286124044311636","series":"1.2.826.0.1.3680043.8.498.58796258958395957900139131459076958147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76545468257774231281618306952011717992","series":"1.2.826.0.1.3680043.8.498.27599432228175709284426659229340865705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76757251008849492985513280752011229119","series":"1.2.826.0.1.3680043.8.498.61258614120351384491469428460266983225","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76903562636417038907113894538863784737","series":"1.2.826.0.1.3680043.8.498.90029678755590747568910788840665176415","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76680092665218514711300086707222109323","series":"1.2.826.0.1.3680043.8.498.68825239703133336208215056873500277222","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76993922254007307143291254174266755701","series":"1.2.826.0.1.3680043.8.498.11929681063596707829827616859736521793","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76841077743272729415666709641073096909","series":"1.2.826.0.1.3680043.8.498.11600323297983556995214492122864341502","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77037121474756969001683126693104350509","series":"1.2.826.0.1.3680043.8.498.28023367278171807139094491533496770222","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77060858802024387619459645469871885412","series":"1.2.826.0.1.3680043.8.498.82630629865191944825575585635731723969","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77164129615096222757740198026718214733","series":"1.2.826.0.1.3680043.8.498.22624750622791414062733220305686058640","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77215476384973989055525592556183934827","series":"1.2.826.0.1.3680043.8.498.66598205083440700990920677492233300702","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77261014467268552758826551926128217025","series":"1.2.826.0.1.3680043.8.498.17780651052016264389751668739938073497","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77283213969268676660453937180363826917","series":"1.2.826.0.1.3680043.8.498.23520294294517657672346506982835983465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76914908131203373740345988718674850939","series":"1.2.826.0.1.3680043.8.498.87826991828498042317417488750770146159","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77150650601457686196626648868552167695","series":"1.2.826.0.1.3680043.8.498.11614978110537962319418151766959233974","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77290477043075828139630734968573461136","series":"1.2.826.0.1.3680043.8.498.53879049114650315823127980262689458854","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77306755058630855443644908572157134062","series":"1.2.826.0.1.3680043.8.498.67833760588277512701201940669890959204","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003","series":"1.2.826.0.1.3680043.8.498.25371003339062235719535685903629725583","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77378803863827312829317008636359414012","series":"1.2.826.0.1.3680043.8.498.50802927580189953800289886282906661552","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77416382267531567401698555072875486512","series":"1.2.826.0.1.3680043.8.498.87297928186494716638082820305632194467","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77433065090757862487482575794398458402","series":"1.2.826.0.1.3680043.8.498.36913672504621463287511387709690171325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77513348456432770689938407624768928320","series":"1.2.826.0.1.3680043.8.498.18634026080600672477552122784204966188","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77577135626047243581253857225991426727","series":"1.2.826.0.1.3680043.8.498.34977023039961846225581837052798645206","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77541648519018799027301684184453151471","series":"1.2.826.0.1.3680043.8.498.80066498469155292524612571279684847812","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77626578713099735143745396709359607445","series":"1.2.826.0.1.3680043.8.498.25389456937397263689131413956785890271","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77716260980321491689913730053309892120","series":"1.2.826.0.1.3680043.8.498.55873435841112104978829390412266329746","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77760453966349959552328150395676117417","series":"1.2.826.0.1.3680043.8.498.70471506144506683170893721757885113010","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77740672313593763051900703940745812581","series":"1.2.826.0.1.3680043.8.498.24622631747447867583478060190348379131","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77801860555102174958520570751272642387","series":"1.2.826.0.1.3680043.8.498.27290762309526000832397237800409068068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77843765997593346714211135369376676497","series":"1.2.826.0.1.3680043.8.498.12568350805844417953195312673726963285","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77893704208186903913532950495036889026","series":"1.2.826.0.1.3680043.8.498.96748163620855535115615494938730211992","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77498121906981512168120474112422492971","series":"1.2.826.0.1.3680043.8.498.51037444671403759945930253354451786131","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77933128806366660573360274958258917742","series":"1.2.826.0.1.3680043.8.498.15041177639595349578424046121434219552","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77593859518024366170735307975173961777","series":"1.2.826.0.1.3680043.8.498.77012535005888356437795639079177188091","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78012314847102407375138809615977711653","series":"1.2.826.0.1.3680043.8.498.12529078943207678403394743045675613602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78084594560404633421210837175859618437","series":"1.2.826.0.1.3680043.8.498.65614551207484406087842150903751996806","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78188863861812651079751164964756388411","series":"1.2.826.0.1.3680043.8.498.11247153284812327269043986976308353516","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78236036694398253902871510456985685293","series":"1.2.826.0.1.3680043.8.498.12032234383257584173258835242906031958","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78296478954955171036523274995555716268","series":"1.2.826.0.1.3680043.8.498.19136650837090895214821749451270999941","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78282552802075313852355443512555224586","series":"1.2.826.0.1.3680043.8.498.78181317637477969377334702318909327167","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78079900750417018394321006978343945239","series":"1.2.826.0.1.3680043.8.498.57752079893959674287656448949502988484","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78476873086705419136436170130940915568","series":"1.2.826.0.1.3680043.8.498.10030972446008853822472209216602063972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78465765647012043478539489340002371960","series":"1.2.826.0.1.3680043.8.498.80052493400385613698303767997832859802","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78521396404736679127326025240279747196","series":"1.2.826.0.1.3680043.8.498.12041425747633242534661825969925803771","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78667999747216717251638754803697959701","series":"1.2.826.0.1.3680043.8.498.87708548160868029129363084789815037970","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78646473961447523033955716877524794707","series":"1.2.826.0.1.3680043.8.498.54782218780244394801357491830436330240","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78808320605816811006382053060951793555","series":"1.2.826.0.1.3680043.8.498.12212874616452345928819640525320387131","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78972123731732056524948076388825106061","series":"1.2.826.0.1.3680043.8.498.70773369525018905747133726100913514220","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78765525474890094481249498938657540052","series":"1.2.826.0.1.3680043.8.498.96774432369146343605363225083230594593","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78853931928960361455025737444267199231","series":"1.2.826.0.1.3680043.8.498.32349782157088907714401823196357337234","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79080521163097116147379189510104170611","series":"1.2.826.0.1.3680043.8.498.50349517242667053869620794254932671651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79117047831806895544174734229992359986","series":"1.2.826.0.1.3680043.8.498.10993933299637169778971795918562442428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79038153803632889185429988483097161887","series":"1.2.826.0.1.3680043.8.498.87930421056325896753428289713088695886","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78685661415919658264222803849222055515","series":"1.2.826.0.1.3680043.8.498.13114830017254987979254914550814609764","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79225094031423353857449795712402404317","series":"1.2.826.0.1.3680043.8.498.79116287154220586195464520786363907335","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79165531588966776246377098630025444815","series":"1.2.826.0.1.3680043.8.498.40159196130767835928052774663568235857","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79247871688929479411597859576284194165","series":"1.2.826.0.1.3680043.8.498.85703980753816571484340720998427564896","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79267465462008228906382361587050826784","series":"1.2.826.0.1.3680043.8.498.48337862714240178292811025027436824962","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79290157182784080975822485697879660942","series":"1.2.826.0.1.3680043.8.498.11084846054384130226086063280550288949","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79326627674334637493320764205397279674","series":"1.2.826.0.1.3680043.8.498.11871165224051826646523980790993465822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79505118958475739396320647697583979079","series":"1.2.826.0.1.3680043.8.498.63054781020704108913619191209987134828","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79551456190098389081551755482042135791","series":"1.2.826.0.1.3680043.8.498.46297495675016978936408645711926516232","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79609204436595306110253502961172135807","series":"1.2.826.0.1.3680043.8.498.76957849803855300343813069148250841045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79648517065511897282938523065537768426","series":"1.2.826.0.1.3680043.8.498.14157750861634906113512471404975887287","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79725520325008084984331823107016254775","series":"1.2.826.0.1.3680043.8.498.87565201875385962233584381961959494496","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79798779812052605703861699094268016306","series":"1.2.826.0.1.3680043.8.498.21945820627480795359359992533758361813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79977436705480715175480363175316375576","series":"1.2.826.0.1.3680043.8.498.25550822983519136385602595070736494736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79565393323323007178523850093527250842","series":"1.2.826.0.1.3680043.8.498.48718189525377665639674418644373441393","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80180578697901833764761405542790185180","series":"1.2.826.0.1.3680043.8.498.36479567505871682810355401482029719420","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80146250663015309456767479960653638934","series":"1.2.826.0.1.3680043.8.498.48369365263762233080831318676990737146","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80207056225476761432712249807198836187","series":"1.2.826.0.1.3680043.8.498.37026520093783663119780058345318987446","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80058869639900582578987760750483296668","series":"1.2.826.0.1.3680043.8.498.10588917377939566997586643267371828699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80290749690597272632725356393056746299","series":"1.2.826.0.1.3680043.8.498.11093395704768549936728399414209129860","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80319546205646617603546887163539306146","series":"1.2.826.0.1.3680043.8.498.11723989359712735249724602745578200098","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80391378873630377472192017862302619947","series":"1.2.826.0.1.3680043.8.498.34777444489118102822010542789446443026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80275294885165896124489476449616436102","series":"1.2.826.0.1.3680043.8.498.85796252806425670445507148103074147138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80458507774711142534182397228254444268","series":"1.2.826.0.1.3680043.8.498.62813766153756972992989403420813685776","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80312559977515578735238310472949287111","series":"1.2.826.0.1.3680043.8.498.13287053153809519233714827328728592270","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80514754572353458107286859773300865138","series":"1.2.826.0.1.3680043.8.498.10503997078852156621350726438299836474","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80612964209397056653334784776523875647","series":"1.2.826.0.1.3680043.8.498.85461031908976979603973131354970495822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80557835452734362083552169469427388003","series":"1.2.826.0.1.3680043.8.498.10539330415833759007434776325285960593","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80629565747112614845746475820152813307","series":"1.2.826.0.1.3680043.8.498.10297620388648925033232776540994538244","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80643480554795312280333659144941545925","series":"1.2.826.0.1.3680043.8.498.14432012861570019271138094593309342993","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80661392139668653552110941896475066204","series":"1.2.826.0.1.3680043.8.498.30298134786899685387838512603649529774","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80735285867775404153100236162962472952","series":"1.2.826.0.1.3680043.8.498.68288668191483503932849102056433937778","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80758336123908522542787716219939751704","series":"1.2.826.0.1.3680043.8.498.53153424392896705025810191881242696043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80861108320341696163987804408524915999","series":"1.2.826.0.1.3680043.8.498.78628782566688474740162205148817563835","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80878057166752747059190481131529937715","series":"1.2.826.0.1.3680043.8.498.76744579049961398688591923476744717329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80931292574057303692838531323879513417","series":"1.2.826.0.1.3680043.8.498.29941011785493870310032648398185752569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80698510014594772882040509716764819716","series":"1.2.826.0.1.3680043.8.498.42427493639391840212067860637127380227","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81018911907787774214591206085583545057","series":"1.2.826.0.1.3680043.8.498.10285700314227499870878223073238087144","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81103809943451790668501089584752866199","series":"1.2.826.0.1.3680043.8.498.55156729103258607655507426510295084154","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81342416992724702402763591633702802359","series":"1.2.826.0.1.3680043.8.498.12430133600375178237186178028991343669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81351574724227319632426427855692606694","series":"1.2.826.0.1.3680043.8.498.13139870725795316444891987863861172221","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81142440700884513521007992957945568924","series":"1.2.826.0.1.3680043.8.498.49418228016389465086283707762818739263","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81269884425000228604280521306949721859","series":"1.2.826.0.1.3680043.8.498.17720807121493490206896861136705780212","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81031057654953109295788159826841315360","series":"1.2.826.0.1.3680043.8.498.88948597818012337244640158359167477197","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81462603621780919719466264356289554669","series":"1.2.826.0.1.3680043.8.498.10035179535472534639341615771004567816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81466575994184158172641823163654418018","series":"1.2.826.0.1.3680043.8.498.67856873324260940777918960278854858840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81495303185504163367736663647489556517","series":"1.2.826.0.1.3680043.8.498.91437862747381898387317849358537792020","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81651132686162360431299931833372141617","series":"1.2.826.0.1.3680043.8.498.72323942264560964036551391339741135996","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81692996137620934043276253858381812472","series":"1.2.826.0.1.3680043.8.498.11069097667501178788576320422006950172","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81567030584250750745410094870790052181","series":"1.2.826.0.1.3680043.8.498.35363323757373440637572305104979678956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81629704995225424568187579791704967233","series":"1.2.826.0.1.3680043.8.498.92632529575479561301185992863540584333","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81738571920712657790861120957086010772","series":"1.2.826.0.1.3680043.8.498.85748864540245146326521223816357695423","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81790990482142045640192729399175106255","series":"1.2.826.0.1.3680043.8.498.15445774213462654244419734144396133360","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81796385087779065457668769443782458844","series":"1.2.826.0.1.3680043.8.498.62908586318660487955104770487688529382","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81832287065207835637503039077685315236","series":"1.2.826.0.1.3680043.8.498.40676829279101764896145178293441368165","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81927409938015158663782061022054347955","series":"1.2.826.0.1.3680043.8.498.65321952107851544099745236110796182211","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81964000393640770822199916465038759069","series":"1.2.826.0.1.3680043.8.498.72589928460235427991574917649631613829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81981151447274200345866874172324295172","series":"1.2.826.0.1.3680043.8.498.63845839824526176985437697433034486721","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81819209302731920404237374320026807282","series":"1.2.826.0.1.3680043.8.498.75110573928275413592434055754046700244","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82012532783011843175172575198326783678","series":"1.2.826.0.1.3680043.8.498.39398901511378513230761425336463077432","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82074883987134049971242727164506046476","series":"1.2.826.0.1.3680043.8.498.57462873381081977617223992402159921999","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254","series":"1.2.826.0.1.3680043.8.498.10347272512623805065972057458376849254","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82144602300666174450721045102799681426","series":"1.2.826.0.1.3680043.8.498.10124696712778542687961602116738150466","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82272793832748838882752586678639564181","series":"1.2.826.0.1.3680043.8.498.10568996911457291723691785963888054957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82414828460683289235183375457850096746","series":"1.2.826.0.1.3680043.8.498.26970293255364970290562906399126709994","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82548251499640077572026404183402934330","series":"1.2.826.0.1.3680043.8.498.11990348412552130864400517394306281424","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82447100001417065190678617013741085481","series":"1.2.826.0.1.3680043.8.498.73070150515907729162246629879218887671","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82579118531566493365068788437906656162","series":"1.2.826.0.1.3680043.8.498.33646824303433278299040293185014867638","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82744514729782879778828512295184306538","series":"1.2.826.0.1.3680043.8.498.51868683876515563231455370893755876675","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82587763533601760002619014353812074700","series":"1.2.826.0.1.3680043.8.498.78463328924956608839864447259723713743","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82655005222804459605894403147145992714","series":"1.2.826.0.1.3680043.8.498.67599725324612311395293453090935170191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82858981777405241601610241542173548956","series":"1.2.826.0.1.3680043.8.498.94394482160369741879330962335215834182","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82498616123349854719801627024073383874","series":"1.2.826.0.1.3680043.8.498.11804850347818953663020086082402319863","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83000639623679673469640458882929412717","series":"1.2.826.0.1.3680043.8.498.42274928172779372316854852774961334177","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82925474533853765570570726554869886925","series":"1.2.826.0.1.3680043.8.498.58177959005902972223809723217788007880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83030371526667747898709613600825477966","series":"1.2.826.0.1.3680043.8.498.79914667095927928930634729144892728014","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83108474538388106770920972885900596508","series":"1.2.826.0.1.3680043.8.498.13041923658224282265573316676798175869","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83055401642066749019091368358783700783","series":"1.2.826.0.1.3680043.8.498.11900821490723913589882333061912792214","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83235374118671754689663732057314083332","series":"1.2.826.0.1.3680043.8.498.75672663977365791127295385136090814940","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83152040259754415849193046299744699239","series":"1.2.826.0.1.3680043.8.498.12332514564499345614885277982353894196","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83253778356746892201320443660102495790","series":"1.2.826.0.1.3680043.8.498.47140493893550172185403938831812976398","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83495448827553002152296206085110883602","series":"1.2.826.0.1.3680043.8.498.16911689972412719931577171363657917405","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83376388091077619281568190257019735638","series":"1.2.826.0.1.3680043.8.498.85321803813972524782912290459156620710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83691358175810149975084604975065122220","series":"1.2.826.0.1.3680043.8.498.11431101652481224918309316035151741859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83456649088980869429062498344457162631","series":"1.2.826.0.1.3680043.8.498.12353569826935775526131268886029588875","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83956129905762530953192730040209719910","series":"1.2.826.0.1.3680043.8.498.50939810170493514425214557938011033201","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84047134932018909998648460546980598926","series":"1.2.826.0.1.3680043.8.498.55422884995119816156004547416446996878","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83747418172021065605877274971789603771","series":"1.2.826.0.1.3680043.8.498.16392686695686624447568905179918267839","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84069767112794890061709127671908414399","series":"1.2.826.0.1.3680043.8.498.50410332078734985089244201312759980236","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83569065014916050100714967945484998462","series":"1.2.826.0.1.3680043.8.498.60808536451958395730511306242290597063","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84230432137191939390680841389370206819","series":"1.2.826.0.1.3680043.8.498.11818753464177599632240822229721907695","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84292569246911697141902796190329671261","series":"1.2.826.0.1.3680043.8.498.10321899433171825545062098856487444495","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84483080389344581106129884869678629612","series":"1.2.826.0.1.3680043.8.498.54234347741256565356784159708919782223","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84646554968523210021448483220679710652","series":"1.2.826.0.1.3680043.8.498.77250226918273983013886006601824740191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84419297518873367123595080276985526259","series":"1.2.826.0.1.3680043.8.498.72437754237895522563325655849519151488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84612593835203397696029057940345763682","series":"1.2.826.0.1.3680043.8.498.13186138582874404416399820477107900509","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84062602050757262792437893691591415560","series":"1.2.826.0.1.3680043.8.498.18474595301641324031168496849883337382","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84748479740284893375192048321707057482","series":"1.2.826.0.1.3680043.8.498.80002710598682321389356708969662713714","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84709539128629951937294117586370340042","series":"1.2.826.0.1.3680043.8.498.11427227625845558493629265265741506442","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84896756080952941537556818257863912066","series":"1.2.826.0.1.3680043.8.498.10234730915522311141420971962025744945","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85036616529391787087213173897728060798","series":"1.2.826.0.1.3680043.8.498.16160709881243768245679045646679888708","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85108719393967663738580518985514195538","series":"1.2.826.0.1.3680043.8.498.74765958751946999030560874322060570355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85294435004325211584698794441083866921","series":"1.2.826.0.1.3680043.8.498.90443248348101182939123313144949576195","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85194002080793317087210934629814349622","series":"1.2.826.0.1.3680043.8.498.15933268726194960283072482223202587078","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85004373913191467860228868467924401547","series":"1.2.826.0.1.3680043.8.498.12073336033731682955256779476166039143","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85397267100597758920280635204297906286","series":"1.2.826.0.1.3680043.8.498.70350327198921407407933322037851873412","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85333821516157983299252782678788406019","series":"1.2.826.0.1.3680043.8.498.13335659559326939760046613953511005440","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85431902914673414035492907270636012892","series":"1.2.826.0.1.3680043.8.498.83181806601721209187815098950655972147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85347263026430684291021533170619978114","series":"1.2.826.0.1.3680043.8.498.21632828382244610364416284568164684310","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84855341377363129926967387137216814518","series":"1.2.826.0.1.3680043.8.498.10572081800211102345216867714469970828","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85464150757448712711832350617010924327","series":"1.2.826.0.1.3680043.8.498.11012752538523839796461479757155312675","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85566900782101872452970957532250214590","series":"1.2.826.0.1.3680043.8.498.41730844389731853321546912199638895375","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85649172620970194090967653766415467836","series":"1.2.826.0.1.3680043.8.498.82285863451881504999702430148345687987","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85624441072689653093363046226467141762","series":"1.2.826.0.1.3680043.8.498.23264327834381853551150516832788847474","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85687351346612079711706269264454944647","series":"1.2.826.0.1.3680043.8.498.10261186076217295470513869215481298395","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85499036441349866810703945289793057098","series":"1.2.826.0.1.3680043.8.498.68044827767799081572937527001298118252","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85735729179216019275685564112535593988","series":"1.2.826.0.1.3680043.8.498.48635603031405692168974485905052039947","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85719235937472088594405415004352845221","series":"1.2.826.0.1.3680043.8.498.65674958163967297710698966278194805045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85882078780717905717989373451578844863","series":"1.2.826.0.1.3680043.8.498.57312018519229090936756940697874532790","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85760360279564672425828346507309104394","series":"1.2.826.0.1.3680043.8.498.12434252780306966490663472528517965418","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85890890545815190189345870912381925290","series":"1.2.826.0.1.3680043.8.498.31284104894238188776923947684630239260","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85992651772359515416050249463715355943","series":"1.2.826.0.1.3680043.8.498.38872809781274245899678645897781648990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85976668143927055109184433855426232055","series":"1.2.826.0.1.3680043.8.498.10008266800101012018024124219906684387","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86029947807373604003071651721618330827","series":"1.2.826.0.1.3680043.8.498.82817641536887601175394055856334201667","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85943737587102032370951456635167782343","series":"1.2.826.0.1.3680043.8.498.55252596630956387584991909049074228063","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85870577082315172424837183881210532800","series":"1.2.826.0.1.3680043.8.498.22118441735689162564622073053597821342","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86117639434986594266345773277419881144","series":"1.2.826.0.1.3680043.8.498.89020365744889614472168677438590815014","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86216155858324761845740646421500394797","series":"1.2.826.0.1.3680043.8.498.10755889085777225712410050491466421875","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86067849233330188988240710585112183308","series":"1.2.826.0.1.3680043.8.498.12597431557706361367818628703175424605","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86251314013737658351649446042801810374","series":"1.2.826.0.1.3680043.8.498.27947262440678262377796455558231902580","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86273853115722509980053190345047859057","series":"1.2.826.0.1.3680043.8.498.60232258552003608819097593215182087494","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86285715293767810771609523819042157072","series":"1.2.826.0.1.3680043.8.498.15194074121584753000927052341852297869","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86351141114604781108401133410934252655","series":"1.2.826.0.1.3680043.8.498.76336692579737814590064197315169360082","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86409878732405036646840908998274047391","series":"1.2.826.0.1.3680043.8.498.60786873671793440549542650658352037802","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86422343639522250286002195622435564347","series":"1.2.826.0.1.3680043.8.498.20079683941980417502748364164483867122","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86332035495941070455209594003499488889","series":"1.2.826.0.1.3680043.8.498.54085398830686309541973596060325241712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86435890312227285216810145658548164773","series":"1.2.826.0.1.3680043.8.498.70828258861016863500804839953742921471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86583531066383395270546012510184348251","series":"1.2.826.0.1.3680043.8.498.21879147278561166181680771772603025457","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86532643576908316499388520854820002133","series":"1.2.826.0.1.3680043.8.498.92211960487474598848166482429925478035","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86881633316011036335786865282908198831","series":"1.2.826.0.1.3680043.8.498.72671356662664670063950888722337163335","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86974853023888933977090539174537083409","series":"1.2.826.0.1.3680043.8.498.11504035152145730194393793003361262158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87018440624359538250658207994111333051","series":"1.2.826.0.1.3680043.8.498.95651206032495862939010646211315360925","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87064372531225943854092598173945043861","series":"1.2.826.0.1.3680043.8.498.99001479494277190089259304685734715497","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86651345701812229172985086531423369421","series":"1.2.826.0.1.3680043.8.498.36188366441272388314154367740839790973","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87102698749288269439517333660896096520","series":"1.2.826.0.1.3680043.8.498.10269066972536277128227334433333853820","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87223930502803944549504594723314220105","series":"1.2.826.0.1.3680043.8.498.88303089780616907740560493705653054656","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86733565200815744326667230375642936291","series":"1.2.826.0.1.3680043.8.498.11067041584623746886795717306570431803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87198191797585172912979594638355685739","series":"1.2.826.0.1.3680043.8.498.13092087597366305016387951094011982229","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87214831397290763080243320084625847148","series":"1.2.826.0.1.3680043.8.498.60270195999898114256618749969681120562","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87253761427012478584873622994909841751","series":"1.2.826.0.1.3680043.8.498.72329455762804127330858522517485249174","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87267658933515453963224923798100651549","series":"1.2.826.0.1.3680043.8.498.58395881973418533799993568226856271805","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87404730020015145302086817782843205171","series":"1.2.826.0.1.3680043.8.498.33864338492147305320115142624953347673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87476514600989622078259281213313867927","series":"1.2.826.0.1.3680043.8.498.79023207846688288220714717632826123168","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87415625824966124550466651648244553892","series":"1.2.826.0.1.3680043.8.498.13854897225755380055000827760979723076","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87650297321648368205324227429768256110","series":"1.2.826.0.1.3680043.8.498.72877733719075765955506448091905512693","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87680825247567493080163504511649789348","series":"1.2.826.0.1.3680043.8.498.92908716682482071335804193039057341334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87563449504927531224663601218159863518","series":"1.2.826.0.1.3680043.8.498.70356018899723732846727639505213848652","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87550037046735743504446944390018580426","series":"1.2.826.0.1.3680043.8.498.67166366687723996954220717498109098847","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87761573479019057997334508266888503487","series":"1.2.826.0.1.3680043.8.498.11593657581716220244445985417722685883","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87797202230328868466634480198668365648","series":"1.2.826.0.1.3680043.8.498.11642693958580472754149852864334916948","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87900330305833722536121052961334699743","series":"1.2.826.0.1.3680043.8.498.88479115611575927120757037243233913582","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87845297884497964690686366000533570840","series":"1.2.826.0.1.3680043.8.498.24961159954945737593806138808575645966","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88070896604159829163161019117970440189","series":"1.2.826.0.1.3680043.8.498.72538141138037613245096515241403305038","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88022456207467042544403752144674777523","series":"1.2.826.0.1.3680043.8.498.32511376168384641660615282913409714209","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88189503151202366936330411627291827689","series":"1.2.826.0.1.3680043.8.498.11134284959205221090895275957229381024","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435","series":"1.2.826.0.1.3680043.8.498.30657822634139739188979013992800834075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88505044370271842163477475980178015566","series":"1.2.826.0.1.3680043.8.498.13730901074113947083248884760120004006","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88439732365857773633852840383740478214","series":"1.2.826.0.1.3680043.8.498.90179231107161225401647833494235970465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88466268868654468582661984391239692466","series":"1.2.826.0.1.3680043.8.498.11742799805820445193847689179737648692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88274753211076645976496220746740420894","series":"1.2.826.0.1.3680043.8.498.50191931812608080101949655870885221037","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88609906210780568637209291182572648571","series":"1.2.826.0.1.3680043.8.498.10064481329092717090324940006461756315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88571985843649843469694556575325271560","series":"1.2.826.0.1.3680043.8.498.48939418294561711115250244044137019850","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88565389747258713872180859306907392080","series":"1.2.826.0.1.3680043.8.498.43269272045798415271544213602590644558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88679328523895187500714090963655363787","series":"1.2.826.0.1.3680043.8.498.28728241058968108595724241669521395616","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88653085234639096911379842849193874077","series":"1.2.826.0.1.3680043.8.498.58224234075116053048907690289786633445","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88726804476694411363564819498296119763","series":"1.2.826.0.1.3680043.8.498.76834786879241380268407390382148294946","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88687994744258163246685527273559741656","series":"1.2.826.0.1.3680043.8.498.23629329040419491912563830218346647066","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88872155446935518174304990308380906112","series":"1.2.826.0.1.3680043.8.498.97542156257832829173003387137728090946","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88896246957673838955012057210098034177","series":"1.2.826.0.1.3680043.8.498.29201324514865661613425369374864768597","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88962401400598535739494671683916272476","series":"1.2.826.0.1.3680043.8.498.51082148881366720962083167558592106479","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89026102874017923932934673207699042800","series":"1.2.826.0.1.3680043.8.498.10101040229452848020149459969845526907","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88753415430913291809543943690942744997","series":"1.2.826.0.1.3680043.8.498.93022923277720600320364435371515083852","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89048907638162719840222508855779172366","series":"1.2.826.0.1.3680043.8.498.48539917837399635932820522824891624333","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89110271437785174288044757208973128908","series":"1.2.826.0.1.3680043.8.498.32615236646542211674664053681939413388","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88827276474694165071267273766912871459","series":"1.2.826.0.1.3680043.8.498.75220792547031947652188148595580731525","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89190511197564127079697151213577962649","series":"1.2.826.0.1.3680043.8.498.98018485122832897038407961215385262568","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89240184178186399632216878563608101260","series":"1.2.826.0.1.3680043.8.498.85865575582367375711471141879340213069","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89302191571937190529967168726521003428","series":"1.2.826.0.1.3680043.8.498.51118140251581704111284469054454531194","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89346975863478164334871204856160071946","series":"1.2.826.0.1.3680043.8.498.93771350695620962039711731848975140246","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89326347774248795083360112232993521530","series":"1.2.826.0.1.3680043.8.498.13215309278085496244309677246991550620","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89357823844365374776758395347764601030","series":"1.2.826.0.1.3680043.8.498.97341504612171875413174894909749801756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89423600201830685256510321877396547497","series":"1.2.826.0.1.3680043.8.498.11283085428988504719256823665855223285","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89576499383950909889665633610791093334","series":"1.2.826.0.1.3680043.8.498.10968891303104749055226988798403237586","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89464179959474561696319051639748844244","series":"1.2.826.0.1.3680043.8.498.53424656729199876389816886027729688510","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89590880980421560369495600204866789646","series":"1.2.826.0.1.3680043.8.498.67611298231637908727513625216922003388","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89729486637365590877469535082675069395","series":"1.2.826.0.1.3680043.8.498.11028439450767729022506106473499940119","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89943815722422273305517145510644432987","series":"1.2.826.0.1.3680043.8.498.10522068282711648719452619495487462444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89857801102910604766959864464865910585","series":"1.2.826.0.1.3680043.8.498.12777917608699964730170252436606963074","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89829843399218627892046362324057240535","series":"1.2.826.0.1.3680043.8.498.12193614395246345038022771337107697028","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90003472003278237920648198243464374280","series":"1.2.826.0.1.3680043.8.498.12055115849874312711205699468826866466","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89967772422936726289461201791871771011","series":"1.2.826.0.1.3680043.8.498.26880821424155747731239615119071500102","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90159195278632080234412316065469499332","series":"1.2.826.0.1.3680043.8.498.11587113107887118432700439151667095391","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90057130466053377217652879986591534745","series":"1.2.826.0.1.3680043.8.498.29922234556909163941651148995030706296","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90172333170788218623219231170360850947","series":"1.2.826.0.1.3680043.8.498.43753689220874776512944599062568478172","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90206716031776606956011363950625953310","series":"1.2.826.0.1.3680043.8.498.11141470347110722444321778645819095406","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90274711315692820326973277297753398634","series":"1.2.826.0.1.3680043.8.498.10911422560395830655919010595908000495","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90264300606047579088362539636050224478","series":"1.2.826.0.1.3680043.8.498.17928384674498826963779898154266856918","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90288598546566291188788375261370919222","series":"1.2.826.0.1.3680043.8.498.96240106589845149701822424514528096061","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90227414485305096825508911050013279704","series":"1.2.826.0.1.3680043.8.498.13247872358623783695590224310691643825","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90433488790413321102007716267340900056","series":"1.2.826.0.1.3680043.8.498.88967144440686012823021873764073919076","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90313529150165069133283531436628039942","series":"1.2.826.0.1.3680043.8.498.18141808203569639170489395213024835997","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90340114291852810896219650388952790164","series":"1.2.826.0.1.3680043.8.498.77281832621160517188133985399849159385","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90514150380296346197442897381548735293","series":"1.2.826.0.1.3680043.8.498.16200009883628005320849442486548972003","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90490891782778285565462083536468192410","series":"1.2.826.0.1.3680043.8.498.58847507987572114764821322724913659976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90751504670572401790063555695835439317","series":"1.2.826.0.1.3680043.8.498.23580036786525362145347010286747656939","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90573769367031841574114648773801425781","series":"1.2.826.0.1.3680043.8.498.93134319033742687879363128441345815551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90668646188784648688693620159738574680","series":"1.2.826.0.1.3680043.8.498.11268091180452583667838268657410870513","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90466423642678253933409966898120881010","series":"1.2.826.0.1.3680043.8.498.31571964375162057252627261836369855488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90976227335921831531105845602655777296","series":"1.2.826.0.1.3680043.8.498.10147445232019973091373874316712623382","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90764713690393618577990540005126771712","series":"1.2.826.0.1.3680043.8.498.54665683653539939600592935127031644440","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90858397443821905121918628339486070979","series":"1.2.826.0.1.3680043.8.498.36869765339731306297504564560703874802","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91064647744940506223536204418057554573","series":"1.2.826.0.1.3680043.8.498.21116469024684068166622500169234030998","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91122873808284731048016330734204992698","series":"1.2.826.0.1.3680043.8.498.41832713334397391238535772998980062045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91168270990167204047014161851340054645","series":"1.2.826.0.1.3680043.8.498.93999576120450815553804137726345495467","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91220255464004258635083940797356822178","series":"1.2.826.0.1.3680043.8.498.38850103767664587203475694859767912011","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91125742927944703615970502025043944089","series":"1.2.826.0.1.3680043.8.498.90913540948969581132463718870717332410","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91346221361264140052760282748336176394","series":"1.2.826.0.1.3680043.8.498.11581357038201549313736784732149663273","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91391446167553025113367459399831345295","series":"1.2.826.0.1.3680043.8.498.89137093393967512219335072909822671125","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91380337141071062166571333367919885984","series":"1.2.826.0.1.3680043.8.498.86032361659667055392608606647765095501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91499489440453244654238202136675083579","series":"1.2.826.0.1.3680043.8.498.54465200225249126737330535826782296602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91468214041781427810665883790357021637","series":"1.2.826.0.1.3680043.8.498.21358023132959313331332246646851084743","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91237082716839283641165609666644466929","series":"1.2.826.0.1.3680043.8.498.85379909208662201549710633899974392990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91606395836371290680039618164703651751","series":"1.2.826.0.1.3680043.8.498.60145168201406346160967855134996787994","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91660278231317256353374962767408532443","series":"1.2.826.0.1.3680043.8.498.12734201828126354649570141613860909097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91323765268136520804399328350809179765","series":"1.2.826.0.1.3680043.8.498.20454221001408768346269964226732581881","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91744384920314309659802891351055961910","series":"1.2.826.0.1.3680043.8.498.92612917056879583698764297098091109186","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91860013313252950044036077977399321593","series":"1.2.826.0.1.3680043.8.498.93774557287685087912999084216491076266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91897419008563027309752122760214275182","series":"1.2.826.0.1.3680043.8.498.38969735588274625319444434328761979654","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91913894091933783677568170210064807665","series":"1.2.826.0.1.3680043.8.498.74652557512986684749617610981252007258","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91727248305411866145131034915485728037","series":"1.2.826.0.1.3680043.8.498.11419252392993469962489494636073049989","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91979363712708769014277114061159398016","series":"1.2.826.0.1.3680043.8.498.10861530093817007276885541128011419259","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92030581934838097812004394123949263069","series":"1.2.826.0.1.3680043.8.498.95606761382746416238097454783170097184","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92071199603599654827155753204585082303","series":"1.2.826.0.1.3680043.8.498.11544221075752804081480882246956908417","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92127426417720107708158713612888411025","series":"1.2.826.0.1.3680043.8.498.85123538825980697283443119345161511734","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92386519544130078776672685639580497728","series":"1.2.826.0.1.3680043.8.498.76919350567825536633087752498947641003","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92301915122994899889226415477164044164","series":"1.2.826.0.1.3680043.8.498.93642620813709868443795528044212541465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92249560662718438332916289759028859881","series":"1.2.826.0.1.3680043.8.498.92876485138135191834565474666079081389","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92410537691481397330646029810090178745","series":"1.2.826.0.1.3680043.8.498.58363623852593675309720163219829122967","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92449204325697143336618138322280397765","series":"1.2.826.0.1.3680043.8.498.35758640376564846750508208181075505980","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92602515666185999130675702469977883616","series":"1.2.826.0.1.3680043.8.498.98915373361308314698393870478367068769","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92626490033376759309712417622692944095","series":"1.2.826.0.1.3680043.8.498.74125508440977419669802760068672778900","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92655291659311409459093330375014445330","series":"1.2.826.0.1.3680043.8.498.99198191456957743037623873366015118798","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92581137608969847756039955144478716506","series":"1.2.826.0.1.3680043.8.498.65606634471674348800906629085846649923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92715232057122971236003034574127234433","series":"1.2.826.0.1.3680043.8.498.60320422028331202523265248117792465813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92758635328468215409701857842011454608","series":"1.2.826.0.1.3680043.8.498.28210117027467244006385887746385513719","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92854445271746966212168817610305595078","series":"1.2.826.0.1.3680043.8.498.45651183303604840337108931334472419174","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93096918345463865518663434763610016449","series":"1.2.826.0.1.3680043.8.498.26726053640714468900813376708597611324","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93141871723058796160969424330424882014","series":"1.2.826.0.1.3680043.8.498.10118120242791986212832215059556105655","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92925901047699000727725798824992891055","series":"1.2.826.0.1.3680043.8.498.42956330769567762540384918288427995560","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93054327536380380487760790221613186305","series":"1.2.826.0.1.3680043.8.498.56482490390794502723505130344542315295","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93301337627536436086612556500687157032","series":"1.2.826.0.1.3680043.8.498.10981920078136634510359344600556169532","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93344237572402384894445814841725878706","series":"1.2.826.0.1.3680043.8.498.12397239377453170491328723459532084884","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93182602591865019295859776146674822677","series":"1.2.826.0.1.3680043.8.498.12350719286613120139321272394734015815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93385433312281352893457225415812880948","series":"1.2.826.0.1.3680043.8.498.31290427729728472257614568042102115039","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93653421810951484288813767943453706233","series":"1.2.826.0.1.3680043.8.498.65833189236206859970047858193804699848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93222021909191248270818335069738080963","series":"1.2.826.0.1.3680043.8.498.94882443657121712579700999728004173336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93718451128867963123946970296603639068","series":"1.2.826.0.1.3680043.8.498.16103883054630185056621898809179663616","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93555726419626687118723083838655757151","series":"1.2.826.0.1.3680043.8.498.53215791983841335496130129585789849157","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93747304211180517979657679136059719120","series":"1.2.826.0.1.3680043.8.498.24712146038044826209016234398325378320","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93848205752465320137433228658870667418","series":"1.2.826.0.1.3680043.8.498.61743573001292487676235759113800028712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93921993164883084178408030385007465064","series":"1.2.826.0.1.3680043.8.498.13160732588216873149417911991389740950","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94027547274004055924993074754507802303","series":"1.2.826.0.1.3680043.8.498.12941934310659148721298156261745030373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94069670011071682445520414370532541612","series":"1.2.826.0.1.3680043.8.498.10583486887151541643865616228282630043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93722352425168280299901366261374293722","series":"1.2.826.0.1.3680043.8.498.51601288960315474687762408438152276425","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93985093305024344873337249433396141838","series":"1.2.826.0.1.3680043.8.498.80545508275343304166170345566827463199","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94296943940827583101208235208463322140","series":"1.2.826.0.1.3680043.8.498.85682509836578029531646275116813748900","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94255124739558547668304022649099251744","series":"1.2.826.0.1.3680043.8.498.25802652950504398801646132082913234631","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94335780236370744781443631231792792083","series":"1.2.826.0.1.3680043.8.498.59240970341534630559382903344275997828","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94393348753936295687030819683853459606","series":"1.2.826.0.1.3680043.8.498.74054401169745871279349788573318605641","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94244885518758327449236878653096161475","series":"1.2.826.0.1.3680043.8.498.44035916311415963049207961172282027673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94447683257072169359896104855962933413","series":"1.2.826.0.1.3680043.8.498.11990117080011704156574510720950387575","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94553664348118184200007654265863262684","series":"1.2.826.0.1.3680043.8.498.10172215480709509782568654919519753034","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94607524665407029611795868588964734480","series":"1.2.826.0.1.3680043.8.498.86323983109760288648631570588443443071","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94594009911808653224418094594012527944","series":"1.2.826.0.1.3680043.8.498.19525394035488600069576706189863701527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94677297888135961827159679484207792618","series":"1.2.826.0.1.3680043.8.498.60152405466586393932378664940986827987","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94729634553835820944505159643192054175","series":"1.2.826.0.1.3680043.8.498.20477532864815202608731467960203855315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94753144127920558221268610279343359449","series":"1.2.826.0.1.3680043.8.498.23158822425506108405372064782419509999","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94635110520016467882028406590905441740","series":"1.2.826.0.1.3680043.8.498.90072003701697367458275881154678876736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94822220672465914426995315295921716764","series":"1.2.826.0.1.3680043.8.498.92357246629041917188951574293008348157","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94774472240035721003264951868030913589","series":"1.2.826.0.1.3680043.8.498.12091150840340800358527833864913536993","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94944218313788305917509271079443871442","series":"1.2.826.0.1.3680043.8.498.77114982564360431519228347656326432073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95060380238431209646600378585422128745","series":"1.2.826.0.1.3680043.8.498.12227192362506794861154851902276096280","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95026524886584356226576936309513335214","series":"1.2.826.0.1.3680043.8.498.44280686858591280561642730138598966808","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94954489705497211843190734684710878341","series":"1.2.826.0.1.3680043.8.498.47779975441532343830920130221814775990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95142886737239438773421088967802136318","series":"1.2.826.0.1.3680043.8.498.10254376833131008216082821079909208804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95091624999382167943525022414631622532","series":"1.2.826.0.1.3680043.8.498.45930284804483229975648398757527291195","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95321750682026667237717880394536750835","series":"1.2.826.0.1.3680043.8.498.10495335028378608556819056815906306171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95334149193711661424105715330320414755","series":"1.2.826.0.1.3680043.8.498.12018155166667611945810330389175577426","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95256518245659249144569318023469597946","series":"1.2.826.0.1.3680043.8.498.94706834391924923552403322755880690405","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95376417067109020835342111882595126109","series":"1.2.826.0.1.3680043.8.498.13181678008625403433285369618017095108","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95196756237527666678816095708949563380","series":"1.2.826.0.1.3680043.8.498.12475471328262859038820261930081455843","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95497572581157858069692837655072209020","series":"1.2.826.0.1.3680043.8.498.11717909564308502556445940550984222590","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95588576819758266413765165887081039863","series":"1.2.826.0.1.3680043.8.498.10811699484927951870569669118853699888","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95570207430778332247200880037915693375","series":"1.2.826.0.1.3680043.8.498.55530892703542407100099851990354408530","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95464807346156547985763374990661923815","series":"1.2.826.0.1.3680043.8.498.97978129203737817168907613414164960619","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95555755747044090477918262640906805885","series":"1.2.826.0.1.3680043.8.498.12889201896549497572311734352414665910","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95653228095828585867177997067051877351","series":"1.2.826.0.1.3680043.8.498.47909343698113837871880603658739232349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95621693348567815920675804584086926261","series":"1.2.826.0.1.3680043.8.498.10523253177169975436113106231615436161","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95705755527332814775935856896567782522","series":"1.2.826.0.1.3680043.8.498.38723588940396933619532018359934877640","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95727100148189624148200652225439854460","series":"1.2.826.0.1.3680043.8.498.83967136765203909953427489271526445908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95777611643095956714301310992151723603","series":"1.2.826.0.1.3680043.8.498.10875575232887719045673354046843867959","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95882731821729917461275694847968503951","series":"1.2.826.0.1.3680043.8.498.11214466458205596033640753377618619741","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95850844945129153291759207212815927078","series":"1.2.826.0.1.3680043.8.498.97791909868984001315293777437868532028","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96051765028927322365509303222455708654","series":"1.2.826.0.1.3680043.8.498.11533653364851466614874713196510396009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96094001324999448553107805455529187871","series":"1.2.826.0.1.3680043.8.498.11360092019960624975498341513354936369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95918254432056542701730614049432316728","series":"1.2.826.0.1.3680043.8.498.56446168128694689940701884752213832711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96146854841734718524530740965296119916","series":"1.2.826.0.1.3680043.8.498.10829402595042374602511894229209267844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96293531288862652713666056362328893428","series":"1.2.826.0.1.3680043.8.498.19461017110548607382135841834653270783","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96448229695053562621553132798380456420","series":"1.2.826.0.1.3680043.8.498.13117901156340869840236878827557766588","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96323485839061204012967074568679195142","series":"1.2.826.0.1.3680043.8.498.52975632608437116858271260721328626493","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96158169859491339496094295508194547199","series":"1.2.826.0.1.3680043.8.498.66075606903930517333169099258422286863","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96510621665552630060821422211995212854","series":"1.2.826.0.1.3680043.8.498.85948445958599789003826206817122286746","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96561978771090020509310535265236331946","series":"1.2.826.0.1.3680043.8.498.11405738615098393538228748005854902835","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96564709711765982786409987515735443801","series":"1.2.826.0.1.3680043.8.498.46867767224708288615225261725085737471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96614828054790815108190478795539254570","series":"1.2.826.0.1.3680043.8.498.71631320898326695691856544638269900298","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96818533353374094273022322108861564926","series":"1.2.826.0.1.3680043.8.498.18679253671987294334054268168630833903","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96859317329616917022799512740280754327","series":"1.2.826.0.1.3680043.8.498.18842565802003680662093981855116887490","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96950011369238410949579095568317214261","series":"1.2.826.0.1.3680043.8.498.12025446437137983545196258943902642476","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97004483591331502283638578516018997228","series":"1.2.826.0.1.3680043.8.498.60738813477876428542942885921170724050","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96657465815875587552665111166265086308","series":"1.2.826.0.1.3680043.8.498.52819770215626676291290353344702176711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97042446513400958172996637321308032556","series":"1.2.826.0.1.3680043.8.498.94589387711956642437397773606916520126","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97087958215554026148585108910609248523","series":"1.2.826.0.1.3680043.8.498.16788010914842515729536979717185551987","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97220604868276521004168045790587997687","series":"1.2.826.0.1.3680043.8.498.56948538257205557841238955735151966957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97259977321685658337087695233264517545","series":"1.2.826.0.1.3680043.8.498.66303995092659101989705513894652915374","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96944926470854551064948200936284887742","series":"1.2.826.0.1.3680043.8.498.22223462134148429086434286618106946698","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97135552871905937467775630206570358085","series":"1.2.826.0.1.3680043.8.498.49429628738599051970736110494929944971","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97248397528214121944398119979516268902","series":"1.2.826.0.1.3680043.8.498.12171766757557808756644764551613488018","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97348112008590316355392924312704081684","series":"1.2.826.0.1.3680043.8.498.31708202554322319762402904596586487394","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97296323335289879687414244774227160450","series":"1.2.826.0.1.3680043.8.498.59488190402380900416396458909803499500","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97400307568415006244714024769069514497","series":"1.2.826.0.1.3680043.8.498.62703830555748841413100581257181788639","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97485476951995325021464683747847126869","series":"1.2.826.0.1.3680043.8.498.27501227804883504236491902271628485343","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97496319463700021339637597270948309369","series":"1.2.826.0.1.3680043.8.498.12390557107074601054949108332059463306","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97520184067825971918316082134210579391","series":"1.2.826.0.1.3680043.8.498.10047149081643727582361533331099856959","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97564564380731258764291146761742841527","series":"1.2.826.0.1.3680043.8.498.12779735820241684524084980174035673960","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662","series":"1.2.826.0.1.3680043.8.498.26303375425736222633996027764568407709","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97689748704772735695349883980413529091","series":"1.2.826.0.1.3680043.8.498.13388466006568704979238256617953118178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97600121574703193898619052529830275028","series":"1.2.826.0.1.3680043.8.498.10363616349203392356541213139184247318","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97731968872354496470096174559789586056","series":"1.2.826.0.1.3680043.8.498.10480598902269102194793415984550182138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97849030682292314880463675622456900933","series":"1.2.826.0.1.3680043.8.498.13300004775023807509919575492003701915","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97944261713902372188894456162512065499","series":"1.2.826.0.1.3680043.8.498.30838528101219329662934041519223113284","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97696436660690920809826140686268769509","series":"1.2.826.0.1.3680043.8.498.11128433477299371553120073051575709292","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98022738778461179495257509215244053198","series":"1.2.826.0.1.3680043.8.498.74778310044625381375405720020290395257","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97797693721516872008302328638374252668","series":"1.2.826.0.1.3680043.8.498.71892990458821437644329480799108587871","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98028018520101798863795620289151810642","series":"1.2.826.0.1.3680043.8.498.10007238824192154774338796293608787949","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98064941971367073978404461105852658265","series":"1.2.826.0.1.3680043.8.498.82563680182842672804030495784014077921","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98112373403331748422888362310572528002","series":"1.2.826.0.1.3680043.8.498.13340453810751749407016918146674847011","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98102475310924339032647376491442498974","series":"1.2.826.0.1.3680043.8.498.83002160890752609128967350982293856428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98146155044178593810385943365143374814","series":"1.2.826.0.1.3680043.8.498.32811399939595489548550698443205961760","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98333658788330318335997032513277632526","series":"1.2.826.0.1.3680043.8.498.12618036230218914146178568578199718663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98351891935507349137982199658103205287","series":"1.2.826.0.1.3680043.8.498.40741919800880076403582100417271781261","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98255454187594895604610283922744869720","series":"1.2.826.0.1.3680043.8.498.36533637428145850361701572294871351322","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98181040478767526925214101502401448074","series":"1.2.826.0.1.3680043.8.498.41875167312752839502958293656379024914","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98581537941208552065033680853629122764","series":"1.2.826.0.1.3680043.8.498.21185789106721388821265406707666884010","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98781117432577714521231232440503778261","series":"1.2.826.0.1.3680043.8.498.47166337429963042718849265061622655908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98824066826946884510314135448574304233","series":"1.2.826.0.1.3680043.8.498.47111351312581112918495089241188013799","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98676079331571304021387833410440136393","series":"1.2.826.0.1.3680043.8.498.87027964012526866599209251864339753869","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98454206323401382140655387224146512225","series":"1.2.826.0.1.3680043.8.498.46948743524877746104571154541975538055","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98958651320983196982606333548522846934","series":"1.2.826.0.1.3680043.8.498.23701403134050340053520057612299581333","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98890851347757949328159691828768030366","series":"1.2.826.0.1.3680043.8.498.12299171993946928726459356548382901095","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99077241529686301278201179461896593659","series":"1.2.826.0.1.3680043.8.498.26297611965069209578322206839597161370","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99108147913292332418681050058312852494","series":"1.2.826.0.1.3680043.8.498.11247768821070127399035787792123207059","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99243902497374852166139435719549686908","series":"1.2.826.0.1.3680043.8.498.24297466937188030770025456540752395309","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99178010550062268668346041362629896201","series":"1.2.826.0.1.3680043.8.498.14483630655722881520771178183454442331","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99303079292320085374389086609761974820","series":"1.2.826.0.1.3680043.8.498.71103253860611776587435100525803283290","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98871650330162988912501872031191530068","series":"1.2.826.0.1.3680043.8.498.93372874090607611724332437955666101409","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99444110025206148999881196086066868280","series":"1.2.826.0.1.3680043.8.498.66890737973748962413552712352001094170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99438805806853675793560604625783460513","series":"1.2.826.0.1.3680043.8.498.35512252819558707290544208946022043606","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99413398303724030328059886749330413303","series":"1.2.826.0.1.3680043.8.498.12775149029690654037375534375753218619","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99454657594216441037514296544803013407","series":"1.2.826.0.1.3680043.8.498.69381700453492471702839986638835382447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99577045720075471058329330051900767206","series":"1.2.826.0.1.3680043.8.498.50119154113720113688072955806113931765","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99725523560254732094432244556623376711","series":"1.2.826.0.1.3680043.8.498.50174723276820764851261348616601790593","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99493257395588095270208592176887619190","series":"1.2.826.0.1.3680043.8.498.72802444739655932625101173307345085614","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99635784824711754116112241806442235135","series":"1.2.826.0.1.3680043.8.498.94086954753345948841387412499146658803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99768154649604643975127490783989114506","series":"1.2.826.0.1.3680043.8.498.18865150805207093116215223644364468212","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99826037216341310731701527407057049820","series":"1.2.826.0.1.3680043.8.498.12723944002573784499653596840821017997","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99708702196778120583644711299950439036","series":"1.2.826.0.1.3680043.8.498.96578625824454484723823066955976307859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99880021280666757817587834481314366555","series":"1.2.826.0.1.3680043.8.498.64324647689002382904014367598143788918","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99926624968240681772735102303531613209","series":"1.2.826.0.1.3680043.8.498.31216809385545367412195846511090977787","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99864345386635409953374818045984558970","series":"1.2.826.0.1.3680043.8.498.87976140363194866351754113977800189119","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99957611207313098696146908027229590966","series":"1.2.826.0.1.3680043.8.498.32916764040885408297619051929589831373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"}]')
print('series a convertir (UID figes depuis crop130 224) :', len(jobs))
assert len(jobs) == 2204
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    # pad/crop spatial identique : si shapes divergent (rare), resize au mode
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w)

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    band = vol[a:b + 1]
    idx = np.linspace(0, band.shape[0] - 1, N_SLICES).round().astype(int)
    band = band[idx]

    out = np.empty((N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    for i in range(N_SLICES):
        sl = cv2.resize(band[i], (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        out[i] = np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    # fallback : parfois series a la racine de train_series
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_320_t1_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / "manifest_t1_part0.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "note": "1 serie T1/non-fluide extra, UID figes depuis crop130 224. Crop 130 mm (PixelSpacing) sinon centre 70 %. Bande profondeur 20-80 %, linspace 24, resize 320, uint8.",
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

# ZipFile n'est pas thread-safe en ecriture : workers convertissent, le
# thread principal ecrit le zip.
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

# Sanity : recharger 3 NPZ, shapes, moments
import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    means = []
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        print(name, vol.shape, vol.dtype, "min", int(vol.min()), "max", int(vol.max()),
              "mean", float(vol.mean()), "std", float(vol.std()))
        assert vol.shape == (N_SLICES, IMG_SIZE, IMG_SIZE), vol.shape
        means.append(float(vol.mean()))
print("mean uint8 (3 samples)", means)
print(f"attendu : shape ({N_SLICES}, {IMG_SIZE}, {IMG_SIZE}) uint8")
print("cible historique NPZ communautaires : mean~42 std~49.5 (crop 130 mm => moments differents, c'est normal)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
